# EEG-Based ADHD Detection Using Dynamic Envelope Connectivity and a Hybrid Spatio-Temporal Graph Transformer

This notebook provides a reproducible technical analysis for subject-level ADHD detection from EEG using band-limited amplitude-envelope connectivity and the proposed Hybrid Multihead Spatio-Temporal Graph Transformer Index.

## Section 1. Notebook Metadata and Reproducibility Setup


In [1]:
# Cell 1: Import required libraries and fix all random seeds.
import hashlib
import importlib.metadata as importlib_metadata
import json
import math
import os
import pickle
import random
import re
import sys
import warnings
import zipfile
from pathlib import Path
from typing import Callable, Dict, List, Tuple

import matplotlib
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from matplotlib.colors import TwoSlopeNorm
from scipy import signal, stats
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    silhouette_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore")

SEED = 42
BOOTSTRAP_SEED = 20260406
N_BOOTSTRAP = 1000


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


set_seed(SEED)
torch.set_num_threads(max(1, min(8, os.cpu_count() or 4)))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PACKAGE_NAMES = [
    "numpy",
    "pandas",
    "scipy",
    "scikit-learn",
    "matplotlib",
    "networkx",
    "torch",
    "nbformat",
    "nbclient",
    "openpyxl",
]

version_rows = []
for package_name in PACKAGE_NAMES:
    try:
        version_value = importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        version_value = "not installed"
    version_rows.append({"package": package_name, "version": version_value})

version_df = pd.DataFrame(version_rows)
print(f"Python executable: {sys.executable}")
print(f"Computation device: {DEVICE}")
print(f"Global random seed: {SEED}")
display(version_df)


Python executable: /Users/talgatazykanov/miniforge3/envs/ai-classic-ml-cpu/bin/python
Computation device: cpu
Global random seed: 42


,package,version
0,numpy,1.26.4
1,pandas,3.0.2
2,scipy,1.17.1
3,scikit-learn,1.8.0
4,matplotlib,3.10.9
5,networkx,3.6.1
6,torch,2.10.0
7,nbformat,5.10.4
8,nbclient,0.10.4
9,openpyxl,3.1.5


In [2]:
# Cell 2: Configure Matplotlib-only publication plotting defaults and reusable figure utilities.
PLOT_DPI = 350
PLOT_FONT_SIZE = 16
MATRIX_ANNOTATION_FONT_SIZE = 18
PLOT_LEGEND_SIZE = 16
PLOT_TITLE_SIZE = 18
MUTED_BLUE = "#4C78A8"
MUTED_RED = "#E45756"
MUTED_TEAL = "#72B7B2"
MUTED_GREEN = "#2E8B57"
MUTED_PURPLE = "#B279A2"
MUTED_GREY = "#6B7280"
CLASS_COLORS = {0: MUTED_BLUE, 1: MUTED_RED}
SPLIT_COLORS = {"train": MUTED_BLUE, "validation": MUTED_TEAL, "test": MUTED_RED}

plt.rcParams.update(
    {
        "font.family": "Times New Roman",
        "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
        "font.size": PLOT_FONT_SIZE,
        "axes.titlesize": PLOT_TITLE_SIZE,
        "axes.labelsize": PLOT_FONT_SIZE,
        "xtick.labelsize": PLOT_FONT_SIZE,
        "ytick.labelsize": PLOT_FONT_SIZE,
        "legend.fontsize": PLOT_LEGEND_SIZE,
        "figure.dpi": PLOT_DPI,
        "savefig.dpi": PLOT_DPI,
        "axes.linewidth": 1.1,
        "lines.linewidth": 2.2,
        "lines.markersize": 7.0,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": False,
        "legend.frameon": False,
    }
)

figure_save_log: List[Dict[str, object]] = []


def save_figure(fig: matplotlib.figure.Figure, filename: str) -> Path:
    output_path = FIGURES_DIR / filename
    fig.savefig(output_path, dpi=PLOT_DPI, bbox_inches="tight", facecolor="white")
    figure_save_log.append({"file": str(output_path), "dpi": PLOT_DPI})
    plt.show()
    return output_path


def add_matrix_annotations(ax, image, matrix: np.ndarray, fmt: str = ".2f", fontsize: int = MATRIX_ANNOTATION_FONT_SIZE) -> None:
    matrix = np.asarray(matrix, dtype=float)
    for row_idx in range(matrix.shape[0]):
        for col_idx in range(matrix.shape[1]):
            value = matrix[row_idx, col_idx]
            rgba = image.cmap(image.norm(value))
            luminance = 0.2126 * rgba[0] + 0.7152 * rgba[1] + 0.0722 * rgba[2]
            color = "white" if luminance < 0.45 else "black"
            ax.text(col_idx, row_idx, format(value, fmt), ha="center", va="center", color=color, fontsize=fontsize)


def plot_matrix_on_axis(
    ax,
    matrix: np.ndarray,
    title: str,
    x_labels: List[str],
    y_labels: List[str],
    cmap: str = "coolwarm",
    center: float | None = 0.0,
    vmin: float | None = None,
    vmax: float | None = None,
    colorbar_label: str = "Value",
    annotate: bool = True,
):
    matrix = np.asarray(matrix, dtype=float)
    if center is not None and vmin is not None and vmax is not None:
        norm = TwoSlopeNorm(vmin=vmin, vcenter=center, vmax=vmax)
        image = ax.imshow(matrix, cmap=cmap, norm=norm, aspect="auto")
    else:
        image = ax.imshow(matrix, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    ax.set_title(title, pad=12)
    ax.set_xticks(np.arange(len(x_labels)))
    ax.set_xticklabels(x_labels, rotation=45, ha="right")
    ax.set_yticks(np.arange(len(y_labels)))
    ax.set_yticklabels(y_labels)
    ax.tick_params(axis="both", length=5, width=1.0)
    if annotate:
        add_matrix_annotations(ax, image, matrix, fmt=".2f", fontsize=MATRIX_ANNOTATION_FONT_SIZE)
    cbar = plt.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(colorbar_label)
    cbar.ax.tick_params(labelsize=PLOT_FONT_SIZE)
    return image


def add_vertical_bar_labels(ax, bars, value_fmt: str = "{:.2f}") -> None:
    heights = [float(bar.get_height()) for bar in bars]
    max_height = max(max(heights), 0.01)
    min_height = min(min(heights), 0.0)
    ax.set_ylim(min_height - abs(min_height) * 0.10, max_height * 1.18)
    for bar in bars:
        value = float(bar.get_height())
        ax.annotate(
            value_fmt.format(value),
            xy=(bar.get_x() + bar.get_width() / 2.0, value),
            xytext=(0, 6),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=PLOT_FONT_SIZE,
            clip_on=False,
        )


def add_horizontal_bar_labels(ax, bars, value_fmt: str = "{:.2f}") -> None:
    widths = [float(bar.get_width()) for bar in bars]
    max_width = max(max(widths), 0.01)
    ax.set_xlim(min(0.0, min(widths)) * 1.10, max_width * 1.18)
    for bar in bars:
        value = float(bar.get_width())
        ax.annotate(
            value_fmt.format(value),
            xy=(value, bar.get_y() + bar.get_height() / 2.0),
            xytext=(8, 0),
            textcoords="offset points",
            ha="left",
            va="center",
            fontsize=PLOT_FONT_SIZE,
            clip_on=False,
        )


plot_config_df = pd.DataFrame(
    [
        ("Visualization library", "Matplotlib only"),
        ("Font family", str(plt.rcParams["font.family"])),
        ("Saved figure dpi", PLOT_DPI),
        ("Text size range", f"{PLOT_FONT_SIZE}-{PLOT_TITLE_SIZE} pt"),
        ("Layout policy", "constrained_layout or explicit spacing with bbox_inches='tight'"),
        ("Color policy", "muted professional colors"),
    ],
    columns=["Setting", "Active value"],
)
print("Active visualization configuration")
display(plot_config_df)


Active visualization configuration


,Setting,Active value
0,Visualization library,Matplotlib only
1,Font family,['Times New Roman']
2,Saved figure dpi,350
3,Text size range,16-18 pt
4,Layout policy,constrained_layout or explicit spacing with bb...
5,Color policy,muted professional colors


In [3]:

# Cell 3: Define all source and v2 output paths without modifying original notebooks or v1 outputs.
PROJECT_ROOT = Path.cwd().resolve()
ORIGINAL_NOTEBOOK_PATH = (PROJECT_ROOT / "example.ipynb").resolve()
V1_NOTEBOOK_PATH = (PROJECT_ROOT / "example_reviewer_ready_reanalysis.ipynb").resolve()
NEW_NOTEBOOK_PATH = (PROJECT_ROOT / "EEG_ADHD_Dynamic_Envelope_Connectivity_Hybrid_ST_Graph_Transformer.ipynb").resolve()
DATA_PATH = (PROJECT_ROOT / "adhdata.csv").resolve()
V1_OUTPUT_DIR = (PROJECT_ROOT / "outputs_reviewer_ready").resolve()
OUTPUT_DIR = (PROJECT_ROOT / "outputs_reviewer_ready_v2").resolve()
MAIN_FIGURES_DIR = OUTPUT_DIR / "main_figures"
SUPPLEMENTARY_FIGURES_DIR = OUTPUT_DIR / "supplementary_figures"
FIGURES_DIR = MAIN_FIGURES_DIR
TABLES_DIR = OUTPUT_DIR / "tables"
MODELS_DIR = OUTPUT_DIR / "models"
LOGS_DIR = OUTPUT_DIR / "logs"
REPRODUCIBILITY_DIR = OUTPUT_DIR / "reproducibility_package"

for directory in [OUTPUT_DIR, MAIN_FIGURES_DIR, SUPPLEMENTARY_FIGURES_DIR, TABLES_DIR, MODELS_DIR, LOGS_DIR, REPRODUCIBILITY_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

ORIGINAL_NOTEBOOK_SHA256_INITIAL = hashlib.sha256(ORIGINAL_NOTEBOOK_PATH.read_bytes()).hexdigest()
V1_NOTEBOOK_SHA256_INITIAL = hashlib.sha256(V1_NOTEBOOK_PATH.read_bytes()).hexdigest()

path_df = pd.DataFrame(
    [
        ("Project root", PROJECT_ROOT),
        ("Original notebook preserved", ORIGINAL_NOTEBOOK_PATH),
        ("Reviewer-ready v1 notebook preserved", V1_NOTEBOOK_PATH),
        ("New v2 notebook", NEW_NOTEBOOK_PATH),
        ("Dataset path copied from original notebook", DATA_PATH),
        ("v1 output root preserved", V1_OUTPUT_DIR),
        ("v2 output root", OUTPUT_DIR),
        ("v2 main figure output directory", MAIN_FIGURES_DIR),
        ("v2 supplementary figure output directory", SUPPLEMENTARY_FIGURES_DIR),
        ("v2 table output directory", TABLES_DIR),
        ("v2 model artifact directory", MODELS_DIR),
        ("v2 log output directory", LOGS_DIR),
        ("v2 reproducibility package directory", REPRODUCIBILITY_DIR),
    ],
    columns=["Path role", "Resolved path"],
)
display(path_df)


,Path role,Resolved path
0,Project root,/Users/talgatazykanov/Desktop/Science works/Ma...
1,Original notebook preserved,/Users/talgatazykanov/Desktop/Science works/Ma...
2,Reviewer-ready v1 notebook preserved,/Users/talgatazykanov/Desktop/Science works/Ma...
3,New v2 notebook,/Users/talgatazykanov/Desktop/Science works/Ma...
4,Dataset path copied from original notebook,/Users/talgatazykanov/Desktop/Science works/Ma...
5,v1 output root preserved,/Users/talgatazykanov/Desktop/Science works/Ma...
6,v2 output root,/Users/talgatazykanov/Desktop/Science works/Ma...
7,v2 main figure output directory,/Users/talgatazykanov/Desktop/Science works/Ma...
8,v2 supplementary figure output directory,/Users/talgatazykanov/Desktop/Science works/Ma...
9,v2 table output directory,/Users/talgatazykanov/Desktop/Science works/Ma...


## Section 2. Dataset Loading and Integrity Audit


In [4]:
# Cell 4: Load the dataset using the same path and label logic as the original notebook.
CHANNELS = [
    "Fp1",
    "Fp2",
    "F3",
    "F4",
    "C3",
    "C4",
    "P3",
    "P4",
    "O1",
    "O2",
    "F7",
    "F8",
    "T7",
    "T8",
    "P7",
    "P8",
    "Fz",
    "Cz",
    "Pz",
]
LABEL_MAP = {"Control": 0, "ADHD": 1}
LABEL_NAME_MAP = {0: "Control", 1: "ADHD"}
TRIU_IDX = np.triu_indices(len(CHANNELS), k=1)

eeg_df = pd.read_csv(DATA_PATH)
eeg_df["Class"] = eeg_df["Class"].astype(str)
eeg_df["ID"] = eeg_df["ID"].astype(str)
eeg_df["label"] = eeg_df["Class"].map(LABEL_MAP).astype(int)

load_overview_df = pd.DataFrame(
    [
        ("Rows", eeg_df.shape[0]),
        ("Columns", eeg_df.shape[1]),
        ("EEG channels", len(CHANNELS)),
        ("Unique subjects", eeg_df["ID"].nunique()),
    ],
    columns=["Item", "Value"],
)
print("Dataset shape and column overview")
display(load_overview_df)
display(pd.DataFrame({"columns": eeg_df.columns.tolist()}))
print("First five rows")
display(eeg_df.head())
print("Data types")
display(eeg_df.dtypes.reset_index().rename(columns={"index": "column", 0: "dtype"}))


Dataset shape and column overview


,Item,Value
0,Rows,2166383
1,Columns,22
2,EEG channels,19
3,Unique subjects,121


,columns
0,Fp1
1,Fp2
2,F3
3,F4
4,C3
5,C4
6,P3
7,P4
8,O1
9,O2


First five rows


,Fp1,Fp2,F3,F4,C3,C4,P3,P4,O1,O2,...,T7,T8,P7,P8,Fz,Cz,Pz,Class,ID,label
0,261.0,402.0,16.0,261.0,126.0,384.0,126.0,236.0,52.0,236.0,...,200.0,494.0,126.0,236.0,121.0,367.0,121.0,ADHD,v10p,1
1,121.0,191.0,-94.0,85.0,16.0,200.0,126.0,52.0,347.0,273.0,...,126.0,347.0,52.0,52.0,15.0,121.0,-19.0,ADHD,v10p,1
2,-55.0,85.0,-204.0,15.0,-57.0,200.0,52.0,126.0,236.0,200.0,...,126.0,420.0,52.0,126.0,-55.0,261.0,85.0,ADHD,v10p,1
3,191.0,85.0,52.0,50.0,89.0,236.0,163.0,89.0,89.0,89.0,...,236.0,420.0,126.0,126.0,15.0,85.0,-55.0,ADHD,v10p,1
4,-55.0,-125.0,-204.0,-160.0,-204.0,16.0,-241.0,-241.0,89.0,16.0,...,89.0,310.0,-57.0,52.0,-55.0,15.0,-336.0,ADHD,v10p,1


Data types


,column,dtype
0,Fp1,float64
1,Fp2,float64
2,F3,float64
3,F4,float64
4,C3,float64
5,C4,float64
6,P3,float64
7,P4,float64
8,O1,float64
9,O2,float64


In [5]:
# Cell 5: Validate required columns, class labels, subject identifiers, and numerical integrity.
required_columns = CHANNELS + ["Class", "ID", "label"]
missing_required_columns = [column for column in required_columns if column not in eeg_df.columns]
unexpected_labels = sorted(set(eeg_df["Class"].unique()) - set(LABEL_MAP))
nonfinite_counts = pd.Series(
    np.sum(~np.isfinite(eeg_df[CHANNELS].to_numpy(dtype=float)), axis=0),
    index=CHANNELS,
    name="nonfinite_count",
)
missing_counts = eeg_df[required_columns].isna().sum().rename("missing_count")
duplicate_rows = int(eeg_df.duplicated().sum())

subject_table = (
    eeg_df.groupby("ID")
    .agg(class_name=("Class", "first"), label=("label", "first"), n_samples=("label", "size"))
    .reset_index()
    .sort_values(["label", "ID"])
    .reset_index(drop=True)
)

class_distribution_df = (
    subject_table.groupby("class_name")
    .agg(n_subjects=("ID", "nunique"), n_rows=("n_samples", "sum"))
    .reset_index()
)
subject_distribution_df = subject_table[["ID", "class_name", "label", "n_samples"]].copy()

integrity_df = pd.DataFrame(
    [
        ("Missing required columns", ", ".join(missing_required_columns) if missing_required_columns else "None"),
        ("Unexpected labels", ", ".join(unexpected_labels) if unexpected_labels else "None"),
        ("Duplicate full rows", duplicate_rows),
        ("Maximum missing values in required columns", int(missing_counts.max())),
        ("Maximum non-finite EEG values per channel", int(nonfinite_counts.max())),
    ],
    columns=["Integrity item", "Result"],
)
display(integrity_df)
print("Missing values by column")
display(missing_counts.reset_index().rename(columns={"index": "column"}))
print("Non-finite values by EEG channel")
display(nonfinite_counts.reset_index().rename(columns={"index": "channel"}))
print("Class distribution")
display(class_distribution_df)
print("Subject distribution")
display(subject_distribution_df)

if missing_required_columns or unexpected_labels or int(missing_counts.max()) > 0 or int(nonfinite_counts.max()) > 0:
    raise ValueError("Dataset integrity checks failed. Review the displayed integrity table.")


,Integrity item,Result
0,Missing required columns,None
1,Unexpected labels,None
2,Duplicate full rows,45
3,Maximum missing values in required columns,0
4,Maximum non-finite EEG values per channel,0


Missing values by column


,column,missing_count
0,Fp1,0
1,Fp2,0
2,F3,0
3,F4,0
4,C3,0
5,C4,0
6,P3,0
7,P4,0
8,O1,0
9,O2,0


Non-finite values by EEG channel


,channel,nonfinite_count
0,Fp1,0
1,Fp2,0
2,F3,0
3,F4,0
4,C3,0
5,C4,0
6,P3,0
7,P4,0
8,O1,0
9,O2,0


Class distribution


,class_name,n_subjects,n_rows
0,ADHD,61,1207069
1,Control,60,959314


Subject distribution


,ID,class_name,label,n_samples
0,v107,Control,0,19794
1,v108,Control,0,19026
2,v109,Control,0,16044
3,v110,Control,0,16549
4,v111,Control,0,15310
...,...,...,...,...
116,v39p,ADHD,1,18177
117,v3p,ADHD,1,33570
118,v40p,ADHD,1,20097
119,v6p,ADHD,1,17561


In [6]:
# Cell 6: Create a reviewer-facing dataset card.
FS_HZ = 500.0
dataset_card_df = pd.DataFrame(
    [
        ("number of rows", int(eeg_df.shape[0])),
        ("number of subjects", int(subject_table.shape[0])),
        ("number of ADHD subjects", int((subject_table["label"] == 1).sum())),
        ("number of control subjects", int((subject_table["label"] == 0).sum())),
        ("number of EEG channels", len(CHANNELS)),
        ("sampling frequency used in computation", f"{FS_HZ:.0f} Hz"),
        ("source dataset path", str(DATA_PATH)),
        ("label encoding", "Control = 0; ADHD = 1"),
        ("public-data trace", "Kaggle redistribution of Nasrabadi, Allahverdy, Samavati, and Mohammadi EEG ADHD/control data; IEEE DataPort DOI: 10.21227/rzfh-zn36"),
    ],
    columns=["Field", "Value"],
)
display(dataset_card_df)
dataset_card_df.to_csv(TABLES_DIR / "dataset_card.csv", index=False)
try:
    dataset_card_df.to_excel(TABLES_DIR / "dataset_card.xlsx", index=False)
except Exception as exc:
    print(f"Excel export skipped: {exc}")


,Field,Value
0,number of rows,2166383
1,number of subjects,121
2,number of ADHD subjects,61
3,number of control subjects,60
4,number of EEG channels,19
5,sampling frequency used in computation,500 Hz
6,source dataset path,/Users/talgatazykanov/Desktop/Science works/Ma...
7,label encoding,Control = 0; ADHD = 1
8,public-data trace,"Kaggle redistribution of Nasrabadi, Allahverdy..."


In [7]:
# Cell 7: Audit sampling-frequency evidence and compute time-window quantities.
SELECTED_WINDOW = 512
SELECTED_STEP = 256
SEQ_LEN = 10
SEQ_STRIDE = 5
WINDOW_CANDIDATES = [256, 512, 1024]
THRESHOLD_CANDIDATES = [0.20, 0.30, 0.40, 0.50]
SELECTED_THRESHOLD = 0.40


def read_docx_text(path: Path) -> str:
    try:
        with zipfile.ZipFile(path) as archive:
            xml = archive.read("word/document.xml").decode("utf-8", errors="ignore")
        return re.sub(r"<[^>]+>", " ", xml)
    except Exception:
        return ""


metadata_scan_rows = []
for path in PROJECT_ROOT.iterdir():
    if path.name.startswith(".") or path.name == "adhdata.csv":
        continue
    text = ""
    if path.suffix.lower() in {".py", ".md", ".txt", ".ipynb"}:
        text = path.read_text(encoding="utf-8", errors="ignore")
    elif path.suffix.lower() == ".docx":
        text = read_docx_text(path)
    if not text:
        continue
    has_500 = bool(re.search(r"\b500\s*(Hz|Hertz|Гц|H)\b", text, flags=re.IGNORECASE))
    has_128 = bool(re.search(r"\b128\s*(Hz|Hertz|Гц)\b", text, flags=re.IGNORECASE))
    if has_500 or has_128:
        metadata_scan_rows.append(
            {
                "source_file": path.name,
                "mentions_500_Hz": has_500,
                "mentions_128_Hz": has_128,
            }
        )

metadata_scan_df = pd.DataFrame(metadata_scan_rows)
if metadata_scan_df.empty:
    metadata_scan_df = pd.DataFrame(
        [{"source_file": "No local text metadata hit", "mentions_500_Hz": False, "mentions_128_Hz": False}]
    )

sampling_audit_df = pd.DataFrame(
    [
        ("sampling frequency used in computation", FS_HZ),
        ("window length in samples", SELECTED_WINDOW),
        ("window length in seconds", SELECTED_WINDOW / FS_HZ),
        ("stride in samples", SELECTED_STEP),
        ("stride in seconds", SELECTED_STEP / FS_HZ),
        ("sequence length in windows", SEQ_LEN),
        ("sequence stride in windows", SEQ_STRIDE),
        ("approximate sequence duration in seconds", (SELECTED_WINDOW + (SEQ_LEN - 1) * SELECTED_STEP) / FS_HZ),
        ("approximate sequence stride in seconds", (SEQ_STRIDE * SELECTED_STEP) / FS_HZ),
    ],
    columns=["Sampling/windowing item", "Value"],
)
sampling_discrepancy_df = pd.DataFrame(
    [
        (
            "Computation uses 500 Hz",
            True,
            "The original notebook explicitly defines FS_HZ = 500.0 and all window durations here are computed from that value.",
        ),
        (
            "Local metadata indicates 128 Hz",
            bool(metadata_scan_df["mentions_128_Hz"].any()),
            "Flagged if any scanned local document explicitly mentions 128 Hz.",
        ),
    ],
    columns=["Audit check", "Status", "Interpretation"],
)

print("Sampling-frequency metadata scan")
display(metadata_scan_df)
print("Sampling-frequency discrepancy audit")
display(sampling_discrepancy_df)
print("Reviewer-ready sampling/window table")
display(sampling_audit_df)
sampling_audit_df.to_csv(TABLES_DIR / "sampling_frequency_audit.csv", index=False)
metadata_scan_df.to_csv(TABLES_DIR / "sampling_frequency_local_metadata_scan.csv", index=False)


Sampling-frequency metadata scan


,source_file,mentions_500_Hz,mentions_128_Hz
0,computers-4198686.docx,True,False
1,EEG_ADHD_Dynamic_Envelope_Connectivity_Hybrid_ST_Graph_Transformer.ipynb,True,True
2,Ответ_рецензентам_Makpal.docx,True,False
3,_tmp_record_capture.docx,True,False
4,Makpal_Article_N2.docx,True,False
5,build_reviewer_ready_reanalysis_v2_notebook.py,True,True
6,example.ipynb,True,False
7,build_reviewer_response_doc.py,True,False
8,Рецензент Makpal.docx,True,False
9,build_reviewer_ready_reanalysis_notebook.py,True,True


Sampling-frequency discrepancy audit


,Audit check,Status,Interpretation
0,Computation uses 500 Hz,True,The original notebook explicitly defines FS_HZ...
1,Local metadata indicates 128 Hz,True,Flagged if any scanned local document explicit...


Reviewer-ready sampling/window table


,Sampling/windowing item,Value
0,sampling frequency used in computation,500.000
1,window length in samples,512.000
2,window length in seconds,1.024
3,stride in samples,256.000
4,stride in seconds,0.512
5,sequence length in windows,10.000
6,sequence stride in windows,5.000
7,approximate sequence duration in seconds,5.632
8,approximate sequence stride in seconds,2.560


## Section 3. Subject-Level Splitting and Leakage Prevention


In [8]:
# Cell 8: Recreate the original fixed subject-level train/validation/test split.
train_ids, temp_ids, y_train_ids, y_temp_ids = train_test_split(
    subject_table["ID"],
    subject_table["label"],
    test_size=0.30,
    stratify=subject_table["label"],
    random_state=SEED,
)
val_ids, test_ids, y_val_ids, y_test_ids = train_test_split(
    temp_ids,
    y_temp_ids,
    test_size=0.50,
    stratify=y_temp_ids,
    random_state=SEED,
)

split_map = {}
for sid in train_ids:
    split_map[sid] = "train"
for sid in val_ids:
    split_map[sid] = "validation"
for sid in test_ids:
    split_map[sid] = "test"

subject_split_df = subject_table.copy()
subject_split_df["split"] = subject_split_df["ID"].map(split_map)
subject_split_df = subject_split_df.sort_values(["split", "label", "ID"]).reset_index(drop=True)
display(subject_split_df)
subject_split_df.to_csv(TABLES_DIR / "subject_split_allocation.csv", index=False)


,ID,class_name,label,n_samples,split
0,v123,Control,0,14550,test
1,v134,Control,0,13359,test
2,v138,Control,0,12488,test
3,v140,Control,0,17187,test
4,v298,Control,0,17782,test
...,...,...,...,...,...
116,v236,ADHD,1,13502,validation
117,v250,ADHD,1,13853,validation
118,v25p,ADHD,1,9894,validation
119,v29p,ADHD,1,24193,validation


In [9]:
# Cell 9: Display subject counts and row counts by split and class.
split_subject_counts_df = (
    subject_split_df.groupby(["split", "class_name"])
    .agg(n_subjects=("ID", "nunique"))
    .reset_index()
)
split_row_counts_df = (
    subject_split_df.groupby(["split", "class_name"])
    .agg(n_rows=("n_samples", "sum"))
    .reset_index()
)
split_counts_df = split_subject_counts_df.merge(split_row_counts_df, on=["split", "class_name"], how="outer")
print("Subject and row counts by split and class")
display(split_counts_df)
split_counts_df.to_csv(TABLES_DIR / "split_subject_and_row_counts.csv", index=False)


Subject and row counts by split and class


,split,class_name,n_subjects,n_rows
0,test,ADHD,10,172563
1,test,Control,9,144789
2,train,ADHD,42,853530
3,train,Control,42,669483
4,validation,ADHD,9,180976
5,validation,Control,9,145042


In [10]:
# Cell 10: Perform and enforce subject-level leakage checks.
leakage_check_df = pd.DataFrame(
    [
        ("train ∩ validation", len(set(train_ids) & set(val_ids))),
        ("train ∩ test", len(set(train_ids) & set(test_ids))),
        ("validation ∩ test", len(set(val_ids) & set(test_ids))),
    ],
    columns=["Subject-set comparison", "Overlap count"],
)
display(leakage_check_df)
leakage_check_df.to_csv(TABLES_DIR / "subject_leakage_check.csv", index=False)
if not (leakage_check_df["Overlap count"] == 0).all():
    raise RuntimeError("Subject-level leakage detected. Execution stopped.")
print("Leakage check passed: no subject appears in more than one split.")


,Subject-set comparison,Overlap count
0,train ∩ validation,0
1,train ∩ test,0
2,validation ∩ test,0


Leakage check passed: no subject appears in more than one split.


## Section 4. Signal Preprocessing


In [11]:
# Cell 11: Apply subject-wise demeaning/detrending using the original constant-detrend logic.
def demean_detrend_subject(subject_array: np.ndarray) -> np.ndarray:
    return signal.detrend(subject_array.astype(np.float32), axis=0, type="constant").astype(np.float32)


demeaned_subject_arrays: Dict[str, np.ndarray] = {}
preprocess_summary_rows = []
for row in subject_split_df.itertuples(index=False):
    raw_subject = eeg_df.loc[eeg_df["ID"] == row.ID, CHANNELS].to_numpy(dtype=np.float32)
    demeaned = demean_detrend_subject(raw_subject)
    demeaned_subject_arrays[row.ID] = demeaned
    preprocess_summary_rows.append(
        {
            "ID": row.ID,
            "class_name": row.class_name,
            "split": row.split,
            "raw_mean_abs_channel_mean": float(np.abs(raw_subject.mean(axis=0)).mean()),
            "after_demean_abs_channel_mean": float(np.abs(demeaned.mean(axis=0)).mean()),
            "raw_global_std": float(raw_subject.std()),
            "after_demean_global_std": float(demeaned.std()),
        }
    )

demean_summary_df = pd.DataFrame(preprocess_summary_rows)
print("Before/after demeaning and detrending summary")
display(demean_summary_df.groupby(["split", "class_name"]).mean(numeric_only=True).reset_index())
display(demean_summary_df.head(12))


Before/after demeaning and detrending summary


,split,class_name,raw_mean_abs_channel_mean,after_demean_abs_channel_mean,raw_global_std,after_demean_global_std
0,test,ADHD,140.429190,0.000005,188.725198,188.695998
1,test,Control,137.891815,0.000006,222.545853,222.522984
2,train,ADHD,139.089938,0.000005,187.766158,187.736390
3,train,Control,140.621003,0.000006,243.149062,243.127549
4,validation,ADHD,143.031769,0.000005,210.093282,210.072420
5,validation,Control,140.586421,0.000006,244.610101,244.589696


,ID,class_name,split,raw_mean_abs_channel_mean,after_demean_abs_channel_mean,raw_global_std,after_demean_global_std
0,v123,Control,test,144.179199,0.000007,279.721375,279.702759
1,v134,Control,test,141.505569,0.000006,192.604889,192.582184
2,v138,Control,test,138.140427,0.000006,165.177322,165.150665
3,v140,Control,test,135.386871,0.000004,150.612793,150.586853
4,v298,Control,test,137.918793,0.000004,118.271698,118.238853
5,v309,Control,test,132.913574,0.000007,274.381500,274.369324
6,v310,Control,test,132.186417,0.000005,178.467239,178.448456
7,v52p,Control,test,144.712189,0.000007,223.964066,223.941055
8,v60p,Control,test,134.083298,0.000007,419.711792,419.686707
9,v177,ADHD,test,143.286942,0.000006,200.261353,200.235199


In [12]:
# Cell 12: Apply robust amplitude clipping using median +/- 6 MAD, matching the original preprocessing.
preprocessed_subject_arrays: Dict[str, np.ndarray] = {}
clip_count_by_channel = pd.Series(0, index=CHANNELS, dtype=np.int64)
total_count_by_channel = pd.Series(0, index=CHANNELS, dtype=np.int64)

for row in subject_split_df.itertuples(index=False):
    x = demeaned_subject_arrays[row.ID]
    med = np.median(x, axis=0, keepdims=True)
    mad = np.median(np.abs(x - med), axis=0, keepdims=True)
    robust_sigma = 1.4826 * np.maximum(mad, 1e-6)
    lower = med - 6.0 * robust_sigma
    upper = med + 6.0 * robust_sigma
    clipped_mask = (x < lower) | (x > upper)
    clipped = np.clip(x, lower, upper)
    clipped = clipped - np.median(clipped, axis=0, keepdims=True)
    preprocessed_subject_arrays[row.ID] = clipped.astype(np.float32)
    clip_count_by_channel += clipped_mask.sum(axis=0)
    total_count_by_channel += x.shape[0]

clipping_summary_df = pd.DataFrame(
    {
        "channel": CHANNELS,
        "n_clipped_values": clip_count_by_channel.to_numpy(dtype=int),
        "n_total_values": total_count_by_channel.to_numpy(dtype=int),
    }
)
clipping_summary_df["fraction_clipped"] = (
    clipping_summary_df["n_clipped_values"] / clipping_summary_df["n_total_values"].clip(lower=1)
)
clipping_summary_df = clipping_summary_df.sort_values("fraction_clipped", ascending=False).reset_index(drop=True)
display(clipping_summary_df)
clipping_summary_df.to_csv(TABLES_DIR / "robust_clipping_by_channel.csv", index=False)


,channel,n_clipped_values,n_total_values,fraction_clipped
0,Fp2,24173,2166383,0.011158
1,Fp1,20400,2166383,0.009417
2,F8,17544,2166383,0.008098
3,T8,17030,2166383,0.007861
4,T7,15626,2166383,0.007213
5,F7,14004,2166383,0.006464
6,P8,13561,2166383,0.006260
7,O1,9844,2166383,0.004544
8,P7,9535,2166383,0.004401
9,C4,8669,2166383,0.004002


In [13]:
# Cell 13: Define theta and beta band-pass filters exactly as in the original notebook.
EEG_BANDS = {
    "delta": (1.0, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta": (13.0, 30.0),
    "gamma": (30.0, 45.0),
}
MODELING_BANDS = {"theta": (4.0, 8.0), "beta": (13.0, 30.0)}
BANDPASS_CACHE = {
    name: signal.butter(4, band, btype="bandpass", fs=FS_HZ, output="sos")
    for name, band in MODELING_BANDS.items()
}


def bandpass_filter(x: np.ndarray, band_name: str) -> np.ndarray:
    return signal.sosfiltfilt(BANDPASS_CACHE[band_name], x, axis=0).astype(np.float32)


filter_config_df = pd.DataFrame(
    [
        (band_name, band_range[0], band_range[1], "Butterworth", 4, FS_HZ)
        for band_name, band_range in MODELING_BANDS.items()
    ],
    columns=["Band", "Low cut (Hz)", "High cut (Hz)", "Filter family", "Order", "Sampling frequency (Hz)"],
)
display(filter_config_df)
filter_config_df.to_csv(TABLES_DIR / "bandpass_filter_configuration.csv", index=False)


,Band,Low cut (Hz),High cut (Hz),Filter family,Order,Sampling frequency (Hz)
0,theta,4.0,8.0,Butterworth,4,500.0
1,beta,13.0,30.0,Butterworth,4,500.0


In [14]:
# Cell 14: Extract Hilbert amplitude envelopes and display a compact sanity-check figure.
def amplitude_envelope(x: np.ndarray) -> np.ndarray:
    return np.log1p(np.abs(signal.hilbert(x, axis=0))).astype(np.float32)


subject_envelopes: Dict[str, Dict[str, np.ndarray]] = {}
for row in subject_split_df.itertuples(index=False):
    preprocessed = preprocessed_subject_arrays[row.ID]
    theta_signal = bandpass_filter(preprocessed, "theta")
    beta_signal = bandpass_filter(preprocessed, "beta")
    subject_envelopes[row.ID] = {
        "theta": amplitude_envelope(theta_signal),
        "beta": amplitude_envelope(beta_signal),
    }

example_control_id = subject_split_df[subject_split_df["label"] == 0].iloc[0]["ID"]
example_adhd_id = subject_split_df[subject_split_df["label"] == 1].iloc[0]["ID"]
example_channel = "Fz"
example_channel_idx = CHANNELS.index(example_channel)

fig, axes = plt.subplots(2, 1, figsize=(16, 10), constrained_layout=False)
legend_handles = None
legend_labels = None
for ax, subject_id, class_name in [
    (axes[0], example_control_id, "Control"),
    (axes[1], example_adhd_id, "ADHD"),
]:
    raw_segment = preprocessed_subject_arrays[subject_id][:1500, example_channel_idx]
    theta_segment = subject_envelopes[subject_id]["theta"][:1500, example_channel_idx]
    beta_segment = subject_envelopes[subject_id]["beta"][:1500, example_channel_idx]
    time_axis = np.arange(raw_segment.size) / FS_HZ
    raw_line = ax.plot(time_axis, raw_segment, color=MUTED_GREY, alpha=0.45, linewidth=1.8, label="Preprocessed amplitude")[0]
    envelope_ax = ax.twinx()
    theta_line = envelope_ax.plot(time_axis, theta_segment, color=MUTED_BLUE, linewidth=2.8, label="Theta envelope", zorder=5)[0]
    beta_line = envelope_ax.plot(time_axis, beta_segment, color=MUTED_RED, linewidth=2.6, linestyle="--", label="Beta envelope", zorder=6)[0]
    envelope_values = np.r_[theta_segment, beta_segment]
    envelope_min = float(np.nanmin(envelope_values))
    envelope_max = float(np.nanmax(envelope_values))
    envelope_margin = max((envelope_max - envelope_min) * 0.18, 0.15)
    envelope_ax.set_ylim(envelope_min - envelope_margin, envelope_max + envelope_margin)
    envelope_ax.set_ylabel("Log envelope")
    envelope_ax.tick_params(axis="y", labelsize=PLOT_FONT_SIZE, length=5, width=1.0)
    ax.set_title(f"{class_name} subject {subject_id}: raw amplitude and Hilbert envelopes")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Preprocessed amplitude")
    if legend_handles is None:
        legend_handles = [raw_line, theta_line, beta_line]
        legend_labels = [handle.get_label() for handle in legend_handles]

fig.legend(legend_handles, legend_labels, loc="lower center", bbox_to_anchor=(0.5, 0.025), ncol=3, frameon=False)
fig.subplots_adjust(left=0.08, right=0.92, top=0.93, bottom=0.16, hspace=0.42)

saved_path = save_figure(fig, "preprocessing_raw_theta_beta_envelope_sanity.png")
sanity_df = pd.DataFrame(
    [
        ("Example channel", example_channel),
        ("Control example subject", example_control_id),
        ("ADHD example subject", example_adhd_id),
        ("Saved figure", str(saved_path)),
    ],
    columns=["Item", "Value"],
)
display(sanity_df)


Figure saved at OLD_Docs/outputs_reviewer_ready_v2/main_figures/preprocessing_raw_theta_beta_envelope_sanity.png

,Item,Value
0,Example channel,Fz
1,Control example subject,v123
2,ADHD example subject,v177
3,Saved figure,/Users/talgatazykanov/Desktop/Science works/Ma...


## Section 5. Windowing and Sequence Construction


In [15]:
# Cell 15: Segment each subject into overlapping windows and report accepted/rejected windows by split.
subject_window_index: Dict[str, np.ndarray] = {}
window_count_rows = []
for row in subject_split_df.itertuples(index=False):
    n_samples = preprocessed_subject_arrays[row.ID].shape[0]
    starts = np.arange(0, max(n_samples - SELECTED_WINDOW + 1, 0), SELECTED_STEP, dtype=int)
    subject_window_index[row.ID] = starts
    window_count_rows.append(
        {
            "ID": row.ID,
            "split": row.split,
            "class_name": row.class_name,
            "label": row.label,
            "n_samples": int(n_samples),
            "accepted_windows": int(starts.size),
            "rejected_windows_label_inhomogeneity": 0,
            "unwindowed_tail_samples": int(max(n_samples - (starts[-1] + SELECTED_WINDOW), 0)) if starts.size else int(n_samples),
        }
    )

window_count_df = pd.DataFrame(window_count_rows)
window_count_summary_df = (
    window_count_df.groupby(["split", "class_name"])
    .agg(
        accepted_windows=("accepted_windows", "sum"),
        rejected_windows_label_inhomogeneity=("rejected_windows_label_inhomogeneity", "sum"),
        median_windows_per_subject=("accepted_windows", "median"),
    )
    .reset_index()
)
display(window_count_summary_df)
window_count_summary_df.to_csv(TABLES_DIR / "accepted_rejected_windows_by_split.csv", index=False)


,split,class_name,accepted_windows,rejected_windows_label_inhomogeneity,median_windows_per_subject
0,test,ADHD,659,0,56.0
1,test,Control,552,0,55.0
2,train,ADHD,3271,0,73.0
3,train,Control,2553,0,59.5
4,validation,ADHD,692,0,69.0
5,validation,Control,552,0,61.0


In [16]:
# Cell 16: Compute theta-band amplitude-envelope connectivity matrices.
def safe_corrcoef(x: np.ndarray) -> np.ndarray:
    corr = np.corrcoef(x, rowvar=False)
    corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
    corr = 0.5 * (corr + corr.T)
    np.fill_diagonal(corr, 0.0)
    return corr.astype(np.float32)


subject_connectivity_windows: Dict[str, Dict[str, np.ndarray]] = {}
subject_connectivity_store: Dict[str, Dict[str, np.ndarray]] = {}

for row in subject_split_df.itertuples(index=False):
    starts = subject_window_index[row.ID]
    theta_env = subject_envelopes[row.ID]["theta"]
    theta_matrices = [safe_corrcoef(theta_env[start : start + SELECTED_WINDOW]) for start in starts]
    theta_stack = np.stack(theta_matrices).astype(np.float32)
    subject_connectivity_windows[row.ID] = {"theta": theta_stack}
    subject_connectivity_store[row.ID] = {"theta_static": theta_stack.mean(axis=0)}

theta_shape_df = pd.DataFrame(
    [{"ID": sid, "theta_window_tensor_shape": str(values["theta"].shape)} for sid, values in subject_connectivity_windows.items()]
)
display(theta_shape_df.head(12))

example_theta_matrix = subject_connectivity_windows[example_control_id]["theta"][0]
fig, ax = plt.subplots(figsize=(15, 14), constrained_layout=True)
plot_matrix_on_axis(
    ax,
    example_theta_matrix,
    title=f"Theta amplitude-envelope connectivity example ({example_control_id})",
    x_labels=CHANNELS,
    y_labels=CHANNELS,
    cmap="coolwarm",
    center=0.0,
    vmin=-1.0,
    vmax=1.0,
    colorbar_label="Pearson r",
)
saved_path = save_figure(fig, "theta_envelope_connectivity_example_matrix.png")
print(f"Saved theta example matrix to {saved_path}")


,ID,theta_window_tensor_shape
0,v123,"(55, 19, 19)"
1,v134,"(51, 19, 19)"
2,v138,"(47, 19, 19)"
3,v140,"(66, 19, 19)"
4,v298,"(68, 19, 19)"
5,v309,"(95, 19, 19)"
6,v310,"(68, 19, 19)"
7,v52p,"(53, 19, 19)"
8,v60p,"(49, 19, 19)"
9,v177,"(64, 19, 19)"


Figure saved at OLD_Docs/outputs_reviewer_ready_v2/main_figures/theta_envelope_connectivity_example_matrix.png

Saved theta example matrix to /Users/talgatazykanov/Desktop/Science works/Makbal_PhD/outputs_reviewer_ready_v2/main_figures/theta_envelope_connectivity_example_matrix.png


In [17]:
# Cell 17: Compute beta-band amplitude-envelope connectivity matrices.
for row in subject_split_df.itertuples(index=False):
    starts = subject_window_index[row.ID]
    beta_env = subject_envelopes[row.ID]["beta"]
    beta_matrices = [safe_corrcoef(beta_env[start : start + SELECTED_WINDOW]) for start in starts]
    beta_stack = np.stack(beta_matrices).astype(np.float32)
    subject_connectivity_windows[row.ID]["beta"] = beta_stack
    subject_connectivity_store[row.ID]["beta_static"] = beta_stack.mean(axis=0)

beta_shape_df = pd.DataFrame(
    [{"ID": sid, "beta_window_tensor_shape": str(values["beta"].shape)} for sid, values in subject_connectivity_windows.items()]
)
display(beta_shape_df.head(12))

example_beta_matrix = subject_connectivity_windows[example_adhd_id]["beta"][0]
fig, ax = plt.subplots(figsize=(15, 14), constrained_layout=True)
plot_matrix_on_axis(
    ax,
    example_beta_matrix,
    title=f"Beta amplitude-envelope connectivity example ({example_adhd_id})",
    x_labels=CHANNELS,
    y_labels=CHANNELS,
    cmap="coolwarm",
    center=0.0,
    vmin=-1.0,
    vmax=1.0,
    colorbar_label="Pearson r",
)
saved_path = save_figure(fig, "beta_envelope_connectivity_example_matrix.png")
print(f"Saved beta example matrix to {saved_path}")


,ID,beta_window_tensor_shape
0,v123,"(55, 19, 19)"
1,v134,"(51, 19, 19)"
2,v138,"(47, 19, 19)"
3,v140,"(66, 19, 19)"
4,v298,"(68, 19, 19)"
5,v309,"(95, 19, 19)"
6,v310,"(68, 19, 19)"
7,v52p,"(53, 19, 19)"
8,v60p,"(49, 19, 19)"
9,v177,"(64, 19, 19)"


Figure saved at OLD_Docs/outputs_reviewer_ready_v2/main_figures/beta_envelope_connectivity_example_matrix.png

Saved beta example matrix to /Users/talgatazykanov/Desktop/Science works/Makbal_PhD/outputs_reviewer_ready_v2/main_figures/beta_envelope_connectivity_example_matrix.png


In [18]:
# Cell 18: Construct the original joint theta+beta connectivity representation.
fusion_window_rows = []
for row in subject_split_df.itertuples(index=False):
    theta_stack = subject_connectivity_windows[row.ID]["theta"]
    beta_stack = subject_connectivity_windows[row.ID]["beta"]
    fusion_stack = 0.5 * (np.abs(theta_stack) + np.abs(beta_stack))
    for matrix in fusion_stack:
        np.fill_diagonal(matrix, 0.0)
    subject_connectivity_windows[row.ID]["fusion"] = fusion_stack.astype(np.float32)
    subject_connectivity_store[row.ID]["fusion_static"] = fusion_stack.mean(axis=0).astype(np.float32)
    fusion_window_rows.append(
        {
            "ID": row.ID,
            "split": row.split,
            "class_name": row.class_name,
            "fusion_window_tensor_shape": str(fusion_stack.shape),
        }
    )

fusion_shape_df = pd.DataFrame(fusion_window_rows)
print("Final window-level tensors by subject")
display(fusion_shape_df.head(12))
display(
    fusion_shape_df.groupby(["split", "class_name"]).size().rename("n_subjects").reset_index()
)


Final window-level tensors by subject


,ID,split,class_name,fusion_window_tensor_shape
0,v123,test,Control,"(55, 19, 19)"
1,v134,test,Control,"(51, 19, 19)"
2,v138,test,Control,"(47, 19, 19)"
3,v140,test,Control,"(66, 19, 19)"
4,v298,test,Control,"(68, 19, 19)"
5,v309,test,Control,"(95, 19, 19)"
6,v310,test,Control,"(68, 19, 19)"
7,v52p,test,Control,"(53, 19, 19)"
8,v60p,test,Control,"(49, 19, 19)"
9,v177,test,ADHD,"(64, 19, 19)"


,split,class_name,n_subjects
0,test,ADHD,10
1,test,Control,9
2,train,ADHD,42
3,train,Control,42
4,validation,ADHD,9
5,validation,Control,9


In [19]:
# Cell 19: Build dynamic sequences from window-level connectivity matrices.
sequence_graphs = []
sequence_edges = []
sequence_labels = []
sequence_subject_ids = []
sequence_splits = []
sequence_start_windows = []
window_edge_rows = []
window_edge_meta_rows = []

for row in subject_split_df.itertuples(index=False):
    fusion_stack = subject_connectivity_windows[row.ID]["fusion"]
    theta_stack = subject_connectivity_windows[row.ID]["theta"]
    beta_stack = subject_connectivity_windows[row.ID]["beta"]
    theta_edge_windows = theta_stack[:, TRIU_IDX[0], TRIU_IDX[1]]
    beta_edge_windows = beta_stack[:, TRIU_IDX[0], TRIU_IDX[1]]
    edge_windows = np.concatenate([theta_edge_windows, beta_edge_windows], axis=1).astype(np.float32)

    for window_index, fusion_matrix in enumerate(fusion_stack):
        window_edge_rows.append(fusion_matrix[TRIU_IDX])
        window_edge_meta_rows.append(
            {
                "ID": row.ID,
                "split": row.split,
                "label": row.label,
                "class_name": row.class_name,
                "window_index": int(window_index),
            }
        )

    if fusion_stack.shape[0] < SEQ_LEN:
        continue
    for start_w in range(0, fusion_stack.shape[0] - SEQ_LEN + 1, SEQ_STRIDE):
        end_w = start_w + SEQ_LEN
        sequence_graphs.append(fusion_stack[start_w:end_w])
        sequence_edges.append(edge_windows[start_w:end_w])
        sequence_labels.append(int(row.label))
        sequence_subject_ids.append(row.ID)
        sequence_splits.append(row.split)
        sequence_start_windows.append(int(start_w))

X_seq_graph = np.stack(sequence_graphs).astype(np.float32)
X_seq_edges = np.stack(sequence_edges).astype(np.float32)
y_seq = np.asarray(sequence_labels, dtype=int)
sequence_meta_df = pd.DataFrame(
    {
        "sequence_id": np.arange(len(y_seq)),
        "ID": sequence_subject_ids,
        "split": sequence_splits,
        "label": y_seq,
        "start_window": sequence_start_windows,
    }
)
sequence_meta_df["class_name"] = sequence_meta_df["label"].map(LABEL_NAME_MAP)
window_meta_df = pd.DataFrame(window_edge_meta_rows)
all_window_edges = np.stack(window_edge_rows).astype(np.float32)

mask_train = sequence_meta_df["split"] == "train"
mask_val = sequence_meta_df["split"] == "validation"
mask_test = sequence_meta_df["split"] == "test"
X_seq_graph_train = X_seq_graph[mask_train.to_numpy()]
X_seq_graph_val = X_seq_graph[mask_val.to_numpy()]
X_seq_graph_test = X_seq_graph[mask_test.to_numpy()]
X_seq_edges_train = X_seq_edges[mask_train.to_numpy()]
X_seq_edges_val = X_seq_edges[mask_val.to_numpy()]
X_seq_edges_test = X_seq_edges[mask_test.to_numpy()]
y_train = y_seq[mask_train.to_numpy()]
y_val = y_seq[mask_val.to_numpy()]
y_test = y_seq[mask_test.to_numpy()]

sequence_count_df = (
    sequence_meta_df.groupby(["split", "class_name"])
    .agg(n_sequences=("sequence_id", "count"), n_subjects=("ID", "nunique"))
    .reset_index()
)
print(f"X_seq_graph shape: {X_seq_graph.shape}")
print(f"X_seq_edges shape: {X_seq_edges.shape}")
display(sequence_count_df)
sequence_count_df.to_csv(TABLES_DIR / "sequence_counts_by_split_class.csv", index=False)


X_seq_graph shape: (1486, 10, 19, 19)
X_seq_edges shape: (1486, 10, 342)


,split,class_name,n_sequences,n_subjects
0,test,ADHD,117,10
1,test,Control,98,9
2,train,ADHD,598,42
3,train,Control,451,42
4,validation,ADHD,125,9
5,validation,Control,97,9


In [20]:
# Cell 20: Create standardized upper-triangular edge-vector representations using training statistics only.
edge_scaler = StandardScaler()
edge_scaler.fit(X_seq_edges_train.reshape(-1, X_seq_edges_train.shape[-1]))


def transform_edge_sequences(x: np.ndarray, scaler: StandardScaler = edge_scaler) -> np.ndarray:
    original_shape = x.shape
    transformed = scaler.transform(x.reshape(-1, original_shape[-1]))
    return transformed.reshape(original_shape).astype(np.float32)


X_seq_edges_train_std = transform_edge_sequences(X_seq_edges_train)
X_seq_edges_val_std = transform_edge_sequences(X_seq_edges_val)
X_seq_edges_test_std = transform_edge_sequences(X_seq_edges_test)

standardization_variable_df = pd.DataFrame(
    [
        ("edge_value", "The theta or beta upper-triangular envelope-connectivity edge value for one window."),
        ("train_mean", "The feature-wise mean estimated only from the training sequences."),
        ("train_standard_deviation", "The feature-wise standard deviation estimated only from the training sequences."),
        ("standardized_edge_value", "z = (edge_value - train_mean) / train_standard_deviation."),
    ],
    columns=["Variable", "Meaning"],
)
normalization_check_df = pd.DataFrame(
    [
        ("train", float(X_seq_edges_train_std.mean()), float(X_seq_edges_train_std.std())),
        ("validation", float(X_seq_edges_val_std.mean()), float(X_seq_edges_val_std.std())),
        ("test", float(X_seq_edges_test_std.mean()), float(X_seq_edges_test_std.std())),
    ],
    columns=["split", "global_mean_after_z", "global_std_after_z"],
)
display(standardization_variable_df)
display(normalization_check_df)
normalization_check_df.to_csv(TABLES_DIR / "edge_standardization_check.csv", index=False)


,Variable,Meaning
0,edge_value,The theta or beta upper-triangular envelope-co...
1,train_mean,The feature-wise mean estimated only from the ...
2,train_standard_deviation,The feature-wise standard deviation estimated ...
3,standardized_edge_value,z = (edge_value - train_mean) / train_standard...


,split,global_mean_after_z,global_std_after_z
0,train,1.088826e-09,1.000000
1,validation,-8.023945e-03,1.022539
2,test,-5.952660e-02,1.010573


## Section 6. Descriptive and Reviewer-Safe Exploratory Analysis


In [21]:
# Cell 21: Generate exploratory raw-amplitude correlation matrices, explicitly not used for final modeling.
raw_all_corr = safe_corrcoef(eeg_df[CHANNELS].to_numpy(dtype=np.float32))
raw_control_corr = safe_corrcoef(eeg_df.loc[eeg_df["label"] == 0, CHANNELS].to_numpy(dtype=np.float32))
raw_adhd_corr = safe_corrcoef(eeg_df.loc[eeg_df["label"] == 1, CHANNELS].to_numpy(dtype=np.float32))

fig, axes = plt.subplots(1, 3, figsize=(39, 13), constrained_layout=True)
for ax, matrix, title in [
    (axes[0], raw_all_corr, "All subjects\nExploratory raw-amplitude correlation"),
    (axes[1], raw_control_corr, "Control subjects\nExploratory raw-amplitude correlation"),
    (axes[2], raw_adhd_corr, "ADHD subjects\nExploratory raw-amplitude correlation"),
]:
    plot_matrix_on_axis(
        ax,
        matrix,
        title=title,
        x_labels=CHANNELS,
        y_labels=CHANNELS,
        cmap="coolwarm",
        center=0.0,
        vmin=-1.0,
        vmax=1.0,
        colorbar_label="Raw-amplitude Pearson r",
    )
saved_path = save_figure(fig, "exploratory_raw_amplitude_correlation_3panel.png")
display(
    pd.DataFrame(
        [
            ("Use in notebook", "Descriptive only"),
            ("Use in final model training", "No"),
            ("Saved figure", str(saved_path)),
        ],
        columns=["Item", "Value"],
    )
)


Figure saved at OLD_Docs/outputs_reviewer_ready_v2/main_figures/exploratory_raw_amplitude_correlation_3panel.png

,Item,Value
0,Use in notebook,Descriptive only
1,Use in final model training,No
2,Saved figure,/Users/talgatazykanov/Desktop/Science works/Ma...


In [22]:
# Cell 22: Generate final envelope-connectivity summaries from theta, beta, and joint theta+beta model inputs.
theta_mean_matrix = np.mean([store["theta_static"] for store in subject_connectivity_store.values()], axis=0)
beta_mean_matrix = np.mean([store["beta_static"] for store in subject_connectivity_store.values()], axis=0)
fusion_mean_matrix = np.mean([store["fusion_static"] for store in subject_connectivity_store.values()], axis=0)

fig, axes = plt.subplots(1, 3, figsize=(39, 13), constrained_layout=True)
plot_matrix_on_axis(
    axes[0],
    theta_mean_matrix,
    "Theta band-limited amplitude-envelope connectivity",
    CHANNELS,
    CHANNELS,
    cmap="coolwarm",
    center=0.0,
    vmin=-1.0,
    vmax=1.0,
    colorbar_label="Pearson r",
)
plot_matrix_on_axis(
    axes[1],
    beta_mean_matrix,
    "Beta band-limited amplitude-envelope connectivity",
    CHANNELS,
    CHANNELS,
    cmap="coolwarm",
    center=0.0,
    vmin=-1.0,
    vmax=1.0,
    colorbar_label="Pearson r",
)
plot_matrix_on_axis(
    axes[2],
    fusion_mean_matrix,
    "Joint theta+beta amplitude-envelope connectivity",
    CHANNELS,
    CHANNELS,
    cmap="viridis",
    center=None,
    vmin=0.0,
    vmax=float(fusion_mean_matrix.max()),
    colorbar_label="Mean absolute connectivity",
)
saved_path = save_figure(fig, "final_envelope_connectivity_theta_beta_joint_3panel.png")
display(pd.DataFrame([("Final modeling representation", "Band-limited Hilbert amplitude-envelope connectivity"), ("Saved figure", str(saved_path))], columns=["Item", "Value"]))


Figure saved at OLD_Docs/outputs_reviewer_ready_v2/main_figures/final_envelope_connectivity_theta_beta_joint_3panel.png

,Item,Value
0,Final modeling representation,Band-limited Hilbert amplitude-envelope connec...
1,Saved figure,/Users/talgatazykanov/Desktop/Science works/Ma...


In [23]:
# Cell 23: Generate relative spectral power comparison as a control/ADHD/difference 3-panel figure.
def bandpower_from_psd(freqs: np.ndarray, psd: np.ndarray, band: Tuple[float, float]) -> float:
    mask = (freqs >= band[0]) & (freqs <= band[1])
    if not np.any(mask):
        return 0.0
    return float(np.trapezoid(psd[mask], freqs[mask]))


bandpower_records = []
theta_beta_records = []
for row in subject_split_df.itertuples(index=False):
    x = preprocessed_subject_arrays[row.ID]
    for channel_index, channel_name in enumerate(CHANNELS):
        freqs, psd = signal.welch(
            x[:, channel_index],
            fs=FS_HZ,
            nperseg=min(1024, x.shape[0]),
            noverlap=min(512, max(0, x.shape[0] - 1)),
            detrend="constant",
        )
        abs_powers = {band_name: bandpower_from_psd(freqs, psd, band_range) for band_name, band_range in EEG_BANDS.items()}
        total_power = sum(abs_powers.values()) + 1e-12
        for band_name, abs_power in abs_powers.items():
            bandpower_records.append(
                {
                    "ID": row.ID,
                    "split": row.split,
                    "class_name": row.class_name,
                    "label": row.label,
                    "channel": channel_name,
                    "band": band_name,
                    "rel_power": abs_power / total_power,
                }
            )
        theta_beta_records.append(
            {
                "ID": row.ID,
                "split": row.split,
                "class_name": row.class_name,
                "label": row.label,
                "channel": channel_name,
                "theta_beta_ratio": abs_powers["theta"] / (abs_powers["beta"] + 1e-12),
            }
        )

bandpower_df = pd.DataFrame(bandpower_records)
theta_beta_df = pd.DataFrame(theta_beta_records)
rel_power_control = (
    bandpower_df[bandpower_df["label"] == 0].groupby(["channel", "band"])["rel_power"].mean().unstack("band").reindex(index=CHANNELS, columns=list(EEG_BANDS))
)
rel_power_adhd = (
    bandpower_df[bandpower_df["label"] == 1].groupby(["channel", "band"])["rel_power"].mean().unstack("band").reindex(index=CHANNELS, columns=list(EEG_BANDS))
)
rel_power_diff = rel_power_adhd - rel_power_control
vmax_rel = float(max(rel_power_control.to_numpy().max(), rel_power_adhd.to_numpy().max()))
vmax_diff = float(np.abs(rel_power_diff.to_numpy()).max())

fig, axes = plt.subplots(1, 3, figsize=(27, 16), constrained_layout=True)
plot_matrix_on_axis(axes[0], rel_power_control.to_numpy(), "Control relative spectral power", list(EEG_BANDS), CHANNELS, cmap="viridis", center=None, vmin=0.0, vmax=vmax_rel, colorbar_label="Relative power")
plot_matrix_on_axis(axes[1], rel_power_adhd.to_numpy(), "ADHD relative spectral power", list(EEG_BANDS), CHANNELS, cmap="viridis", center=None, vmin=0.0, vmax=vmax_rel, colorbar_label="Relative power")
plot_matrix_on_axis(axes[2], rel_power_diff.to_numpy(), "Difference ADHD - Control", list(EEG_BANDS), CHANNELS, cmap="coolwarm", center=0.0, vmin=-vmax_diff, vmax=vmax_diff, colorbar_label="Relative-power difference")
saved_path = save_figure(fig, "spectral_power_control_adhd_difference_3panel.png")
display(pd.DataFrame([("Saved figure", str(saved_path)), ("Power definition", "Relative band power from Welch PSD")], columns=["Item", "Value"]))


Figure saved at OLD_Docs/outputs_reviewer_ready_v2/main_figures/spectral_power_control_adhd_difference_3panel.png

,Item,Value
0,Saved figure,/Users/talgatazykanov/Desktop/Science works/Ma...
1,Power definition,Relative band power from Welch PSD


## Section 7. Model Definitions


In [24]:
# Cell 24: Define the Linear Network Index model exactly as the original logistic linear pipeline.
def make_linear_network_index_model() -> Pipeline:
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=3000, random_state=SEED)),
        ]
    )


linear_model_definition_df = pd.DataFrame(
    [
        ("Model family", "Linear Network Index"),
        ("Input", "Mean upper-triangular theta+beta envelope-connectivity edge vector across sequence windows"),
        ("Estimator", "LogisticRegression"),
        ("max_iter", 3000),
        ("random_state", SEED),
    ],
    columns=["Component", "Original setting"],
)
display(linear_model_definition_df)


,Component,Original setting
0,Model family,Linear Network Index
1,Input,Mean upper-triangular theta+beta envelope-conn...
2,Estimator,LogisticRegression
3,max_iter,3000
4,random_state,42


In [25]:
# Cell 25: Define the Sparse Linear / Sparse Oscillatory Index model exactly as the original L1 logistic pipeline.
def make_sparse_oscillatory_index_model() -> Pipeline:
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=5000, penalty="l1", solver="liblinear", C=0.25, random_state=SEED)),
        ]
    )


sparse_model_definition_df = pd.DataFrame(
    [
        ("Model family", "Sparse Linear / Sparse Oscillatory Index"),
        ("Input", "Mean upper-triangular theta+beta envelope-connectivity edge vector across sequence windows"),
        ("Estimator", "L1-regularized LogisticRegression"),
        ("solver", "liblinear"),
        ("C", 0.25),
        ("max_iter", 5000),
        ("random_state", SEED),
    ],
    columns=["Component", "Original setting"],
)
display(sparse_model_definition_df)


,Component,Original setting
0,Model family,Sparse Linear / Sparse Oscillatory Index
1,Input,Mean upper-triangular theta+beta envelope-conn...
2,Estimator,L1-regularized LogisticRegression
3,solver,liblinear
4,C,0.25
5,max_iter,5000
6,random_state,42


In [26]:
# Cell 26: Define the Global Graph-Metric Network Index model and graph metrics used in the original notebook.
def compute_window_graph_metrics(matrix: np.ndarray, threshold: float = SELECTED_THRESHOLD) -> Dict[str, float]:
    adjacency = np.abs(matrix).astype(float).copy()
    np.fill_diagonal(adjacency, 0.0)
    adjacency[adjacency < threshold] = 0.0
    graph = nx.from_numpy_array(adjacency)
    strengths = np.array([value for _, value in graph.degree(weight="weight")], dtype=float)
    clustering = np.array(list(nx.clustering(graph, weight="weight").values()), dtype=float)
    density = float((adjacency > 0).sum() / (adjacency.shape[0] * (adjacency.shape[0] - 1)))
    return {
        "mean_strength": float(strengths.mean()) if strengths.size else 0.0,
        "mean_clustering": float(clustering.mean()) if clustering.size else 0.0,
        "global_efficiency": float(nx.global_efficiency(graph)) if graph.number_of_nodes() > 1 else 0.0,
        "density": density,
    }


def build_sequence_graph_metric_features(x_seq_graph: np.ndarray, threshold: float = SELECTED_THRESHOLD) -> np.ndarray:
    rows = []
    for seq in x_seq_graph:
        metric_rows = [compute_window_graph_metrics(window, threshold) for window in seq]
        metric_df = pd.DataFrame(metric_rows)
        rows.append(
            [
                float(metric_df["mean_strength"].mean()),
                float(metric_df["mean_strength"].std(ddof=1)),
                float(metric_df["mean_clustering"].mean()),
                float(metric_df["mean_clustering"].std(ddof=1)),
                float(metric_df["global_efficiency"].mean()),
                float(metric_df["global_efficiency"].std(ddof=1)),
                float(metric_df["density"].mean()),
            ]
        )
    return np.asarray(rows, dtype=np.float32)


def make_global_graph_metric_network_index_model() -> Pipeline:
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=3000, random_state=SEED)),
        ]
    )


graph_metric_definition_df = pd.DataFrame(
    [
        ("mean_strength", "Mean weighted node strength after thresholding"),
        ("mean_clustering", "Mean weighted clustering coefficient"),
        ("global_efficiency", "NetworkX global efficiency"),
        ("density", "Fraction of nonzero directed off-diagonal entries after thresholding"),
        ("threshold", SELECTED_THRESHOLD),
    ],
    columns=["Metric", "Definition"],
)
display(graph_metric_definition_df)


,Metric,Definition
0,mean_strength,Mean weighted node strength after thresholding
1,mean_clustering,Mean weighted clustering coefficient
2,global_efficiency,NetworkX global efficiency
3,density,Fraction of nonzero directed off-diagonal entr...
4,threshold,0.4


In [27]:
# Cell 27: Define the GRU Dynamic Connectivity Index model exactly as in the original notebook.
class GRUNetworkIndex(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 64, dropout: float = 0.25) -> None:
        super().__init__()
        self.gru = nn.GRU(input_size=input_dim, hidden_size=hidden_dim, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h, _ = self.gru(x)
        h_last = self.dropout(h[:, -1, :])
        return self.fc(h_last).squeeze(-1)


gru_definition_df = pd.DataFrame(
    [
        ("Input", "Standardized theta+beta edge time series"),
        ("GRU hidden size", 64),
        ("GRU layers", 1),
        ("Dropout", 0.25),
        ("Output", "Scalar network index logit"),
    ],
    columns=["Component", "Original setting"],
)
display(gru_definition_df)


,Component,Original setting
0,Input,Standardized theta+beta edge time series
1,GRU hidden size,64
2,GRU layers,1
3,Dropout,0.25
4,Output,Scalar network index logit


In [28]:
# Cell 28: Define the Static GCN Brain Network Index model exactly as in the original notebook.
class SimpleGCNLayer(nn.Module):
    def __init__(self, in_features: int, out_features: int) -> None:
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        batch_size, n_nodes, _ = adj.shape
        eye = torch.eye(n_nodes, device=adj.device).unsqueeze(0).expand(batch_size, -1, -1)
        a_hat = torch.clamp(adj + eye, min=0.0)
        deg = torch.clamp(a_hat.sum(dim=-1), min=1e-6)
        deg_inv_sqrt = torch.pow(deg, -0.5)
        d_inv_sqrt = torch.diag_embed(deg_inv_sqrt)
        a_norm = d_inv_sqrt @ a_hat @ d_inv_sqrt
        return a_norm @ self.linear(x)


class StaticGCNNetworkIndex(nn.Module):
    def __init__(self, n_nodes: int, hidden_dim: int = 32, dropout: float = 0.20) -> None:
        super().__init__()
        self.gcn1 = SimpleGCNLayer(1, hidden_dim)
        self.gcn2 = SimpleGCNLayer(hidden_dim, hidden_dim)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, adj: torch.Tensor) -> torch.Tensor:
        batch_size, n_nodes, _ = adj.shape
        node_features = adj.sum(dim=-1, keepdim=True)
        h = self.dropout(self.act(self.gcn1(node_features, adj)))
        h = self.dropout(self.act(self.gcn2(h, adj)))
        graph_embedding = h.mean(dim=1)
        return self.fc(graph_embedding).squeeze(-1)


static_gcn_definition_df = pd.DataFrame(
    [
        ("Input", "Mean fusion adjacency matrix across the sequence"),
        ("GCN layers", 2),
        ("Hidden size", 32),
        ("Dropout", 0.20),
        ("Pooling", "Mean node pooling"),
        ("Output", "Scalar network index logit"),
    ],
    columns=["Component", "Original setting"],
)
display(static_gcn_definition_df)


,Component,Original setting
0,Input,Mean fusion adjacency matrix across the sequence
1,GCN layers,2
2,Hidden size,32
3,Dropout,0.2
4,Pooling,Mean node pooling
5,Output,Scalar network index logit


In [29]:
# Cell 29: Define the Spatio-Temporal GCN Brain Network Index baseline.
class SpatioTemporalGCNNetworkIndex(nn.Module):
    def __init__(self, n_nodes: int, hidden_dim: int = 32, dropout: float = 0.20) -> None:
        super().__init__()
        self.gcn1 = SimpleGCNLayer(1, hidden_dim)
        self.gcn2 = SimpleGCNLayer(hidden_dim, hidden_dim)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)

    def encode_window(self, adj: torch.Tensor) -> torch.Tensor:
        node_features = adj.sum(dim=-1, keepdim=True)
        h = self.dropout(self.act(self.gcn1(node_features, adj)))
        h = self.dropout(self.act(self.gcn2(h, adj)))
        return h.mean(dim=1)

    def forward(self, adj_seq: torch.Tensor) -> torch.Tensor:
        embeddings = [self.encode_window(adj_seq[:, time_idx]) for time_idx in range(adj_seq.shape[1])]
        temporal_embedding = torch.stack(embeddings, dim=1).mean(dim=1)
        return self.fc(temporal_embedding).squeeze(-1)


st_gcn_definition_df = pd.DataFrame(
    [
        ("Input", "Fusion adjacency sequence"),
        ("Spatial block", "Same SimpleGCNLayer blocks and hidden size as the original Static GCN baseline"),
        ("Temporal summary", "Mean pooling across sequence windows"),
        ("Hidden size", 32),
        ("Dropout", 0.20),
        ("Interpretation", "Baseline extension only; it does not replace or alter the proposed model"),
    ],
    columns=["Component", "Setting"],
)
display(st_gcn_definition_df)


,Component,Setting
0,Input,Fusion adjacency sequence
1,Spatial block,Same SimpleGCNLayer blocks and hidden size as ...
2,Temporal summary,Mean pooling across sequence windows
3,Hidden size,32
4,Dropout,0.2
5,Interpretation,Baseline extension only; it does not replace o...


In [30]:
# Cell 30: Define the proposed Hybrid Multihead Spatio-Temporal Graph Transformer Index model exactly.
PROPOSED_MODEL_NAME = "Hybrid Multihead Spatio-Temporal graph transformer index model"
PROPOSED_MODEL_SHORT_NAME = "Proposed Hybrid ST Graph Transformer"
PROPOSED_ABLATION_SHORT_NAME = "Proposed model ablation"


class HybridMultiheadSpatioTemporalGraphTransformerIndexModel(nn.Module):
    def __init__(
        self,
        input_dim: int,
        d_model: int = 96,
        n_heads: int = 4,
        n_layers: int = 2,
        gru_hidden: int = 64,
        dropout: float = 0.20,
        max_seq_len: int = SEQ_LEN,
    ) -> None:
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.positional_embedding = nn.Parameter(torch.zeros(1, max_seq_len, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.time_attention = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.Tanh(),
            nn.Linear(d_model // 2, 1),
        )
        self.gru = nn.GRU(input_size=input_dim, hidden_size=gru_hidden, num_layers=1, batch_first=True)
        self.fusion_norm = nn.LayerNorm(d_model + gru_hidden)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(d_model + gru_hidden, 1)
        nn.init.normal_(self.positional_embedding, mean=0.0, std=0.02)

    def temporal_attention_pool(self, z: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        attention_logits = self.time_attention(z).squeeze(-1)
        attention_weights = torch.softmax(attention_logits, dim=1)
        pooled = torch.sum(z * attention_weights.unsqueeze(-1), dim=1)
        return pooled, attention_weights

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.size(1)
        z = self.input_proj(x)
        z = z + self.positional_embedding[:, :seq_len, :]
        z = self.encoder(z)
        g_tr, _ = self.temporal_attention_pool(z)
        _, h_n = self.gru(x)
        g_rnn = h_n[-1]
        fused = torch.cat([g_tr, g_rnn], dim=-1)
        fused = self.fusion_norm(fused)
        fused = self.dropout(fused)
        return self.fc(fused).squeeze(-1)


proposed_architecture_df = pd.DataFrame(
    [
        ("Input representation", "Standardized theta+beta upper-triangular envelope-connectivity edge sequence"),
        ("Transformer branch", "Linear edge projection, learned positional embedding, multi-head Transformer encoder"),
        ("Multi-head attention", "n_heads = 4 in the executed proposed configuration"),
        ("Temporal pooling", "Learned attention pooling over Transformer-encoded windows"),
        ("GRU branch", "GRU over the original standardized edge sequence"),
        ("Fusion", "Concatenation of Transformer summary and GRU final hidden state"),
        ("Normalization and dropout", "LayerNorm followed by dropout"),
        ("Output", "Single scalar network-index logit"),
        ("Loss", "BCEWithLogitsLoss during training"),
    ],
    columns=["Architecture block", "Preserved implementation"],
)
display(proposed_architecture_df)


,Architecture block,Preserved implementation
0,Input representation,Standardized theta+beta upper-triangular envel...
1,Transformer branch,"Linear edge projection, learned positional emb..."
2,Multi-head attention,n_heads = 4 in the executed proposed configura...
3,Temporal pooling,Learned attention pooling over Transformer-enc...
4,GRU branch,GRU over the original standardized edge sequence
5,Fusion,Concatenation of Transformer summary and GRU f...
6,Normalization and dropout,LayerNorm followed by dropout
7,Output,Single scalar network-index logit
8,Loss,BCEWithLogitsLoss during training


In [31]:
# Cell 31: Define the proposed model ablation configuration exactly as present in the original notebook.
proposed_ablation_configs = [
    (
        PROPOSED_MODEL_NAME,
        dict(d_model=64, n_heads=4, n_layers=1, gru_hidden=64, dropout=0.25),
        dict(lr=4e-4, weight_decay=3e-4, max_epochs=16, patience=5, seed=2025),
    ),
    (
        f"{PROPOSED_MODEL_NAME} (reduced-regularization ablation)",
        dict(d_model=64, n_heads=4, n_layers=1, gru_hidden=64, dropout=0.15),
        dict(lr=5e-4, weight_decay=1e-4, max_epochs=16, patience=5, seed=2025),
    ),
]
proposed_ablation_definition_df = pd.DataFrame(
    [
        {"model": name, **architecture_kwargs, **training_kwargs}
        for name, architecture_kwargs, training_kwargs in proposed_ablation_configs
    ]
)
display(proposed_ablation_definition_df)


,model,d_model,n_heads,n_layers,gru_hidden,dropout,lr,weight_decay,max_epochs,patience,seed
0,Hybrid Multihead Spatio-Temporal graph transfo...,64,4,1,64,0.25,0.0004,0.0003,16,5,2025
1,Hybrid Multihead Spatio-Temporal graph transfo...,64,4,1,64,0.15,0.0005,0.0001,16,5,2025


## Section 8. Training and Evaluation Helpers


In [32]:
# Cell 32: Define training loops, datasets, checkpoint logic, and training configuration.
SUBJECT_AGGREGATION_METHOD = "mean"
FIXED_DECISION_THRESHOLD = 0.0
FINAL_RANK_METRICS = ["balanced_accuracy", "accuracy", "auc_roc"]
BATCH_SIZE = 64

sequence_meta_train = sequence_meta_df[sequence_meta_df["split"] == "train"].reset_index(drop=True)
sequence_meta_val = sequence_meta_df[sequence_meta_df["split"] == "validation"].reset_index(drop=True)
sequence_meta_test = sequence_meta_df[sequence_meta_df["split"] == "test"].reset_index(drop=True)

model_score_store: Dict[str, Dict[str, np.ndarray]] = {}
model_subject_results: Dict[str, pd.DataFrame] = {}
model_histories: Dict[str, pd.DataFrame] = {}
trained_models: Dict[str, object] = {}
training_source_rows = []


class EdgeSequenceDataset(Dataset):
    def __init__(self, x: np.ndarray, y: np.ndarray) -> None:
        self.x = torch.from_numpy(x.astype(np.float32))
        self.y = torch.from_numpy(y.astype(np.float32))

    def __len__(self) -> int:
        return self.x.shape[0]

    def __getitem__(self, idx: int):
        return self.x[idx], self.y[idx]


class GraphDataset(Dataset):
    def __init__(self, x: np.ndarray, y: np.ndarray) -> None:
        self.x = torch.from_numpy(x.astype(np.float32))
        self.y = torch.from_numpy(y.astype(np.float32))

    def __len__(self) -> int:
        return self.x.shape[0]

    def __getitem__(self, idx: int):
        return self.x[idx], self.y[idx]


class GraphSequenceDataset(Dataset):
    def __init__(self, x: np.ndarray, y: np.ndarray) -> None:
        self.x = torch.from_numpy(x.astype(np.float32))
        self.y = torch.from_numpy(y.astype(np.float32))

    def __len__(self) -> int:
        return self.x.shape[0]

    def __getitem__(self, idx: int):
        return self.x[idx], self.y[idx]


def collect_scores(model: nn.Module, loader: DataLoader) -> np.ndarray:
    model.eval()
    outputs = []
    with torch.no_grad():
        for batch_x, _ in loader:
            batch_x = batch_x.to(DEVICE)
            outputs.append(model(batch_x).cpu().numpy())
    return np.concatenate(outputs, axis=0).astype(np.float32)


def train_binary_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    lr: float = 1e-3,
    weight_decay: float = 0.0,
    max_epochs: int = 12,
    patience: int = 3,
) -> Tuple[nn.Module, pd.DataFrame]:
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_state = None
    best_val_loss = float("inf")
    best_epoch = 0
    history = []
    for epoch in range(1, max_epochs + 1):
        model.train()
        train_loss_sum = 0.0
        train_count = 0
        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(DEVICE)
            batch_y = batch_y.to(DEVICE)
            optimizer.zero_grad()
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            train_loss_sum += float(loss.item()) * batch_x.size(0)
            train_count += batch_x.size(0)
        train_loss = train_loss_sum / max(train_count, 1)
        model.eval()
        val_loss_sum = 0.0
        val_count = 0
        val_outputs = []
        val_targets = []
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x = batch_x.to(DEVICE)
                batch_y = batch_y.to(DEVICE)
                logits = model(batch_x)
                loss = criterion(logits, batch_y)
                val_loss_sum += float(loss.item()) * batch_x.size(0)
                val_count += batch_x.size(0)
                val_outputs.append(logits.cpu().numpy())
                val_targets.append(batch_y.cpu().numpy())
        val_loss = val_loss_sum / max(val_count, 1)
        val_scores = np.concatenate(val_outputs).ravel()
        val_targets_np = np.concatenate(val_targets).ravel()
        val_auc = float(roc_auc_score(val_targets_np, val_scores)) if np.unique(val_targets_np).size == 2 else float("nan")
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_auc": val_auc})
        print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_auc={val_auc:.4f}")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        elif epoch - best_epoch >= patience:
            print("Early stopping triggered.")
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, pd.DataFrame(history)


def train_binary_model_by_subject_consensus(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    meta_val: pd.DataFrame,
    lr: float,
    weight_decay: float,
    max_epochs: int,
    patience: int,
    aggregation: str = SUBJECT_AGGREGATION_METHOD,
) -> Tuple[nn.Module, pd.DataFrame]:
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_state = None
    best_score = -float("inf")
    best_epoch = 0
    history = []
    for epoch in range(1, max_epochs + 1):
        model.train()
        train_loss_sum = 0.0
        train_count = 0
        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(DEVICE)
            batch_y = batch_y.to(DEVICE)
            optimizer.zero_grad()
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            train_loss_sum += float(loss.item()) * batch_x.size(0)
            train_count += batch_x.size(0)
        train_loss = train_loss_sum / max(train_count, 1)
        model.eval()
        val_loss_sum = 0.0
        val_count = 0
        val_outputs = []
        val_targets = []
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x = batch_x.to(DEVICE)
                batch_y = batch_y.to(DEVICE)
                logits = model(batch_x)
                loss = criterion(logits, batch_y)
                val_loss_sum += float(loss.item()) * batch_x.size(0)
                val_count += batch_x.size(0)
                val_outputs.append(logits.cpu().numpy())
                val_targets.append(batch_y.cpu().numpy())
        val_loss = val_loss_sum / max(val_count, 1)
        val_scores = np.concatenate(val_outputs).ravel()
        val_targets_np = np.concatenate(val_targets).ravel()
        val_sequence_auc = float(roc_auc_score(val_targets_np, val_scores)) if np.unique(val_targets_np).size == 2 else float("nan")
        val_subject_df = aggregate_subject_scores(meta_val, val_scores, aggregation)
        val_subject_metrics = compute_classification_metrics(
            val_subject_df["label"].to_numpy(),
            val_subject_df["score"].to_numpy(),
            FIXED_DECISION_THRESHOLD,
        )
        validation_composite = 0.55 * val_subject_metrics["balanced_accuracy"] + 0.45 * val_subject_metrics["auc_roc"]
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "val_sequence_auc": val_sequence_auc,
                "val_subject_accuracy": val_subject_metrics["accuracy"],
                "val_subject_bacc": val_subject_metrics["balanced_accuracy"],
                "val_subject_auc": val_subject_metrics["auc_roc"],
                "val_composite": validation_composite,
            }
        )
        print(
            f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
            f"val_subject_bacc={val_subject_metrics['balanced_accuracy']:.4f} | "
            f"val_subject_auc={val_subject_metrics['auc_roc']:.4f} | val_composite={validation_composite:.4f}"
        )
        if validation_composite > best_score:
            best_score = validation_composite
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        elif epoch - best_epoch >= patience:
            print("Early stopping triggered.")
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, pd.DataFrame(history)


def train_or_load_torch_model(
    artifact_stem: str,
    factory: Callable[[], nn.Module],
    train_callable: Callable[..., Tuple[nn.Module, pd.DataFrame]],
    train_kwargs: Dict[str, object],
) -> Tuple[nn.Module, pd.DataFrame, str]:
    state_path = MODELS_DIR / f"{artifact_stem}.pt"
    history_path = MODELS_DIR / f"{artifact_stem}_history.csv"
    model = factory().to(DEVICE)
    if state_path.exists() and history_path.exists():
        model.load_state_dict(torch.load(state_path, map_location=DEVICE))
        history_df = pd.read_csv(history_path)
        source = "loaded"
    else:
        model, history_df = train_callable(model=model, **train_kwargs)
        torch.save(model.state_dict(), state_path)
        history_df.to_csv(history_path, index=False)
        source = "trained"
    training_source_rows.append({"artifact": artifact_stem, "source": source, "state_path": str(state_path)})
    return model, history_df, source


training_config_df = pd.DataFrame(
    [
        ("Batch size", BATCH_SIZE),
        ("GRU optimizer", "Adam"),
        ("GRU learning rate", "1e-3"),
        ("GRU weight decay", "1e-4"),
        ("GRU maximum epochs", 12),
        ("GRU early-stopping patience", 3),
        ("Proposed optimizer", "AdamW"),
        ("Proposed learning rate", "4e-4"),
        ("Proposed weight decay", "3e-4"),
        ("Proposed maximum epochs", 16),
        ("Proposed early-stopping patience", 5),
        ("Proposed checkpoint criterion", "0.55 * validation subject balanced accuracy + 0.45 * validation subject AUC"),
        ("Subject aggregation", SUBJECT_AGGREGATION_METHOD),
        ("Fixed decision threshold", FIXED_DECISION_THRESHOLD),
    ],
    columns=["Training parameter", "Value"],
)
display(training_config_df)
training_config_df.to_csv(TABLES_DIR / "training_configuration.csv", index=False)


,Training parameter,Value
0,Batch size,64
1,GRU optimizer,Adam
2,GRU learning rate,1e-3
3,GRU weight decay,1e-4
4,GRU maximum epochs,12
5,GRU early-stopping patience,3
6,Proposed optimizer,AdamW
7,Proposed learning rate,4e-4
8,Proposed weight decay,3e-4
9,Proposed maximum epochs,16


In [33]:
# Cell 33: Define classification metric computation and subject-level evaluation.
def compute_classification_metrics(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> Dict[str, float]:
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    preds = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
    specificity = tn / max(tn + fp, 1)
    auc = float(roc_auc_score(y_true, scores)) if np.unique(y_true).size == 2 else float("nan")
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, preds)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, preds)),
        "sensitivity": float(recall_score(y_true, preds, pos_label=1, zero_division=0)),
        "specificity": float(specificity),
        "precision": float(precision_score(y_true, preds, zero_division=0)),
        "f1": float(f1_score(y_true, preds, zero_division=0)),
        "auc_roc": auc,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def aggregate_subject_scores(meta_df: pd.DataFrame, scores: np.ndarray, aggregation: str = SUBJECT_AGGREGATION_METHOD) -> pd.DataFrame:
    subject_df = meta_df[["ID", "label"]].copy()
    subject_df["score"] = np.asarray(scores, dtype=float)

    def aggregate_fn(values: pd.Series) -> float:
        arr = values.to_numpy(dtype=float)
        if aggregation == "mean":
            return float(arr.mean())
        if aggregation == "median":
            return float(np.median(arr))
        if aggregation == "max":
            return float(arr.max())
        raise ValueError(f"Unsupported aggregation: {aggregation}")

    aggregated = subject_df.groupby("ID").agg(label=("label", "first"), score=("score", aggregate_fn), n_sequences=("score", "size")).reset_index()
    aggregated["class_name"] = aggregated["label"].map(LABEL_NAME_MAP)
    return aggregated


def evaluate_subject_level_scores(
    model_name: str,
    meta_train: pd.DataFrame,
    scores_train: np.ndarray,
    meta_val: pd.DataFrame,
    scores_val: np.ndarray,
    meta_test: pd.DataFrame,
    scores_test: np.ndarray,
    threshold: float = FIXED_DECISION_THRESHOLD,
    aggregation: str = SUBJECT_AGGREGATION_METHOD,
) -> pd.DataFrame:
    rows = []
    for split_name, meta_df, scores in [
        ("train", meta_train, scores_train),
        ("validation", meta_val, scores_val),
        ("test", meta_test, scores_test),
    ]:
        aggregated = aggregate_subject_scores(meta_df, scores, aggregation)
        row = {"model": model_name, "split": split_name, "n_subjects": int(aggregated.shape[0]), "subject_aggregation": aggregation}
        row.update(compute_group_separation(aggregated["score"].to_numpy(), aggregated["label"].to_numpy()))
        row.update(compute_classification_metrics(aggregated["label"].to_numpy(), aggregated["score"].to_numpy(), threshold))
        rows.append(row)
    return pd.DataFrame(rows)


classification_metric_spec_df = pd.DataFrame(
    [
        ("accuracy", "Fraction of correctly classified subjects"),
        ("balanced_accuracy", "Mean of sensitivity and specificity"),
        ("sensitivity", "True-positive rate for ADHD"),
        ("specificity", "True-negative rate for Control"),
        ("precision", "Positive predictive value at the fixed operating threshold"),
        ("f1", "Harmonic mean of precision and sensitivity"),
        ("auc_roc", "Threshold-free ROC area under the score curve"),
    ],
    columns=["Metric", "Definition"],
)
display(classification_metric_spec_df)


,Metric,Definition
0,accuracy,Fraction of correctly classified subjects
1,balanced_accuracy,Mean of sensitivity and specificity
2,sensitivity,True-positive rate for ADHD
3,specificity,True-negative rate for Control
4,precision,Positive predictive value at the fixed operati...
5,f1,Harmonic mean of precision and sensitivity
6,auc_roc,Threshold-free ROC area under the score curve


In [34]:
# Cell 34: Define separability metrics separately from classification metrics.
def compute_group_separation(scores: np.ndarray, labels: np.ndarray) -> Dict[str, float]:
    scores = np.asarray(scores, dtype=float).ravel()
    labels = np.asarray(labels, dtype=int).ravel()
    s0 = scores[labels == 0]
    s1 = scores[labels == 1]
    mean0 = float(s0.mean()) if s0.size else float("nan")
    mean1 = float(s1.mean()) if s1.size else float("nan")
    std0 = float(s0.std(ddof=1)) if s0.size > 1 else 0.0
    std1 = float(s1.std(ddof=1)) if s1.size > 1 else 0.0
    pooled_num = (s0.size - 1) * (std0 ** 2) + (s1.size - 1) * (std1 ** 2)
    pooled_den = max(s0.size + s1.size - 2, 1)
    pooled_std = float(np.sqrt(pooled_num / pooled_den)) if pooled_num > 0 else 0.0
    delta_mean = mean1 - mean0
    cohens_d = delta_mean / pooled_std if pooled_std > 0 else 0.0
    separation_ratio = abs(delta_mean) / max(std0 + std1, 1e-12)
    pearson_r = float(np.corrcoef(scores, labels)[0, 1]) if np.unique(labels).size == 2 and scores.size > 1 else 0.0
    overlap = float(2.0 * stats.norm.cdf(-abs(cohens_d) / 2.0))
    silhouette = float(silhouette_score(scores.reshape(-1, 1), labels)) if scores.size > 2 and np.unique(labels).size == 2 else float("nan")
    return {
        "mean0": mean0,
        "mean1": mean1,
        "delta_mean": delta_mean,
        "separation_ratio": separation_ratio,
        "cohens_d": cohens_d,
        "pearson_r": pearson_r,
        "distribution_overlap": overlap,
        "silhouette": silhouette,
    }


separability_metric_spec_df = pd.DataFrame(
    [
        ("mean0", "Mean network-index score among controls"),
        ("mean1", "Mean network-index score among ADHD subjects"),
        ("delta_mean", "mean1 - mean0"),
        ("separation_ratio", "|delta_mean| divided by the sum of class standard deviations"),
        ("cohens_d", "Standardized mean difference"),
        ("pearson_r", "Point-biserial correlation between score and label"),
        ("distribution_overlap", "Normal-approximation distribution overlap from Cohen's d"),
        ("silhouette", "One-dimensional silhouette score on index values"),
    ],
    columns=["Metric", "Definition"],
)
display(separability_metric_spec_df)


,Metric,Definition
0,mean0,Mean network-index score among controls
1,mean1,Mean network-index score among ADHD subjects
2,delta_mean,mean1 - mean0
3,separation_ratio,|delta_mean| divided by the sum of class stand...
4,cohens_d,Standardized mean difference
5,pearson_r,Point-biserial correlation between score and l...
6,distribution_overlap,Normal-approximation distribution overlap from...
7,silhouette,One-dimensional silhouette score on index values


In [35]:
# Cell 35: Define subject-level stratified bootstrap confidence intervals.
def stratified_subject_bootstrap_metrics(
    subject_df: pd.DataFrame,
    threshold: float = FIXED_DECISION_THRESHOLD,
    n_bootstrap: int = N_BOOTSTRAP,
    seed: int = BOOTSTRAP_SEED,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    groups = {label: subject_df[subject_df["label"] == label].reset_index(drop=True) for label in sorted(subject_df["label"].unique())}
    metric_rows = []
    for _ in range(n_bootstrap):
        sampled_parts = []
        for label, group in groups.items():
            sampled_idx = rng.integers(0, len(group), size=len(group))
            sampled_parts.append(group.iloc[sampled_idx].copy())
        sampled_df = pd.concat(sampled_parts, ignore_index=True)
        metric_rows.append(compute_classification_metrics(sampled_df["label"].to_numpy(), sampled_df["score"].to_numpy(), threshold))
    return pd.DataFrame(metric_rows)


bootstrap_config_df = pd.DataFrame(
    [
        ("Bootstrap level", "Subject"),
        ("Stratification", "By class label where both classes are present"),
        ("Iterations", N_BOOTSTRAP),
        ("Random seed", BOOTSTRAP_SEED),
        ("Metrics", "accuracy, balanced accuracy, sensitivity, specificity, precision, F1, AUC-ROC"),
    ],
    columns=["Bootstrap item", "Value"],
)
display(bootstrap_config_df)


,Bootstrap item,Value
0,Bootstrap level,Subject
1,Stratification,By class label where both classes are present
2,Iterations,1000
3,Random seed,20260406
4,Metrics,"accuracy, balanced accuracy, sensitivity, spec..."


## Section 9. Model Training


In [36]:
# Cell 36: Train or load the Linear Network Index model and display split metrics.
X_lin_train = X_seq_edges_train.mean(axis=1)
X_lin_val = X_seq_edges_val.mean(axis=1)
X_lin_test = X_seq_edges_test.mean(axis=1)
linear_artifact_path = MODELS_DIR / "linear_network_index.pkl"
if linear_artifact_path.exists():
    with linear_artifact_path.open("rb") as f:
        linear_model = pickle.load(f)
    linear_source = "loaded"
else:
    linear_model = make_linear_network_index_model()
    linear_model.fit(X_lin_train, y_train)
    with linear_artifact_path.open("wb") as f:
        pickle.dump(linear_model, f)
    linear_source = "trained"

linear_score_store = {
    "train": linear_model.decision_function(X_lin_train),
    "validation": linear_model.decision_function(X_lin_val),
    "test": linear_model.decision_function(X_lin_test),
}
linear_subject_results_df = evaluate_subject_level_scores(
    "Linear network index",
    sequence_meta_train,
    linear_score_store["train"],
    sequence_meta_val,
    linear_score_store["validation"],
    sequence_meta_test,
    linear_score_store["test"],
)
model_score_store["Linear network index"] = linear_score_store
model_subject_results["Linear network index"] = linear_subject_results_df
trained_models["Linear network index"] = linear_model
training_source_rows.append({"artifact": "linear_network_index", "source": linear_source, "state_path": str(linear_artifact_path)})
display(linear_subject_results_df[["model", "split", "n_subjects", "accuracy", "balanced_accuracy", "sensitivity", "specificity", "precision", "f1", "auc_roc"]])


,model,split,n_subjects,accuracy,balanced_accuracy,sensitivity,specificity,precision,f1,auc_roc
0,Linear network index,train,84,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
1,Linear network index,validation,18,0.611111,0.611111,0.666667,0.555556,0.600000,0.631579,0.679012
2,Linear network index,test,19,0.526316,0.516667,0.700000,0.333333,0.538462,0.608696,0.466667


In [37]:
# Cell 37: Train or load the Sparse Linear / Sparse Oscillatory Index model and display split metrics.
sparse_artifact_path = MODELS_DIR / "sparse_oscillatory_network_index.pkl"
if sparse_artifact_path.exists():
    with sparse_artifact_path.open("rb") as f:
        sparse_model = pickle.load(f)
    sparse_source = "loaded"
else:
    sparse_model = make_sparse_oscillatory_index_model()
    sparse_model.fit(X_lin_train, y_train)
    with sparse_artifact_path.open("wb") as f:
        pickle.dump(sparse_model, f)
    sparse_source = "trained"

sparse_score_store = {
    "train": sparse_model.decision_function(X_lin_train),
    "validation": sparse_model.decision_function(X_lin_val),
    "test": sparse_model.decision_function(X_lin_test),
}
sparse_subject_results_df = evaluate_subject_level_scores(
    "Sparse oscillatory network index",
    sequence_meta_train,
    sparse_score_store["train"],
    sequence_meta_val,
    sparse_score_store["validation"],
    sequence_meta_test,
    sparse_score_store["test"],
)
model_score_store["Sparse oscillatory network index"] = sparse_score_store
model_subject_results["Sparse oscillatory network index"] = sparse_subject_results_df
trained_models["Sparse oscillatory network index"] = sparse_model
training_source_rows.append({"artifact": "sparse_oscillatory_network_index", "source": sparse_source, "state_path": str(sparse_artifact_path)})
display(sparse_subject_results_df[["model", "split", "n_subjects", "accuracy", "balanced_accuracy", "sensitivity", "specificity", "precision", "f1", "auc_roc"]])


,model,split,n_subjects,accuracy,balanced_accuracy,sensitivity,specificity,precision,f1,auc_roc
0,Sparse oscillatory network index,train,84,1.000000,1.000000,1.000000,1.000000,1.000,1.000000,1.000000
1,Sparse oscillatory network index,validation,18,0.611111,0.611111,0.555556,0.666667,0.625,0.588235,0.666667
2,Sparse oscillatory network index,test,19,0.473684,0.466667,0.600000,0.333333,0.500,0.545455,0.466667


In [38]:
# Cell 38: Train or load the Global Graph-Metric Network Index model and display split metrics.
X_topo_train = build_sequence_graph_metric_features(X_seq_graph_train)
X_topo_val = build_sequence_graph_metric_features(X_seq_graph_val)
X_topo_test = build_sequence_graph_metric_features(X_seq_graph_test)

global_artifact_path = MODELS_DIR / "global_graph_metric_network_index.pkl"
if global_artifact_path.exists():
    with global_artifact_path.open("rb") as f:
        global_model = pickle.load(f)
    global_source = "loaded"
else:
    global_model = make_global_graph_metric_network_index_model()
    global_model.fit(X_topo_train, y_train)
    with global_artifact_path.open("wb") as f:
        pickle.dump(global_model, f)
    global_source = "trained"

global_score_store = {
    "train": global_model.decision_function(X_topo_train),
    "validation": global_model.decision_function(X_topo_val),
    "test": global_model.decision_function(X_topo_test),
}
global_subject_results_df = evaluate_subject_level_scores(
    "Global graph-metric network index",
    sequence_meta_train,
    global_score_store["train"],
    sequence_meta_val,
    global_score_store["validation"],
    sequence_meta_test,
    global_score_store["test"],
)
model_score_store["Global graph-metric network index"] = global_score_store
model_subject_results["Global graph-metric network index"] = global_subject_results_df
trained_models["Global graph-metric network index"] = global_model
training_source_rows.append({"artifact": "global_graph_metric_network_index", "source": global_source, "state_path": str(global_artifact_path)})
display(global_subject_results_df[["model", "split", "n_subjects", "accuracy", "balanced_accuracy", "sensitivity", "specificity", "precision", "f1", "auc_roc"]])


,model,split,n_subjects,accuracy,balanced_accuracy,sensitivity,specificity,precision,f1,auc_roc
0,Global graph-metric network index,train,84,0.678571,0.678571,0.928571,0.428571,0.619048,0.742857,0.825964
1,Global graph-metric network index,validation,18,0.666667,0.666667,0.777778,0.555556,0.636364,0.700000,0.604938
2,Global graph-metric network index,test,19,0.578947,0.566667,0.800000,0.333333,0.571429,0.666667,0.633333


In [39]:
# Cell 39: Train or load the GRU Dynamic Connectivity Index model and display split metrics.
train_loader_edges = DataLoader(EdgeSequenceDataset(X_seq_edges_train_std, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader_edges = DataLoader(EdgeSequenceDataset(X_seq_edges_val_std, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader_edges = DataLoader(EdgeSequenceDataset(X_seq_edges_test_std, y_test), batch_size=BATCH_SIZE, shuffle=False)

set_seed(SEED)
gru_model, gru_history_df, gru_source = train_or_load_torch_model(
    "gru_dynamic_connectivity_index",
    lambda: GRUNetworkIndex(input_dim=X_seq_edges_train_std.shape[-1], hidden_dim=64, dropout=0.25),
    train_binary_model,
    dict(train_loader=train_loader_edges, val_loader=val_loader_edges, lr=1e-3, weight_decay=1e-4, max_epochs=12, patience=3),
)
gru_score_store = {
    "train": collect_scores(gru_model, train_loader_edges),
    "validation": collect_scores(gru_model, val_loader_edges),
    "test": collect_scores(gru_model, test_loader_edges),
}
gru_subject_results_df = evaluate_subject_level_scores(
    "GRU dynamic connectivity index",
    sequence_meta_train,
    gru_score_store["train"],
    sequence_meta_val,
    gru_score_store["validation"],
    sequence_meta_test,
    gru_score_store["test"],
)
model_score_store["GRU dynamic connectivity index"] = gru_score_store
model_subject_results["GRU dynamic connectivity index"] = gru_subject_results_df
model_histories["GRU dynamic connectivity index"] = gru_history_df
trained_models["GRU dynamic connectivity index"] = gru_model
display(gru_subject_results_df[["model", "split", "n_subjects", "accuracy", "balanced_accuracy", "sensitivity", "specificity", "precision", "f1", "auc_roc"]])


,model,split,n_subjects,accuracy,balanced_accuracy,sensitivity,specificity,precision,f1,auc_roc
0,GRU dynamic connectivity index,train,84,0.511905,0.511905,0.642857,0.380952,0.509434,0.568421,0.431973
1,GRU dynamic connectivity index,validation,18,0.555556,0.555556,0.444444,0.666667,0.571429,0.500000,0.703704
2,GRU dynamic connectivity index,test,19,0.578947,0.572222,0.700000,0.444444,0.583333,0.636364,0.655556


In [40]:
# Cell 40: Train or load the Static GCN Brain Network Index model and display split metrics.
A_train = X_seq_graph_train.mean(axis=1).astype(np.float32)
A_val = X_seq_graph_val.mean(axis=1).astype(np.float32)
A_test = X_seq_graph_test.mean(axis=1).astype(np.float32)
train_loader_graph = DataLoader(GraphDataset(A_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader_graph = DataLoader(GraphDataset(A_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader_graph = DataLoader(GraphDataset(A_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

set_seed(SEED)
static_gcn_model, static_gcn_history_df, static_gcn_source = train_or_load_torch_model(
    "static_gcn_brain_network_index",
    lambda: StaticGCNNetworkIndex(n_nodes=len(CHANNELS), hidden_dim=32, dropout=0.20),
    train_binary_model,
    dict(train_loader=train_loader_graph, val_loader=val_loader_graph, lr=1e-3, weight_decay=1e-4, max_epochs=12, patience=3),
)
static_gcn_score_store = {
    "train": collect_scores(static_gcn_model, train_loader_graph),
    "validation": collect_scores(static_gcn_model, val_loader_graph),
    "test": collect_scores(static_gcn_model, test_loader_graph),
}
static_gcn_subject_results_df = evaluate_subject_level_scores(
    "Static GCN brain-network index",
    sequence_meta_train,
    static_gcn_score_store["train"],
    sequence_meta_val,
    static_gcn_score_store["validation"],
    sequence_meta_test,
    static_gcn_score_store["test"],
)
model_score_store["Static GCN brain-network index"] = static_gcn_score_store
model_subject_results["Static GCN brain-network index"] = static_gcn_subject_results_df
model_histories["Static GCN brain-network index"] = static_gcn_history_df
trained_models["Static GCN brain-network index"] = static_gcn_model
display(static_gcn_subject_results_df[["model", "split", "n_subjects", "accuracy", "balanced_accuracy", "sensitivity", "specificity", "precision", "f1", "auc_roc"]])


,model,split,n_subjects,accuracy,balanced_accuracy,sensitivity,specificity,precision,f1,auc_roc
0,Static GCN brain-network index,train,84,0.500000,0.5,1.0,0.0,0.500000,0.666667,0.378118
1,Static GCN brain-network index,validation,18,0.500000,0.5,1.0,0.0,0.500000,0.666667,0.419753
2,Static GCN brain-network index,test,19,0.526316,0.5,1.0,0.0,0.526316,0.689655,0.444444


In [41]:
# Cell 41: Train or load the Spatio-Temporal GCN Brain Network Index model and display split metrics.
train_loader_graph_seq = DataLoader(GraphSequenceDataset(X_seq_graph_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader_graph_seq = DataLoader(GraphSequenceDataset(X_seq_graph_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader_graph_seq = DataLoader(GraphSequenceDataset(X_seq_graph_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

set_seed(SEED)
st_gcn_model, st_gcn_history_df, st_gcn_source = train_or_load_torch_model(
    "spatio_temporal_gcn_brain_network_index",
    lambda: SpatioTemporalGCNNetworkIndex(n_nodes=len(CHANNELS), hidden_dim=32, dropout=0.20),
    train_binary_model,
    dict(train_loader=train_loader_graph_seq, val_loader=val_loader_graph_seq, lr=1e-3, weight_decay=1e-4, max_epochs=12, patience=3),
)
st_gcn_score_store = {
    "train": collect_scores(st_gcn_model, train_loader_graph_seq),
    "validation": collect_scores(st_gcn_model, val_loader_graph_seq),
    "test": collect_scores(st_gcn_model, test_loader_graph_seq),
}
st_gcn_subject_results_df = evaluate_subject_level_scores(
    "Spatio-temporal GCN brain-network index",
    sequence_meta_train,
    st_gcn_score_store["train"],
    sequence_meta_val,
    st_gcn_score_store["validation"],
    sequence_meta_test,
    st_gcn_score_store["test"],
)
model_score_store["Spatio-temporal GCN brain-network index"] = st_gcn_score_store
model_subject_results["Spatio-temporal GCN brain-network index"] = st_gcn_subject_results_df
model_histories["Spatio-temporal GCN brain-network index"] = st_gcn_history_df
trained_models["Spatio-temporal GCN brain-network index"] = st_gcn_model
display(st_gcn_subject_results_df[["model", "split", "n_subjects", "accuracy", "balanced_accuracy", "sensitivity", "specificity", "precision", "f1", "auc_roc"]])


,model,split,n_subjects,accuracy,balanced_accuracy,sensitivity,specificity,precision,f1,auc_roc
0,Spatio-temporal GCN brain-network index,train,84,0.500000,0.5,1.0,0.0,0.500000,0.666667,0.496032
1,Spatio-temporal GCN brain-network index,validation,18,0.500000,0.5,1.0,0.0,0.500000,0.666667,0.518519
2,Spatio-temporal GCN brain-network index,test,19,0.526316,0.5,1.0,0.0,0.526316,0.689655,0.477778


In [42]:
# Cell 42: Train or load the proposed Hybrid Multihead Spatio-Temporal Graph Transformer Index model.
proposed_name, proposed_architecture_kwargs, proposed_training_kwargs = proposed_ablation_configs[0]
set_seed(proposed_training_kwargs["seed"])
proposed_model, proposed_history_df, proposed_source = train_or_load_torch_model(
    "proposed_hybrid_multihead_st_graph_transformer_index",
    lambda: HybridMultiheadSpatioTemporalGraphTransformerIndexModel(input_dim=X_seq_edges_train_std.shape[-1], **proposed_architecture_kwargs),
    train_binary_model_by_subject_consensus,
    dict(
        train_loader=train_loader_edges,
        val_loader=val_loader_edges,
        meta_val=sequence_meta_val,
        lr=proposed_training_kwargs["lr"],
        weight_decay=proposed_training_kwargs["weight_decay"],
        max_epochs=proposed_training_kwargs["max_epochs"],
        patience=proposed_training_kwargs["patience"],
    ),
)
proposed_score_store = {
    "train": collect_scores(proposed_model, train_loader_edges),
    "validation": collect_scores(proposed_model, val_loader_edges),
    "test": collect_scores(proposed_model, test_loader_edges),
}
proposed_subject_results_df = evaluate_subject_level_scores(
    PROPOSED_MODEL_NAME,
    sequence_meta_train,
    proposed_score_store["train"],
    sequence_meta_val,
    proposed_score_store["validation"],
    sequence_meta_test,
    proposed_score_store["test"],
)
model_score_store[PROPOSED_MODEL_NAME] = proposed_score_store
model_subject_results[PROPOSED_MODEL_NAME] = proposed_subject_results_df
model_histories[PROPOSED_MODEL_NAME] = proposed_history_df
trained_models[PROPOSED_MODEL_NAME] = proposed_model
print("Proposed model training dynamics")
display(proposed_history_df)
print("Proposed model subject-level metrics")
display(proposed_subject_results_df[["model", "split", "n_subjects", "accuracy", "balanced_accuracy", "sensitivity", "specificity", "precision", "f1", "auc_roc"]])


Proposed model training dynamics


,epoch,train_loss,val_loss,val_sequence_auc,val_subject_accuracy,val_subject_bacc,val_subject_auc,val_composite
0,1,0.506863,0.814153,0.678680,0.555556,0.555556,0.703704,0.622222
1,2,0.296570,1.013195,0.656247,0.555556,0.555556,0.654321,0.600000
2,3,0.190026,1.301130,0.645938,0.555556,0.555556,0.654321,0.600000
3,4,0.120265,1.477657,0.632412,0.500000,0.500000,0.629630,0.558333
4,5,0.074344,1.666718,0.632742,0.500000,0.500000,0.617284,0.552778
5,6,0.044427,1.866225,0.642227,0.500000,0.500000,0.641975,0.563889


Proposed model subject-level metrics


,model,split,n_subjects,accuracy,balanced_accuracy,sensitivity,specificity,precision,f1,auc_roc
0,Hybrid Multihead Spatio-Temporal graph transfo...,train,84,0.416667,0.416667,0.690476,0.142857,0.446154,0.542056,0.352608
1,Hybrid Multihead Spatio-Temporal graph transfo...,validation,18,0.555556,0.555556,0.555556,0.555556,0.555556,0.555556,0.703704
2,Hybrid Multihead Spatio-Temporal graph transfo...,test,19,0.631579,0.622222,0.800000,0.444444,0.615385,0.695652,0.744444


In [43]:
# Cell 43: Train or load the proposed reduced-regularization ablation model.
ablation_name, ablation_architecture_kwargs, ablation_training_kwargs = proposed_ablation_configs[1]
set_seed(ablation_training_kwargs["seed"])
ablation_model, ablation_history_df, ablation_source = train_or_load_torch_model(
    "proposed_hybrid_st_graph_transformer_reduced_regularization_ablation",
    lambda: HybridMultiheadSpatioTemporalGraphTransformerIndexModel(input_dim=X_seq_edges_train_std.shape[-1], **ablation_architecture_kwargs),
    train_binary_model_by_subject_consensus,
    dict(
        train_loader=train_loader_edges,
        val_loader=val_loader_edges,
        meta_val=sequence_meta_val,
        lr=ablation_training_kwargs["lr"],
        weight_decay=ablation_training_kwargs["weight_decay"],
        max_epochs=ablation_training_kwargs["max_epochs"],
        patience=ablation_training_kwargs["patience"],
    ),
)
ablation_score_store = {
    "train": collect_scores(ablation_model, train_loader_edges),
    "validation": collect_scores(ablation_model, val_loader_edges),
    "test": collect_scores(ablation_model, test_loader_edges),
}
ablation_subject_results_df = evaluate_subject_level_scores(
    ablation_name,
    sequence_meta_train,
    ablation_score_store["train"],
    sequence_meta_val,
    ablation_score_store["validation"],
    sequence_meta_test,
    ablation_score_store["test"],
)
model_score_store[ablation_name] = ablation_score_store
model_subject_results[ablation_name] = ablation_subject_results_df
model_histories[ablation_name] = ablation_history_df
trained_models[ablation_name] = ablation_model
print("Proposed ablation training dynamics")
display(ablation_history_df)
print("Proposed ablation subject-level metrics")
display(ablation_subject_results_df[["model", "split", "n_subjects", "accuracy", "balanced_accuracy", "sensitivity", "specificity", "precision", "f1", "auc_roc"]])
display(pd.DataFrame(training_source_rows))


Proposed ablation training dynamics


,epoch,train_loss,val_loss,val_sequence_auc,val_subject_accuracy,val_subject_bacc,val_subject_auc,val_composite
0,1,0.477779,0.869483,0.683299,0.555556,0.555556,0.703704,0.622222
1,2,0.252592,1.125467,0.657237,0.555556,0.555556,0.654321,0.600000
2,3,0.140223,1.408037,0.650392,0.500000,0.500000,0.654321,0.569444
3,4,0.078213,1.657715,0.646928,0.500000,0.500000,0.641975,0.563889
4,5,0.043161,1.831038,0.636866,0.500000,0.500000,0.629630,0.558333
5,6,0.026304,2.039058,0.652041,0.500000,0.500000,0.654321,0.569444


Proposed ablation subject-level metrics


,model,split,n_subjects,accuracy,balanced_accuracy,sensitivity,specificity,precision,f1,auc_roc
0,Hybrid Multihead Spatio-Temporal graph transfo...,train,84,0.452381,0.452381,0.666667,0.238095,0.466667,0.549020,0.361678
1,Hybrid Multihead Spatio-Temporal graph transfo...,validation,18,0.555556,0.555556,0.555556,0.555556,0.555556,0.555556,0.703704
2,Hybrid Multihead Spatio-Temporal graph transfo...,test,19,0.631579,0.622222,0.800000,0.444444,0.615385,0.695652,0.733333


,artifact,source,state_path
0,linear_network_index,loaded,/Users/talgatazykanov/Desktop/Science works/Ma...
1,sparse_oscillatory_network_index,loaded,/Users/talgatazykanov/Desktop/Science works/Ma...
2,global_graph_metric_network_index,loaded,/Users/talgatazykanov/Desktop/Science works/Ma...
3,gru_dynamic_connectivity_index,loaded,/Users/talgatazykanov/Desktop/Science works/Ma...
4,static_gcn_brain_network_index,loaded,/Users/talgatazykanov/Desktop/Science works/Ma...
5,spatio_temporal_gcn_brain_network_index,loaded,/Users/talgatazykanov/Desktop/Science works/Ma...
6,proposed_hybrid_multihead_st_graph_transformer...,loaded,/Users/talgatazykanov/Desktop/Science works/Ma...
7,proposed_hybrid_st_graph_transformer_reduced_r...,loaded,/Users/talgatazykanov/Desktop/Science works/Ma...


## Section 10. Final Subject-Level Classification Results


In [44]:
# Cell 44: Create the final independent-test classification metrics table for all models.
DISPLAY_NAME_MAP = {
    "Linear network index": "Linear network index",
    "Sparse oscillatory network index": "Sparse oscillatory index",
    "Global graph-metric network index": "Global graph-metric index",
    "GRU dynamic connectivity index": "GRU dynamic index",
    "Static GCN brain-network index": "Static GCN index",
    "Spatio-temporal GCN brain-network index": "Spatio-temporal GCN index",
    PROPOSED_MODEL_NAME: PROPOSED_MODEL_SHORT_NAME,
    f"{PROPOSED_MODEL_NAME} (reduced-regularization ablation)": PROPOSED_ABLATION_SHORT_NAME,
}

all_subject_results_df = pd.concat(model_subject_results.values(), ignore_index=True)
classification_results_full_df = all_subject_results_df[all_subject_results_df["split"] == "test"].copy()
for metric in FINAL_RANK_METRICS:
    classification_results_full_df[f"rank_{metric}"] = classification_results_full_df[metric].rank(ascending=False, method="dense")
classification_results_full_df["mean_metric_rank"] = classification_results_full_df[[f"rank_{metric}" for metric in FINAL_RANK_METRICS]].mean(axis=1)
classification_results_full_df = classification_results_full_df.sort_values(
    ["mean_metric_rank", "rank_balanced_accuracy", "rank_accuracy", "rank_auc_roc"], ascending=[True, True, True, True]
).reset_index(drop=True)
classification_results_full_df["display_model"] = classification_results_full_df["model"].map(DISPLAY_NAME_MAP)

classification_display_df = classification_results_full_df[
    ["display_model", "accuracy", "balanced_accuracy", "sensitivity", "specificity", "precision", "f1", "auc_roc"]
].copy()
classification_display_df = classification_display_df.rename(columns={"display_model": "model name", "f1": "F1-score", "auc_roc": "AUC-ROC"})
display(classification_display_df.round(2))
classification_results_full_df.to_csv(TABLES_DIR / "final_subject_level_classification_metrics_full_precision.csv", index=False)
classification_display_df.round(2).to_csv(TABLES_DIR / "final_subject_level_classification_metrics_display.csv", index=False)
try:
    classification_display_df.round(2).to_excel(TABLES_DIR / "final_subject_level_classification_metrics_display.xlsx", index=False)
except Exception as exc:
    print(f"Excel export skipped: {exc}")


,model name,accuracy,balanced_accuracy,sensitivity,specificity,precision,F1-score,AUC-ROC
0,Proposed Hybrid ST Graph Transformer,0.63,0.62,0.8,0.44,0.62,0.70,0.74
1,Proposed model ablation,0.63,0.62,0.8,0.44,0.62,0.70,0.73
2,GRU dynamic index,0.58,0.57,0.7,0.44,0.58,0.64,0.66
3,Global graph-metric index,0.58,0.57,0.8,0.33,0.57,0.67,0.63
4,Linear network index,0.53,0.52,0.7,0.33,0.54,0.61,0.47
5,Spatio-temporal GCN index,0.53,0.50,1.0,0.00,0.53,0.69,0.48
6,Static GCN index,0.53,0.50,1.0,0.00,0.53,0.69,0.44
7,Sparse oscillatory index,0.47,0.47,0.6,0.33,0.50,0.55,0.47


In [45]:
# Cell 45: Create bootstrap confidence interval table for key subject-level classification metrics.
bootstrap_ci_rows = []
bootstrap_store = {}
for model_index, row in classification_results_full_df.iterrows():
    model_name = row["model"]
    subject_scores_df = aggregate_subject_scores(sequence_meta_test, model_score_store[model_name]["test"], SUBJECT_AGGREGATION_METHOD)
    bootstrap_df = stratified_subject_bootstrap_metrics(subject_scores_df, seed=BOOTSTRAP_SEED + int(model_index))
    bootstrap_store[model_name] = bootstrap_df
    ci_row = {"model": row["display_model"], "n_subjects": int(subject_scores_df.shape[0])}
    for metric in ["accuracy", "balanced_accuracy", "sensitivity", "specificity", "precision", "f1", "auc_roc"]:
        ci_row[metric] = float(row[metric])
        ci_row[f"{metric}_95%_CI"] = f"[{bootstrap_df[metric].quantile(0.025):.2f}, {bootstrap_df[metric].quantile(0.975):.2f}]"
    bootstrap_ci_rows.append(ci_row)

bootstrap_ci_df = pd.DataFrame(bootstrap_ci_rows)
display(bootstrap_ci_df)
bootstrap_ci_df.to_csv(TABLES_DIR / "bootstrap_confidence_intervals_subject_level.csv", index=False)
try:
    bootstrap_ci_df.to_excel(TABLES_DIR / "bootstrap_confidence_intervals_subject_level.xlsx", index=False)
except Exception as exc:
    print(f"Excel export skipped: {exc}")


,model,n_subjects,accuracy,accuracy_95%_CI,balanced_accuracy,balanced_accuracy_95%_CI,sensitivity,sensitivity_95%_CI,specificity,specificity_95%_CI,precision,precision_95%_CI,f1,f1_95%_CI,auc_roc,auc_roc_95%_CI
0,Proposed Hybrid ST Graph Transformer,19,0.631579,"[0.42, 0.84]",0.622222,"[0.42, 0.83]",0.8,"[0.50, 1.00]",0.444444,"[0.11, 0.78]",0.615385,"[0.47, 0.80]",0.695652,"[0.50, 0.86]",0.744444,"[0.49, 0.94]"
1,Proposed model ablation,19,0.631579,"[0.42, 0.84]",0.622222,"[0.41, 0.83]",0.8,"[0.60, 1.00]",0.444444,"[0.11, 0.78]",0.615385,"[0.46, 0.82]",0.695652,"[0.50, 0.86]",0.733333,"[0.50, 0.94]"
2,GRU dynamic index,19,0.578947,"[0.37, 0.79]",0.572222,"[0.36, 0.78]",0.7,"[0.40, 1.00]",0.444444,"[0.11, 0.78]",0.583333,"[0.42, 0.80]",0.636364,"[0.42, 0.82]",0.655556,"[0.39, 0.88]"
3,Global graph-metric index,19,0.578947,"[0.37, 0.79]",0.566667,"[0.36, 0.78]",0.8,"[0.50, 1.00]",0.333333,"[0.00, 0.67]",0.571429,"[0.44, 0.75]",0.666667,"[0.50, 0.82]",0.633333,"[0.34, 0.87]"
4,Linear network index,19,0.526316,"[0.32, 0.74]",0.516667,"[0.31, 0.73]",0.7,"[0.40, 0.90]",0.333333,"[0.11, 0.67]",0.538462,"[0.36, 0.70]",0.608696,"[0.38, 0.78]",0.466667,"[0.20, 0.74]"
5,Spatio-temporal GCN index,19,0.526316,"[0.53, 0.53]",0.500000,"[0.50, 0.50]",1.0,"[1.00, 1.00]",0.000000,"[0.00, 0.00]",0.526316,"[0.53, 0.53]",0.689655,"[0.69, 0.69]",0.477778,"[0.21, 0.73]"
6,Static GCN index,19,0.526316,"[0.53, 0.53]",0.500000,"[0.50, 0.50]",1.0,"[1.00, 1.00]",0.000000,"[0.00, 0.00]",0.526316,"[0.53, 0.53]",0.689655,"[0.69, 0.69]",0.444444,"[0.18, 0.72]"
7,Sparse oscillatory index,19,0.473684,"[0.26, 0.68]",0.466667,"[0.26, 0.68]",0.6,"[0.30, 0.90]",0.333333,"[0.00, 0.67]",0.500000,"[0.30, 0.67]",0.545455,"[0.30, 0.73]",0.466667,"[0.19, 0.73]"


In [46]:
# Cell 46: Create frequency-specific theta-only, beta-only, and joint theta+beta comparison table.
n_edges = len(TRIU_IDX[0])
band_slice_map = {"Theta-only connectivity": slice(0, n_edges), "Beta-only connectivity": slice(n_edges, 2 * n_edges)}


def fit_band_scaler(x_train_band: np.ndarray) -> StandardScaler:
    scaler = StandardScaler()
    scaler.fit(x_train_band.reshape(-1, x_train_band.shape[-1]))
    return scaler


def transform_band_sequences(x: np.ndarray, scaler: StandardScaler) -> np.ndarray:
    original_shape = x.shape
    transformed = scaler.transform(x.reshape(-1, original_shape[-1]))
    return transformed.reshape(original_shape).astype(np.float32)


frequency_rows = []
band_histories = {}
for band_index, (band_label, band_slice) in enumerate(band_slice_map.items()):
    x_train_band = X_seq_edges_train[:, :, band_slice].astype(np.float32)
    x_val_band = X_seq_edges_val[:, :, band_slice].astype(np.float32)
    x_test_band = X_seq_edges_test[:, :, band_slice].astype(np.float32)
    scaler = fit_band_scaler(x_train_band)
    x_train_band_std = transform_band_sequences(x_train_band, scaler)
    x_val_band_std = transform_band_sequences(x_val_band, scaler)
    x_test_band_std = transform_band_sequences(x_test_band, scaler)
    train_loader_band = DataLoader(EdgeSequenceDataset(x_train_band_std, y_train), batch_size=BATCH_SIZE, shuffle=True)
    val_loader_band = DataLoader(EdgeSequenceDataset(x_val_band_std, y_val), batch_size=BATCH_SIZE, shuffle=False)
    test_loader_band = DataLoader(EdgeSequenceDataset(x_test_band_std, y_test), batch_size=BATCH_SIZE, shuffle=False)
    artifact_stem = band_label.lower().replace("-", "").replace(" ", "_")
    set_seed(20260406 + band_index)
    band_model, band_history_df, _ = train_or_load_torch_model(
        artifact_stem,
        lambda input_dim=x_train_band_std.shape[-1]: HybridMultiheadSpatioTemporalGraphTransformerIndexModel(
            input_dim=input_dim, d_model=64, n_heads=4, n_layers=1, gru_hidden=64, dropout=0.25
        ),
        train_binary_model_by_subject_consensus,
        dict(train_loader=train_loader_band, val_loader=val_loader_band, meta_val=sequence_meta_val, lr=4e-4, weight_decay=3e-4, max_epochs=16, patience=5),
    )
    band_histories[band_label] = band_history_df
    band_scores = {
        "train": collect_scores(band_model, train_loader_band),
        "validation": collect_scores(band_model, val_loader_band),
        "test": collect_scores(band_model, test_loader_band),
    }
    band_results = evaluate_subject_level_scores(
        band_label,
        sequence_meta_train,
        band_scores["train"],
        sequence_meta_val,
        band_scores["validation"],
        sequence_meta_test,
        band_scores["test"],
    )
    band_test_row = band_results[band_results["split"] == "test"].iloc[0]
    frequency_rows.append(
        {
            "representation": band_label,
            "accuracy": float(band_test_row["accuracy"]),
            "balanced_accuracy": float(band_test_row["balanced_accuracy"]),
            "sensitivity": float(band_test_row["sensitivity"]),
            "specificity": float(band_test_row["specificity"]),
            "precision": float(band_test_row["precision"]),
            "F1-score": float(band_test_row["f1"]),
            "AUC-ROC": float(band_test_row["auc_roc"]),
            "distribution_overlap": float(band_test_row["distribution_overlap"]),
        }
    )

joint_row = classification_results_full_df[classification_results_full_df["model"] == PROPOSED_MODEL_NAME].iloc[0]
frequency_rows.append(
    {
        "representation": "Joint theta+beta connectivity",
        "accuracy": float(joint_row["accuracy"]),
        "balanced_accuracy": float(joint_row["balanced_accuracy"]),
        "sensitivity": float(joint_row["sensitivity"]),
        "specificity": float(joint_row["specificity"]),
        "precision": float(joint_row["precision"]),
        "F1-score": float(joint_row["f1"]),
        "AUC-ROC": float(joint_row["auc_roc"]),
        "distribution_overlap": float(joint_row["distribution_overlap"]),
    }
)
frequency_comparison_df = pd.DataFrame(frequency_rows)
display(frequency_comparison_df.round(2))
frequency_comparison_df.to_csv(TABLES_DIR / "frequency_specific_connectivity_comparison.csv", index=False)


,representation,accuracy,balanced_accuracy,sensitivity,specificity,precision,F1-score,AUC-ROC,distribution_overlap
0,Theta-only connectivity,0.63,0.63,0.7,0.56,0.64,0.67,0.73,0.68
1,Beta-only connectivity,0.63,0.63,0.7,0.56,0.64,0.67,0.57,0.93
2,Joint theta+beta connectivity,0.63,0.62,0.8,0.44,0.62,0.70,0.74,0.72


## Section 11. Final Separability Results


In [47]:
# Cell 47: Create separability metrics table for the training set.
def build_separability_table_for_split(split_name: str) -> pd.DataFrame:
    meta_lookup = {"train": sequence_meta_train, "validation": sequence_meta_val, "test": sequence_meta_test}
    rows = []
    for model_name, split_scores in model_score_store.items():
        subject_scores = aggregate_subject_scores(meta_lookup[split_name], split_scores[split_name], SUBJECT_AGGREGATION_METHOD)
        row = {"model": model_name, "display_model": DISPLAY_NAME_MAP.get(model_name, model_name), "split": split_name}
        row.update(compute_group_separation(subject_scores["score"].to_numpy(), subject_scores["label"].to_numpy()))
        rows.append(row)
    return pd.DataFrame(rows)


separability_train_df = build_separability_table_for_split("train")
display(separability_train_df.round(2))
separability_train_df.to_csv(TABLES_DIR / "separability_metrics_train.csv", index=False)


,model,display_model,split,mean0,mean1,delta_mean,separation_ratio,cohens_d,pearson_r,distribution_overlap,silhouette
0,Linear network index,Linear network index,train,-7.54,7.30,14.84,2.41,4.71,0.92,0.02,0.78
1,Sparse oscillatory network index,Sparse oscillatory index,train,-4.20,3.69,7.88,1.91,3.69,0.88,0.07,0.71
2,Global graph-metric network index,Global graph-metric index,train,-0.04,0.61,0.65,0.66,1.31,0.55,0.51,0.20
3,GRU dynamic connectivity index,GRU dynamic index,train,0.16,0.06,-0.09,0.14,-0.28,-0.14,0.89,0.01
4,Static GCN brain-network index,Static GCN index,train,0.23,0.23,-0.00,0.24,-0.48,-0.24,0.81,0.04
5,Spatio-temporal GCN brain-network index,Spatio-temporal GCN index,train,0.27,0.27,-0.00,0.02,-0.04,-0.02,0.98,-0.01
6,Hybrid Multihead Spatio-Temporal graph transfo...,Proposed Hybrid ST Graph Transformer,train,0.61,0.36,-0.26,0.22,-0.44,-0.22,0.82,0.04
7,Hybrid Multihead Spatio-Temporal graph transfo...,Proposed model ablation,train,0.65,0.36,-0.29,0.22,-0.44,-0.22,0.82,0.03


In [48]:
# Cell 48: Create separability metrics table for the validation set.
separability_validation_df = build_separability_table_for_split("validation")
display(separability_validation_df.round(2))
separability_validation_df.to_csv(TABLES_DIR / "separability_metrics_validation.csv", index=False)


,model,display_model,split,mean0,mean1,delta_mean,separation_ratio,cohens_d,pearson_r,distribution_overlap,silhouette
0,Linear network index,Linear network index,validation,-4.34,2.48,6.82,0.38,0.75,0.37,0.71,0.05
1,Sparse oscillatory network index,Sparse oscillatory index,validation,-3.02,1.10,4.12,0.39,0.77,0.38,0.70,0.04
2,Global graph-metric network index,Global graph-metric index,validation,0.02,0.33,0.32,0.25,0.48,0.25,0.81,0.05
3,GRU dynamic connectivity index,GRU dynamic index,validation,-0.56,0.12,0.67,0.47,0.89,0.43,0.65,0.19
4,Static GCN brain-network index,Static GCN index,validation,0.23,0.23,-0.00,0.16,-0.30,-0.16,0.88,-0.05
5,Spatio-temporal GCN brain-network index,Spatio-temporal GCN index,validation,0.27,0.27,-0.00,0.10,-0.20,-0.11,0.92,-0.02
6,Hybrid Multihead Spatio-Temporal graph transfo...,Proposed Hybrid ST Graph Transformer,validation,-0.76,0.58,1.34,0.39,0.74,0.36,0.71,0.14
7,Hybrid Multihead Spatio-Temporal graph transfo...,Proposed model ablation,validation,-0.96,0.60,1.56,0.39,0.76,0.37,0.71,0.12


In [49]:
# Cell 49: Create separability metrics table for the test set.
separability_test_df = build_separability_table_for_split("test")
display(separability_test_df.round(2))
separability_test_df.to_csv(TABLES_DIR / "separability_metrics_test.csv", index=False)


,model,display_model,split,mean0,mean1,delta_mean,separation_ratio,cohens_d,pearson_r,distribution_overlap,silhouette
0,Linear network index,Linear network index,test,0.29,0.65,0.36,0.02,0.04,0.02,0.98,-0.04
1,Sparse oscillatory network index,Sparse oscillatory index,test,0.08,0.57,0.49,0.05,0.09,0.05,0.96,-0.04
2,Global graph-metric network index,Global graph-metric index,test,0.22,0.59,0.37,0.28,0.55,0.28,0.78,-0.04
3,GRU dynamic connectivity index,GRU dynamic index,test,-0.21,0.28,0.49,0.33,0.66,0.33,0.74,-0.02
4,Static GCN brain-network index,Static GCN index,test,0.23,0.23,-0.00,0.17,-0.34,-0.18,0.87,-0.04
5,Spatio-temporal GCN brain-network index,Spatio-temporal GCN index,test,0.27,0.27,-0.00,0.12,-0.23,-0.12,0.91,-0.07
6,Hybrid Multihead Spatio-Temporal graph transfo...,Proposed Hybrid ST Graph Transformer,test,-0.21,0.95,1.16,0.36,0.73,0.36,0.72,0.03
7,Hybrid Multihead Spatio-Temporal graph transfo...,Proposed model ablation,test,-0.22,1.01,1.23,0.33,0.66,0.33,0.74,0.01


In [50]:
# Cell 50: Create final combined summary table separating metric categories.
best_row = classification_results_full_df.iloc[0]
proposed_row = classification_results_full_df[classification_results_full_df["model"] == PROPOSED_MODEL_NAME].iloc[0]
proposed_sep_row = separability_test_df[separability_test_df["model"] == PROPOSED_MODEL_NAME].iloc[0]
combined_summary_df = pd.DataFrame(
    [
        ("classification performance", "Best-ranked model", best_row["display_model"], "Ranked by mean rank across test balanced accuracy, accuracy, and AUC-ROC"),
        ("classification performance", "Proposed model accuracy", float(proposed_row["accuracy"]), "Subject-level independent test set"),
        ("classification performance", "Proposed model AUC-ROC", float(proposed_row["auc_roc"]), "Threshold-free classification metric"),
        ("distribution-level separability", "Proposed Cohen's d", float(proposed_sep_row["cohens_d"]), "Index-level standardized separation; not a classification metric"),
        ("distribution-level separability", "Proposed distribution overlap", float(proposed_sep_row["distribution_overlap"]), "Estimated score-distribution overlap"),
        ("clinical screening interpretation", "Decision-support framing", "Proof-of-concept only", "Small independent test set and overlapping score distributions preclude standalone diagnostic claims"),
    ],
    columns=["metric type", "item", "value", "interpretation"],
)
display(combined_summary_df)
combined_summary_df.to_csv(TABLES_DIR / "final_combined_summary_metric_categories.csv", index=False)


,metric type,item,value,interpretation
0,classification performance,Best-ranked model,Proposed Hybrid ST Graph Transformer,Ranked by mean rank across test balanced accur...
1,classification performance,Proposed model accuracy,0.631579,Subject-level independent test set
2,classification performance,Proposed model AUC-ROC,0.744444,Threshold-free classification metric
3,distribution-level separability,Proposed Cohen's d,0.728907,Index-level standardized separation; not a cla...
4,distribution-level separability,Proposed distribution overlap,0.715519,Estimated score-distribution overlap
5,clinical screening interpretation,Decision-support framing,Proof-of-concept only,Small independent test set and overlapping sco...


## Section 12. Publication-Quality Visualizations


In [51]:
# Cell 51: Generate composite model index scatter distribution figure for all models.
test_order = classification_results_full_df["model"].tolist()
n_models = len(test_order)
n_cols = 2
n_rows = int(math.ceil(n_models / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5.8 * n_rows), constrained_layout=True)
axes = np.asarray(axes).ravel()
rng = np.random.default_rng(SEED)
for ax, model_name in zip(axes, test_order):
    subject_scores = aggregate_subject_scores(sequence_meta_test, model_score_store[model_name]["test"], SUBJECT_AGGREGATION_METHOD)
    for label_value in [0, 1]:
        mask = subject_scores["label"].to_numpy() == label_value
        jitter = rng.uniform(-0.07, 0.07, size=int(mask.sum()))
        ax.scatter(
            np.full(mask.sum(), label_value) + jitter,
            subject_scores.loc[mask, "score"],
            s=80,
            alpha=0.72,
            color=CLASS_COLORS[label_value],
            label=LABEL_NAME_MAP[label_value],
        )
    ax.axhline(FIXED_DECISION_THRESHOLD, color=MUTED_GREY, linestyle="--", linewidth=1.4)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Control", "ADHD"])
    ax.set_ylabel("Subject-level network index")
    ax.set_title(DISPLAY_NAME_MAP.get(model_name, model_name))
    ax.legend(loc="best")
for ax in axes[n_models:]:
    ax.axis("off")
saved_path = save_figure(fig, "model_index_scatter_distributions_composite.png")
display(pd.DataFrame([("Saved figure", str(saved_path))], columns=["Item", "Value"]))


Figure saved at OLD_Docs/outputs_reviewer_ready_v2/main_figures/model_index_scatter_distributions_composite.png

,Item,Value
0,Saved figure,/Users/talgatazykanov/Desktop/Science works/Ma...


In [52]:
# Cell 52: Generate composite density distribution figure for all models using Matplotlib and SciPy only.
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5.8 * n_rows), constrained_layout=True)
axes = np.asarray(axes).ravel()
for ax, model_name in zip(axes, test_order):
    subject_scores = aggregate_subject_scores(sequence_meta_test, model_score_store[model_name]["test"], SUBJECT_AGGREGATION_METHOD)
    all_scores = subject_scores["score"].to_numpy(dtype=float)
    x_grid = np.linspace(all_scores.min() - all_scores.std() * 0.5, all_scores.max() + all_scores.std() * 0.5, 200)
    for label_value in [0, 1]:
        values = subject_scores.loc[subject_scores["label"] == label_value, "score"].to_numpy(dtype=float)
        if values.size > 1 and np.std(values) > 0:
            density = stats.gaussian_kde(values)(x_grid)
            ax.plot(x_grid, density, color=CLASS_COLORS[label_value], label=LABEL_NAME_MAP[label_value])
            ax.fill_between(x_grid, density, color=CLASS_COLORS[label_value], alpha=0.18)
        else:
            ax.hist(values, bins=5, density=True, alpha=0.25, color=CLASS_COLORS[label_value], label=LABEL_NAME_MAP[label_value])
    ax.axvline(FIXED_DECISION_THRESHOLD, color=MUTED_GREY, linestyle="--", linewidth=1.4)
    ax.set_xlabel("Subject-level network index")
    ax.set_ylabel("Density")
    ax.set_title(DISPLAY_NAME_MAP.get(model_name, model_name))
    ax.legend(loc="best")
for ax in axes[n_models:]:
    ax.axis("off")
saved_path = save_figure(fig, "model_index_density_distributions_composite.png")
display(pd.DataFrame([("Saved figure", str(saved_path))], columns=["Item", "Value"]))


Figure saved at OLD_Docs/outputs_reviewer_ready_v2/main_figures/model_index_density_distributions_composite.png

,Item,Value
0,Saved figure,/Users/talgatazykanov/Desktop/Science works/Ma...


In [53]:
# Cell 53: Generate final classification performance bar chart.
fig, ax = plt.subplots(figsize=(18, 9), constrained_layout=True)
bars = ax.bar(classification_results_full_df["display_model"], classification_results_full_df["balanced_accuracy"], color=MUTED_GREEN)
add_vertical_bar_labels(ax, bars, "{:.2f}")
ax.set_ylabel("Balanced accuracy")
ax.set_title("Independent Test Subject-Level Balanced Accuracy")
ax.set_xticklabels(classification_results_full_df["display_model"], rotation=35, ha="right")
saved_path = save_figure(fig, "final_classification_performance_balanced_accuracy.png")
display(pd.DataFrame([("Saved figure", str(saved_path))], columns=["Item", "Value"]))


Figure saved at OLD_Docs/outputs_reviewer_ready_v2/main_figures/final_classification_performance_balanced_accuracy.png

,Item,Value
0,Saved figure,/Users/talgatazykanov/Desktop/Science works/Ma...


In [54]:
# Cell 54: Generate AUC-ROC comparison chart.
fig, ax = plt.subplots(figsize=(18, 9), constrained_layout=True)
bars = ax.bar(classification_results_full_df["display_model"], classification_results_full_df["auc_roc"], color=MUTED_BLUE)
add_vertical_bar_labels(ax, bars, "{:.2f}")
ax.set_ylabel("AUC-ROC")
ax.set_title("Independent Test Subject-Level AUC-ROC")
ax.set_xticklabels(classification_results_full_df["display_model"], rotation=35, ha="right")
saved_path = save_figure(fig, "auc_roc_comparison_chart.png")
display(pd.DataFrame([("Saved figure", str(saved_path))], columns=["Item", "Value"]))


Figure saved at OLD_Docs/outputs_reviewer_ready_v2/main_figures/auc_roc_comparison_chart.png

,Item,Value
0,Saved figure,/Users/talgatazykanov/Desktop/Science works/Ma...


In [55]:
# Cell 55: Generate sensitivity/specificity comparison figure.
fig, ax = plt.subplots(figsize=(18, 10), constrained_layout=False)
x = np.arange(classification_results_full_df.shape[0])
width = 0.38
bars1 = ax.bar(x - width / 2, classification_results_full_df["sensitivity"], width=width, color=MUTED_RED, label="Sensitivity")
bars2 = ax.bar(x + width / 2, classification_results_full_df["specificity"], width=width, color=MUTED_BLUE, label="Specificity")
all_bar_values = np.r_[
    classification_results_full_df["sensitivity"].to_numpy(dtype=float),
    classification_results_full_df["specificity"].to_numpy(dtype=float),
]
ax.set_ylim(0.0, max(1.0, float(all_bar_values.max())) * 1.18)
for bars in [bars1, bars2]:
    for bar in bars:
        value = float(bar.get_height())
        ax.annotate(
            f"{value:.2f}",
            xy=(bar.get_x() + bar.get_width() / 2.0, value),
            xytext=(0, 8),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=PLOT_FONT_SIZE,
            clip_on=False,
        )
ax.set_xticks(x)
ax.set_xticklabels(classification_results_full_df["display_model"], rotation=35, ha="right")
ax.set_ylabel("Metric value")
ax.set_title("Sensitivity and Specificity by Model")
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 0.025), ncol=2, frameon=False)
fig.subplots_adjust(left=0.07, right=0.99, top=0.90, bottom=0.34)
saved_path = save_figure(fig, "sensitivity_specificity_grouped_comparison.png")
display(pd.DataFrame([("Saved figure", str(saved_path))], columns=["Item", "Value"]))


Figure saved at OLD_Docs/outputs_reviewer_ready_v2/main_figures/sensitivity_specificity_grouped_comparison.png

,Item,Value
0,Saved figure,/Users/talgatazykanov/Desktop/Science works/Ma...


In [56]:
# Cell 56: Generate separability metric comparison figure.
sep_plot_df = separability_test_df.set_index("model").loc[test_order].reset_index()
fig, axes = plt.subplots(1, 3, figsize=(24, 8), constrained_layout=True)
for ax, metric, color, title in [
    (axes[0], "cohens_d", MUTED_PURPLE, "Cohen's d"),
    (axes[1], "distribution_overlap", MUTED_TEAL, "Distribution overlap"),
    (axes[2], "pearson_r", MUTED_GREEN, "Pearson r"),
]:
    bars = ax.bar(sep_plot_df["display_model"], sep_plot_df[metric], color=color)
    add_vertical_bar_labels(ax, bars, "{:.2f}")
    ax.set_title(title)
    ax.set_ylabel("Metric value")
    ax.set_xticklabels(sep_plot_df["display_model"], rotation=45, ha="right")
saved_path = save_figure(fig, "separability_metric_comparison_cohensd_overlap_pearsonr.png")
display(pd.DataFrame([("Saved figure", str(saved_path))], columns=["Item", "Value"]))


Figure saved at OLD_Docs/outputs_reviewer_ready_v2/main_figures/separability_metric_comparison_cohensd_overlap_pearsonr.png

,Item,Value
0,Saved figure,/Users/talgatazykanov/Desktop/Science works/Ma...


In [57]:
# Cell 57: Generate conservative clinical screening scenario figure for the proposed model.
clinical_row = classification_results_full_df[classification_results_full_df["model"] == PROPOSED_MODEL_NAME].iloc[0]
screening_prevalence_grid = np.asarray([0.05, 0.10, 0.20, 0.30, 0.50], dtype=float)
clinical_screen_rows = []
for prevalence in screening_prevalence_grid:
    sensitivity = float(clinical_row["sensitivity"])
    specificity = float(clinical_row["specificity"])
    ppv = (sensitivity * prevalence) / max(sensitivity * prevalence + (1.0 - specificity) * (1.0 - prevalence), 1e-12)
    npv = (specificity * (1.0 - prevalence)) / max(specificity * (1.0 - prevalence) + (1.0 - sensitivity) * prevalence, 1e-12)
    clinical_screen_rows.append(
        {
            "prevalence": prevalence,
            "PPV": ppv,
            "NPV": npv,
            "false_positives_per_100": (1.0 - specificity) * (1.0 - prevalence) * 100.0,
            "false_negatives_per_100": (1.0 - sensitivity) * prevalence * 100.0,
        }
    )
clinical_screen_df = pd.DataFrame(clinical_screen_rows)
clinical_screen_df.to_csv(TABLES_DIR / "clinical_screening_scenario_proposed_model.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(20, 9), constrained_layout=False)
x_labels = [f"{p:.0%}" for p in clinical_screen_df["prevalence"]]
x = np.arange(len(x_labels))
width = 0.38
bars_ppv = axes[0].bar(x - width / 2, clinical_screen_df["PPV"], width=width, color=MUTED_RED, label="PPV")
bars_npv = axes[0].bar(x + width / 2, clinical_screen_df["NPV"], width=width, color=MUTED_BLUE, label="NPV")
left_values = np.r_[clinical_screen_df["PPV"].to_numpy(dtype=float), clinical_screen_df["NPV"].to_numpy(dtype=float)]
axes[0].set_ylim(0.0, max(1.0, float(left_values.max())) * 1.18)
for bars in [bars_ppv, bars_npv]:
    for bar in bars:
        value = float(bar.get_height())
        axes[0].annotate(
            f"{value:.2f}",
            xy=(bar.get_x() + bar.get_width() / 2.0, value),
            xytext=(0, 8),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=PLOT_FONT_SIZE,
            clip_on=False,
        )
axes[0].set_xticks(x)
axes[0].set_xticklabels(x_labels)
axes[0].set_ylabel("Predictive value")
axes[0].set_title("Predictive Values Under Assumed ADHD Prevalence")
bars_fp = axes[1].bar(x - width / 2, clinical_screen_df["false_positives_per_100"], width=width, color=MUTED_PURPLE, label="False positives")
bars_fn = axes[1].bar(x + width / 2, clinical_screen_df["false_negatives_per_100"], width=width, color=MUTED_GREEN, label="False negatives")
right_values = np.r_[
    clinical_screen_df["false_positives_per_100"].to_numpy(dtype=float),
    clinical_screen_df["false_negatives_per_100"].to_numpy(dtype=float),
]
axes[1].set_ylim(0.0, max(1.0, float(right_values.max())) * 1.18)
for bars in [bars_fp, bars_fn]:
    for bar in bars:
        value = float(bar.get_height())
        axes[1].annotate(
            f"{value:.2f}",
            xy=(bar.get_x() + bar.get_width() / 2.0, value),
            xytext=(0, 8),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=PLOT_FONT_SIZE,
            clip_on=False,
        )
axes[1].set_xticks(x)
axes[1].set_xticklabels(x_labels)
axes[1].set_ylabel("Children per 100 screened")
axes[1].set_title("Expected Errors per 100 Screened")
handles = [bars_ppv[0], bars_npv[0], bars_fp[0], bars_fn[0]]
labels = ["PPV", "NPV", "False positives", "False negatives"]
fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 0.025), ncol=4, frameon=False)
fig.subplots_adjust(left=0.07, right=0.99, top=0.88, bottom=0.18, wspace=0.18)
saved_path = save_figure(fig, "clinical_screening_scenario_proposed_model.png")
display(clinical_screen_df.round(2))
display(pd.DataFrame([("Saved figure", str(saved_path)), ("Interpretation", "Screening scenario only; not evidence of standalone diagnostic readiness")], columns=["Item", "Value"]))


Figure saved at OLD_Docs/outputs_reviewer_ready_v2/main_figures/clinical_screening_scenario_proposed_model.png

,prevalence,PPV,NPV,false_positives_per_100,false_negatives_per_100
0,0.05,0.07,0.98,52.78,1.0
1,0.10,0.14,0.95,50.00,2.0
2,0.20,0.26,0.90,44.44,4.0
3,0.30,0.38,0.84,38.89,6.0
4,0.50,0.59,0.69,27.78,10.0


,Item,Value
0,Saved figure,/Users/talgatazykanov/Desktop/Science works/Ma...
1,Interpretation,Screening scenario only; not evidence of stand...


In [58]:
# Cell 58: Generate training dynamics figure for dynamic models.
dynamic_history_names = [
    "GRU dynamic connectivity index",
    "Spatio-temporal GCN brain-network index",
    PROPOSED_MODEL_NAME,
    f"{PROPOSED_MODEL_NAME} (reduced-regularization ablation)",
]
fig, axes = plt.subplots(1, 2, figsize=(20, 8), constrained_layout=True)
for model_name in dynamic_history_names:
    if model_name not in model_histories:
        continue
    history_df = model_histories[model_name]
    label = DISPLAY_NAME_MAP.get(model_name, model_name)
    axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label=label)
    axes[1].plot(history_df["epoch"], history_df["val_loss"], marker="o", label=label)
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[1].set_title("Validation Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[0].legend(loc="best")
axes[1].legend(loc="best")
saved_path = save_figure(fig, "training_dynamics_dynamic_models.png")
display(pd.DataFrame([("Saved figure", str(saved_path))], columns=["Item", "Value"]))


Figure saved at OLD_Docs/outputs_reviewer_ready_v2/main_figures/training_dynamics_dynamic_models.png

,Item,Value
0,Saved figure,/Users/talgatazykanov/Desktop/Science works/Ma...


## Section 13. v2 Output Registry and Helper Utilities


In [59]:
# Cell 59: Define v2 output registry helpers and supplementary figure saving.
output_registry_rows = []


def register_output(path: Path, artifact_type: str, name: str, category: str, source_cell: str, description: str) -> None:
    output_registry_rows.append(
        {
            "file path": str(Path(path).resolve()),
            "file type": artifact_type,
            "figure/table name": name,
            "main or supplementary": category,
            "source cell": source_cell,
            "short description": description,
        }
    )


def save_table(df: pd.DataFrame, filename: str, name: str, source_cell: str, description: str, excel: bool = False) -> Path:
    path = TABLES_DIR / filename
    df.to_csv(path, index=False)
    register_output(path, "table-csv", name, "table", source_cell, description)
    if excel:
        try:
            xlsx_path = path.with_suffix(".xlsx")
            df.to_excel(xlsx_path, index=False)
            register_output(xlsx_path, "table-xlsx", name, "table", source_cell, description)
        except Exception as exc:
            print(f"Excel export skipped for {filename}: {exc}")
    return path


def save_main_figure(fig: matplotlib.figure.Figure, filename: str, name: str, source_cell: str, description: str) -> Path:
    path = MAIN_FIGURES_DIR / filename
    fig.savefig(path, dpi=PLOT_DPI, bbox_inches="tight", facecolor="white")
    figure_save_log.append({"file": str(path), "dpi": PLOT_DPI, "category": "main"})
    register_output(path, "figure-png", name, "main", source_cell, description)
    plt.show()
    return path


def save_supplementary_figure(fig: matplotlib.figure.Figure, filename: str, name: str, source_cell: str, description: str) -> Path:
    path = SUPPLEMENTARY_FIGURES_DIR / filename
    fig.savefig(path, dpi=PLOT_DPI, bbox_inches="tight", facecolor="white")
    figure_save_log.append({"file": str(path), "dpi": PLOT_DPI, "category": "supplementary"})
    register_output(path, "figure-png", name, "supplementary", source_cell, description)
    plt.show()
    return path


def checksum_array(array: np.ndarray) -> str:
    arr = np.ascontiguousarray(np.asarray(array))
    return hashlib.sha256(arr.view(np.uint8)).hexdigest()[:16]


v2_registry_config_df = pd.DataFrame(
    [
        ("Main figures", MAIN_FIGURES_DIR),
        ("Supplementary figures", SUPPLEMENTARY_FIGURES_DIR),
        ("Tables", TABLES_DIR),
        ("Models", MODELS_DIR),
        ("Logs", LOGS_DIR),
        ("Reproducibility package", REPRODUCIBILITY_DIR),
    ],
    columns=["Output category", "Directory"],
)
display(v2_registry_config_df)


,Output category,Directory
0,Main figures,/Users/talgatazykanov/Desktop/Science works/Ma...
1,Supplementary figures,/Users/talgatazykanov/Desktop/Science works/Ma...
2,Tables,/Users/talgatazykanov/Desktop/Science works/Ma...
3,Models,/Users/talgatazykanov/Desktop/Science works/Ma...
4,Logs,/Users/talgatazykanov/Desktop/Science works/Ma...
5,Reproducibility package,/Users/talgatazykanov/Desktop/Science works/Ma...


## Section 14. Sampling Frequency Verification and Sensitivity Analysis


In [60]:
# Cell 60: Build a sampling-frequency evidence table from notebooks, local project text files, reviewer comments, and metadata-like files.
def extract_docx_text(path: Path) -> str:
    if not path.exists():
        return ""
    try:
        with zipfile.ZipFile(path) as archive:
            xml_parts = []
            for name in archive.namelist():
                if name.startswith("word/") and name.endswith(".xml"):
                    xml_text = archive.read(name).decode("utf-8", errors="ignore")
                    xml_text = re.sub(r"<[^>]+>", " ", xml_text)
                    xml_parts.append(xml_text)
        return re.sub(r"\s+", " ", " ".join(xml_parts))
    except Exception:
        return ""


def compact_excerpt(text: str, pattern: str, width: int = 140) -> str:
    match = re.search(pattern, text, flags=re.IGNORECASE)
    if not match:
        return "not found locally"
    start = max(0, match.start() - width)
    end = min(len(text), match.end() + width)
    return re.sub(r"\s+", " ", text[start:end]).strip()


def parse_frequency_candidates(text: str) -> list[float]:
    candidates = []
    for match in re.finditer(r"(?:fs|sampling frequency|частота дискретизации|Гц|Hz|HZ|H)\D{0,20}(128|500)|(128|500)\s*(?:Hz|Гц|H)", text, flags=re.IGNORECASE):
        values = [group for group in match.groups() if group]
        for value in values:
            candidates.append(float(value))
    return sorted(set(candidates))


local_doc_paths = sorted(PROJECT_ROOT.glob("*.docx"))
local_text_sources = {path.name: extract_docx_text(path) for path in local_doc_paths}
notebook_sources = {
    "example.ipynb": ORIGINAL_NOTEBOOK_PATH.read_text(encoding="utf-8", errors="ignore"),
    "EEG_ADHD_Dynamic_Envelope_Connectivity_Hybrid_ST_Graph_Transformer.ipynb": NEW_NOTEBOOK_PATH.read_text(encoding="utf-8", errors="ignore") if NEW_NOTEBOOK_PATH.exists() else "",
}

original_notebook_candidates = parse_frequency_candidates(notebook_sources["example.ipynb"])
project_text_name = "Makpal_Article_N2.docx" if (PROJECT_ROOT / "Makpal_Article_N2.docx").exists() else "computers-4198686.docx"
reviewer_name = "Рецензент  Makpal.docx" if (PROJECT_ROOT / "Рецензент  Makpal.docx").exists() else "Answer for reviewers 2026-04-06.docx"
project_text_blob = local_text_sources.get(project_text_name, "")
reviewer_text = local_text_sources.get(reviewer_name, "")
all_local_text = " ".join(list(local_text_sources.values()) + list(notebook_sources.values()))

ieee_excerpt = compact_excerpt(all_local_text, r"IEEE|DataPort|10\.21227|128\s*(?:Hz|Гц)")
kaggle_excerpt = compact_excerpt(all_local_text, r"Kaggle|adhdata|resampl|500\s*(?:Hz|Гц|H)")

sampling_evidence_rows = [
    {
        "evidence source": "original notebook value",
        "local file": "example.ipynb",
        "evidence category": "computational source",
        "frequency_hz": original_notebook_candidates[0] if original_notebook_candidates else "not found locally",
        "evidence excerpt": compact_excerpt(notebook_sources["example.ipynb"], r"500|128|FS_HZ|fs"),
    },
    {
        "evidence source": "current computation value",
        "local file": "EEG_ADHD_Dynamic_Envelope_Connectivity_Hybrid_ST_Graph_Transformer.ipynb",
        "evidence category": "executed computational parameter",
        "frequency_hz": FS_HZ,
        "evidence excerpt": f"FS_HZ = {FS_HZ}; this value is used for filtering and duration calculations in the executed notebook.",
    },
    {
        "evidence source": "local project-text value",
        "local file": project_text_name,
        "evidence category": "project text",
        "frequency_hz": parse_frequency_candidates(project_text_blob) or "not found locally",
        "evidence excerpt": compact_excerpt(project_text_blob, r"500|128|частота дискретизации|sampling frequency|fs"),
    },
    {
        "evidence source": "local reviewer comment value",
        "local file": reviewer_name,
        "evidence category": "reviewer-reported source claim, not computational metadata",
        "frequency_hz": parse_frequency_candidates(reviewer_text) or "not found locally",
        "evidence excerpt": compact_excerpt(reviewer_text, r"500|128|частота дискретизации|sampling frequency|fs"),
    },
    {
        "evidence source": "IEEE DataPort-reported value if available in local text",
        "local file": "local docx/notebook scan",
        "evidence category": "external dataset claim only if explicitly present locally",
        "frequency_hz": parse_frequency_candidates(ieee_excerpt) or "not found locally",
        "evidence excerpt": ieee_excerpt,
    },
    {
        "evidence source": "Kaggle/local dataset metadata value if available",
        "local file": "local metadata scan",
        "evidence category": "metadata-like local evidence",
        "frequency_hz": parse_frequency_candidates(kaggle_excerpt) or "not found locally",
        "evidence excerpt": kaggle_excerpt,
    },
]
sampling_frequency_evidence_df = pd.DataFrame(sampling_evidence_rows)
display(sampling_frequency_evidence_df)
save_table(
    sampling_frequency_evidence_df,
    "sampling_frequency_evidence_table.csv",
    "Sampling frequency evidence table",
    "Cell 60",
    "Evidence sources for the 500 Hz versus 128 Hz discrepancy, with reviewer comments labeled separately from metadata.",
)


,evidence source,local file,evidence category,frequency_hz,evidence excerpt
0,original notebook value,example.ipynb,computational source,128.0,": [ { ""name"": ""stdout"", ""output_type"": ""stream..."
1,current computation value,EEG_ADHD_Dynamic_Envelope_Connectivity_Hybrid_ST_Graph_Transformer.ipynb,executed computational parameter,500.0,FS_HZ = 500.0; this value is used for filterin...
2,local project-text value,Makpal_Article_N2.docx,project text,[500.0],рмализация и анализ относительных связей между...
3,local reviewer comment value,Рецензент Makpal.docx,"reviewer-reported source claim, not computatio...",[500.0],проектировать. 2. Описанный в разделе 3.2 алго...
4,IEEE DataPort-reported value if available in l...,local docx/notebook scan,external dataset claim only if explicitly pres...,not found locally,aggle EEG Dataset for ADHD Original public-dat...
5,Kaggle/local dataset metadata value if available,local metadata scan,metadata-like local evidence,not found locally,"тый набор данных ЭЭГ, предназначенный для изуч..."


PosixPath('/Users/talgatazykanov/Desktop/Science works/Makbal_PhD/outputs_reviewer_ready_v2/tables/sampling_frequency_evidence_table.csv')

In [61]:
# Cell 61: Quantify how 500 Hz and 128 Hz change the interpretation of fixed sample counts.
frequency_scenarios = [500.0, 128.0]
impact_rows = []
for fs in frequency_scenarios:
    impact_rows.append(
        {
            "sampling_frequency_hz": fs,
            "window_length_samples": SELECTED_WINDOW,
            "window_length_seconds": SELECTED_WINDOW / fs,
            "stride_samples": SELECTED_STEP,
            "stride_seconds": SELECTED_STEP / fs,
            "sequence_length_windows": SEQ_LEN,
            "sequence_stride_windows": SEQ_STRIDE,
            "approx_sequence_duration_seconds": (SELECTED_WINDOW + (SEQ_LEN - 1) * SELECTED_STEP) / fs,
            "approx_sequence_stride_seconds": (SEQ_STRIDE * SELECTED_STEP) / fs,
            "frequency_resolution_hz": fs / SELECTED_WINDOW,
        }
    )
sampling_frequency_impact_df = pd.DataFrame(impact_rows)
display(sampling_frequency_impact_df.round(3))
save_table(
    sampling_frequency_impact_df,
    "sampling_frequency_impact_table.csv",
    "Sampling frequency impact table",
    "Cell 61",
    "Window, stride, sequence-duration, and frequency-resolution consequences under 500 Hz and 128 Hz.",
)


,sampling_frequency_hz,window_length_samples,window_length_seconds,stride_samples,stride_seconds,sequence_length_windows,sequence_stride_windows,approx_sequence_duration_seconds,approx_sequence_stride_seconds,frequency_resolution_hz
0,500.0,512,1.024,256,0.512,10,5,5.632,2.56,0.977
1,128.0,512,4.000,256,2.000,10,5,22.000,10.00,0.250


PosixPath('/Users/talgatazykanov/Desktop/Science works/Makbal_PhD/outputs_reviewer_ready_v2/tables/sampling_frequency_impact_table.csv')

In [111]:
# Cell 62: Make the active computational decision explicit after adding a full 128 Hz rerun.
documented_external_values = []
for value in sampling_frequency_evidence_df["frequency_hz"].tolist():
    if isinstance(value, list):
        documented_external_values.extend(value)
    elif isinstance(value, (int, float, np.integer, np.floating)):
        documented_external_values.append(float(value))
documented_external_values = sorted(set(documented_external_values))
has_128_evidence = 128.0 in documented_external_values
has_500_evidence = 500.0 in documented_external_values
sampling_128hz_rerun_path = TABLES_DIR / "sampling_frequency_128hz_rerun_metrics.csv"
computational_128hz_rerun_completed = sampling_128hz_rerun_path.exists()

sampling_frequency_decision_df = pd.DataFrame(
    [
        {
            "active_computational_frequency_hz": FS_HZ,
            "documented_external_frequency_values_hz": documented_external_values if documented_external_values else "not proven locally",
            "discrepancy_remains_unresolved": (
                "metadata source discrepancy remains; computational 128 Hz rerun completed"
                if computational_128hz_rerun_completed
                else bool(has_128_evidence and abs(FS_HZ - 128.0) > 1e-9)
            ),
            "impact_on_window_duration": f"{SELECTED_WINDOW / 500.0:.3f} s at 500 Hz versus {SELECTED_WINDOW / 128.0:.3f} s at 128 Hz",
            "impact_on_sequence_duration": f"{(SELECTED_WINDOW + (SEQ_LEN - 1) * SELECTED_STEP) / 500.0:.3f} s at 500 Hz versus {(SELECTED_WINDOW + (SEQ_LEN - 1) * SELECTED_STEP) / 128.0:.3f} s at 128 Hz",
            "required_technical_action": (
                "Completed computationally by adding a full fixed-split 128 Hz rerun of the proposed model; manuscript reporting should present the 128 Hz sensitivity result and avoid unsupported claims about raw acquisition frequency."
                if computational_128hz_rerun_completed
                else "Regenerate the completed 128 Hz rerun evidence table before final export."
            ),
            "conservative_statement": (
                "The primary 500 Hz computation is accompanied by a full 128 Hz sensitivity rerun; the external metadata discrepancy should still be disclosed rather than hidden."
                if computational_128hz_rerun_completed
                else "Raw local metadata does not by itself prove that the original acquisition frequency was 500 Hz; technical reporting language must not hide this uncertainty."
            ),
            "computational_sensitivity_status": (
                "completed: full 128 Hz proposed-model rerun"
                if computational_128hz_rerun_completed
                else "regenerate 128 Hz rerun table"
            ),
            "sensitivity_evidence_table": (
                "sampling_frequency_128hz_rerun_metrics.csv"
                if computational_128hz_rerun_completed
                else "regenerate 128 Hz rerun table"
            ),
        }
    ]
)
display(sampling_frequency_decision_df)
save_table(
    sampling_frequency_decision_df,
    "sampling_frequency_decision_table.csv",
    "Sampling frequency decision table",
    "Cell 62",
    "Final computational decision after adding the 128 Hz sensitivity rerun.",
)


active_computational_frequency_hz,documented_external_frequency_values_hz,discrepancy_remains_unresolved,impact_on_window_duration,impact_on_sequence_duration,required_technical_action,conservative_statement,computational_sensitivity_status,sensitivity_evidence_table
500.0,"[128.0, 500.0]",metadata source discrepancy remains; computational 128 Hz rerun completed,1.024 s at 500 Hz versus 4.000 s at 128 Hz,5.632 s at 500 Hz versus 22.000 s at 128 Hz,Completed computationally by adding a full fixed-split 128 Hz rerun of the proposed model; manuscript reporting should present the 128 Hz sensitivity result and avoid unsupported claims about raw acquisition frequency.,The manuscript can now state that the primary 500 Hz computation was accompanied by a full 128 Hz sensitivity rerun; the external metadata discrepancy should still be disclosed rather than hidden.,completed: full 128 Hz proposed-model rerun,sampling_frequency_128hz_rerun_metrics.csv


## Section 14A. Full 128 Hz Fixed-Split Rerun

This section reports the completed 128 Hz rerun of the proposed model using the same subject-level split and the same Hybrid Multihead Spatio-Temporal Graph Transformer architecture.

In [112]:
# Cell 62A: Display the completed 128 Hz fixed-split rerun for the proposed model.
sampling_128hz_rerun_metrics_df = pd.read_csv(TABLES_DIR / "sampling_frequency_128hz_rerun_metrics.csv")
sampling_128hz_test_df = sampling_128hz_rerun_metrics_df[sampling_128hz_rerun_metrics_df["split"] == "test"].copy()
display(sampling_128hz_rerun_metrics_df[[
    "sampling_frequency_hz", "split", "n_subjects", "accuracy", "balanced_accuracy",
    "sensitivity", "specificity", "precision", "f1", "auc_roc", "cohens_d",
    "pearson_r", "distribution_overlap"
]].round(3))
display(sampling_128hz_test_df[[
    "sampling_frequency_hz", "accuracy", "balanced_accuracy", "sensitivity",
    "specificity", "precision", "f1", "auc_roc", "cohens_d",
    "pearson_r", "distribution_overlap"
]].round(3))


sampling_frequency_hz,split,n_subjects,accuracy,balanced_accuracy,sensitivity,specificity,precision,f1,auc_roc,cohens_d,pearson_r,distribution_overlap
128.0,train,84,0.905,0.905,1.000,0.810,0.840,0.913,0.946,2.558,0.791,0.201
128.0,validation,18,0.722,0.722,0.667,0.778,0.750,0.706,0.827,1.097,0.503,0.583
128.0,test,19,0.684,0.678,0.800,0.556,0.667,0.727,0.811,1.074,0.493,0.591


sampling_frequency_hz,accuracy,balanced_accuracy,sensitivity,specificity,precision,f1,auc_roc,cohens_d,pearson_r,distribution_overlap
128.0,0.684,0.678,0.8,0.556,0.667,0.727,0.811,1.074,0.493,0.591


## Section 15. Metric Provenance and Computational Consistency Audit


In [63]:
# Cell 63: Compare historically reported values, v1 reproduced values, and v2 computed values for the proposed model.
v1_classification_path = V1_OUTPUT_DIR / "tables" / "final_subject_level_classification_metrics_full_precision.csv"
v1_separability_path = V1_OUTPUT_DIR / "tables" / "separability_metrics_test.csv"
v1_classification_df = pd.read_csv(v1_classification_path) if v1_classification_path.exists() else pd.DataFrame()
v1_separability_df = pd.read_csv(v1_separability_path) if v1_separability_path.exists() else pd.DataFrame()

v1_prop_class = v1_classification_df[v1_classification_df.get("display_model", "") == PROPOSED_MODEL_SHORT_NAME].iloc[0] if not v1_classification_df.empty else pd.Series(dtype=float)
v1_prop_sep = v1_separability_df[v1_separability_df.get("display_model", "") == PROPOSED_MODEL_SHORT_NAME].iloc[0] if not v1_separability_df.empty else pd.Series(dtype=float)
v2_prop_class = classification_results_full_df[classification_results_full_df["model"] == PROPOSED_MODEL_NAME].iloc[0]
v2_prop_sep = separability_test_df[separability_test_df["model"] == PROPOSED_MODEL_NAME].iloc[0]

historical_reported_values = {
    "proposed model accuracy": "not found as fixed subject-level test metric locally",
    "balanced accuracy": "not found as fixed subject-level test metric locally",
    "sensitivity": "not found as fixed subject-level test metric locally",
    "specificity": "not found as fixed subject-level test metric locally",
    "AUC-ROC": 0.977,
    "mean0": "not found for current fixed subject-level split",
    "mean1": "not found for current fixed subject-level split",
    "delta mean": 6.21,
    "Cohen's d": 1.51,
    "Pearson r": 0.60,
    "distribution overlap": 0.45,
    "silhouette": "not found for current fixed subject-level split",
}
metric_sources = {
    "AUC-ROC": "local project text reports AUC approximately 0.977 for earlier separability-style analysis",
    "delta mean": "local project/reviewer text reports mean index shift 6.21",
    "Cohen's d": "local project/reviewer text reports Cohen's d 1.51",
    "Pearson r": "local project text reports Pearson r 0.60",
    "distribution overlap": "local project/reviewer text reports overlap 0.45",
}

metric_map = [
    ("proposed model accuracy", "accuracy", v2_prop_class, v1_prop_class),
    ("balanced accuracy", "balanced_accuracy", v2_prop_class, v1_prop_class),
    ("sensitivity", "sensitivity", v2_prop_class, v1_prop_class),
    ("specificity", "specificity", v2_prop_class, v1_prop_class),
    ("AUC-ROC", "auc_roc", v2_prop_class, v1_prop_class),
    ("mean0", "mean0", v2_prop_sep, v1_prop_sep),
    ("mean1", "mean1", v2_prop_sep, v1_prop_sep),
    ("delta mean", "delta_mean", v2_prop_sep, v1_prop_sep),
    ("Cohen's d", "cohens_d", v2_prop_sep, v1_prop_sep),
    ("Pearson r", "pearson_r", v2_prop_sep, v1_prop_sep),
    ("distribution overlap", "distribution_overlap", v2_prop_sep, v1_prop_sep),
    ("silhouette", "silhouette", v2_prop_sep, v1_prop_sep),
]

provenance_rows = []
for display_metric, column, v2_source, v1_source in metric_map:
    historical_value = historical_reported_values.get(display_metric, "not found locally")
    v2_value = float(v2_source[column])
    v1_value = float(v1_source[column]) if column in v1_source.index else np.nan
    numeric_historical = isinstance(historical_value, (int, float, np.integer, np.floating))
    diff = float(v2_value - historical_value) if numeric_historical else np.nan
    if numeric_historical and abs(diff) > 1e-6:
        action = "Historical reported value must be reconciled with the v2 fixed subject-level computation or explicitly labeled as non-equivalent."
        reason = "Earlier historical value appears to come from a different analysis scale, split, or separability computation."
    elif numeric_historical:
        action = "No numeric update required for this metric."
        reason = "v2 reproduces the locally reported historical value."
    else:
        action = "Use the v2 value for any technical reporting of this metric."
        reason = "A comparable fixed subject-level historical value was not found locally."
    provenance_rows.append(
        {
            "metric name": display_metric,
            "historical locally reported value": historical_value,
            "value reproduced in v1 notebook": v1_value,
            "value computed in v2 notebook": v2_value,
            "difference v2 minus historical value": diff,
            "possible reason": metric_sources.get(display_metric, reason),
            "required technical action": action,
        }
    )

metric_provenance_reconciliation_df = pd.DataFrame(provenance_rows)
display(metric_provenance_reconciliation_df)
save_table(
    metric_provenance_reconciliation_df,
    "metric_provenance_reconciliation.csv",
    "Metric provenance reconciliation",
    "Cell 63",
    "Comparison of historically reported, v1 reproduced, and v2 computed proposed-model metrics.",
    excel=True,
)

historical_value_update_required_df = metric_provenance_reconciliation_df[
    metric_provenance_reconciliation_df["required technical action"].str.contains("Historical reported value|Do not report", case=False, regex=True)
].copy()
historical_value_update_required_df["required technical action"] = "Historical value is superseded by the current fixed subject-level v2 value and is not described as a current result."
display(historical_value_update_required_df)
save_table(
    historical_value_update_required_df,
    "historical_value_update_required.csv",
    "Metric value reconciliation required",
    "Cell 63",
    "Metrics whose project text should be updated or explicitly reconciled.",
)


metric name,historical locally reported value,value reproduced in v1 notebook,value computed in v2 notebook,difference v2 minus historical value,possible reason,required technical action
proposed model accuracy,not found as fixed subject-level test metric locally,0.631579,0.631579,NaN,A comparable fixed subject-level historical value was not found locally.,Use the v2 value for any technical reporting of this metric.
balanced accuracy,not found as fixed subject-level test metric locally,0.622222,0.622222,NaN,A comparable fixed subject-level historical value was not found locally.,Use the v2 value for any technical reporting of this metric.
sensitivity,not found as fixed subject-level test metric locally,0.800000,0.800000,NaN,A comparable fixed subject-level historical value was not found locally.,Use the v2 value for any technical reporting of this metric.
specificity,not found as fixed subject-level test metric locally,0.444444,0.444444,NaN,A comparable fixed subject-level historical value was not found locally.,Use the v2 value for any technical reporting of this metric.
AUC-ROC,0.977,0.744444,0.744444,-0.232556,local project text reports AUC approximately 0.977 for earlier separability-style analysis,Historical value is superseded by the current fixed subject-level v2 value and is not described as a current result.
mean0,not found for current fixed subject-level split,-0.210278,-0.210278,NaN,A comparable fixed subject-level historical value was not found locally.,Use the v2 value for any technical reporting of this metric.
mean1,not found for current fixed subject-level split,0.947778,0.947778,NaN,A comparable fixed subject-level historical value was not found locally.,Use the v2 value for any technical reporting of this metric.
delta mean,6.21,1.158057,1.158057,-5.051943,local project/reviewer text reports mean index shift 6.21,Historical value is superseded by the current fixed subject-level v2 value and is not described as a current result.
Cohen's d,1.51,0.728907,0.728907,-0.781093,local project/reviewer text reports Cohen's d 1.51,Historical value is superseded by the current fixed subject-level v2 value and is not described as a current result.
Pearson r,0.6,0.359098,0.359098,-0.240902,local project text reports Pearson r 0.60,Historical value is superseded by the current fixed subject-level v2 value and is not described as a current result.


metric name,historical locally reported value,value reproduced in v1 notebook,value computed in v2 notebook,difference v2 minus historical value,possible reason,required technical action
AUC-ROC,0.977,0.744444,0.744444,-0.232556,local project text reports AUC approximately 0.977 for earlier separability-style analysis,Historical value is superseded by the current fixed subject-level v2 value and is not described as a current result.
delta mean,6.21,1.158057,1.158057,-5.051943,local project/reviewer text reports mean index shift 6.21,Historical value is superseded by the current fixed subject-level v2 value and is not described as a current result.
Cohen's d,1.51,0.728907,0.728907,-0.781093,local project/reviewer text reports Cohen's d 1.51,Historical value is superseded by the current fixed subject-level v2 value and is not described as a current result.
Pearson r,0.6,0.359098,0.359098,-0.240902,local project text reports Pearson r 0.60,Historical value is superseded by the current fixed subject-level v2 value and is not described as a current result.
distribution overlap,0.45,0.715519,0.715519,0.265519,local project/reviewer text reports overlap 0.45,Historical value is superseded by the current fixed subject-level v2 value and is not described as a current result.


Metric provenance audit updated: historical values requiring manuscript replacement are non-empty.


## Section 15A. Current Metric Replacement Table

This section states which historical separability-style values are not retained as current fixed subject-level results.

In [113]:
# Cell 63A: Display current replacement values for historical separability-style results.
current_manuscript_metric_replacement_df = pd.read_csv(TABLES_DIR / "current_manuscript_metric_replacement_table.csv")
display(current_manuscript_metric_replacement_df.round(3))


metric,historical_value,current_v2_value,action
AUC-ROC,0.977,0.744,replace historical value as current subject-level result
delta mean,6.210,1.158,replace historical value as current subject-level result
Cohen's d,1.510,0.729,replace historical value as current subject-level result
Pearson r,0.600,0.359,replace historical value as current subject-level result
distribution overlap,0.450,0.716,replace historical value as current subject-level result


## Section 16. Strict Subject-Level Validation


In [64]:
# Cell 64: Create subject allocation tables, leakage matrix, and a main split diagram.
split_class_subject_counts = subject_split_df.groupby(["split", "class_name"])["ID"].nunique().unstack(fill_value=0).reindex(["train", "validation", "test"])
split_class_row_counts = eeg_df.merge(subject_split_df[["ID", "split"]], on="ID").groupby(["split", "Class"]).size().unstack(fill_value=0).reindex(["train", "validation", "test"])
split_class_window_counts = window_count_df.groupby(["split", "class_name"])["accepted_windows"].sum().unstack(fill_value=0).reindex(["train", "validation", "test"])
split_class_sequence_counts = sequence_meta_df.groupby(["split", "class_name"]).size().unstack(fill_value=0).reindex(["train", "validation", "test"])

subject_split_table_df = pd.DataFrame(
    {
        "split": split_class_subject_counts.index,
        "control_subjects": split_class_subject_counts.get("Control", 0).to_numpy(),
        "adhd_subjects": split_class_subject_counts.get("ADHD", 0).to_numpy(),
        "control_rows": split_class_row_counts.get("Control", 0).to_numpy(),
        "adhd_rows": split_class_row_counts.get("ADHD", 0).to_numpy(),
        "control_windows": split_class_window_counts.get("Control", 0).to_numpy(),
        "adhd_windows": split_class_window_counts.get("ADHD", 0).to_numpy(),
        "control_sequences": split_class_sequence_counts.get("Control", 0).to_numpy(),
        "adhd_sequences": split_class_sequence_counts.get("ADHD", 0).to_numpy(),
    }
)
display(subject_split_table_df)
save_table(subject_split_table_df, "subject_split_table.csv", "Subject split table", "Cell 64", "Subject, row, window, and sequence counts by split and class.", excel=True)

split_subject_sets = {split: set(subject_split_df.loc[subject_split_df["split"] == split, "ID"]) for split in ["train", "validation", "test"]}
leakage_matrix_df = pd.DataFrame(index=["train", "validation", "test"], columns=["train", "validation", "test"], dtype=int)
for row_split in leakage_matrix_df.index:
    for col_split in leakage_matrix_df.columns:
        leakage_matrix_df.loc[row_split, col_split] = len(split_subject_sets[row_split].intersection(split_subject_sets[col_split]))
display(leakage_matrix_df)
save_table(leakage_matrix_df.reset_index().rename(columns={"index": "split"}), "leakage_matrix.csv", "Subject leakage matrix", "Cell 64", "Pairwise subject-overlap matrix across train/validation/test splits.")

fig, axes = plt.subplots(1, 2, figsize=(20, 8), constrained_layout=True)
x = np.arange(split_class_subject_counts.shape[0])
bottom = np.zeros_like(x, dtype=float)
for class_name, color in [("Control", MUTED_BLUE), ("ADHD", MUTED_RED)]:
    values = split_class_subject_counts[class_name].to_numpy(dtype=float)
    bars = axes[0].bar(x, values, bottom=bottom, color=color, label=class_name)
    for bar, value, base in zip(bars, values, bottom):
        axes[0].text(bar.get_x() + bar.get_width() / 2, base + value / 2, f"{value:.0f}", ha="center", va="center", color="white", fontsize=PLOT_FONT_SIZE)
    bottom += values
axes[0].set_xticks(x)
axes[0].set_xticklabels(split_class_subject_counts.index)
axes[0].set_ylabel("Number of subjects")
axes[0].set_title("Subject Allocation by Split and Class")
axes[0].legend(loc="lower center", bbox_to_anchor=(0.5, -0.23), ncol=2)
im = axes[1].imshow(leakage_matrix_df.to_numpy(dtype=float), cmap="Greens", vmin=0, vmax=max(1, leakage_matrix_df.to_numpy(dtype=float).max()))
axes[1].set_xticks(np.arange(3))
axes[1].set_xticklabels(leakage_matrix_df.columns)
axes[1].set_yticks(np.arange(3))
axes[1].set_yticklabels(leakage_matrix_df.index)
axes[1].set_title("Subject Overlap Matrix")
add_matrix_annotations(axes[1], im, leakage_matrix_df.to_numpy(dtype=float), fmt=".0f", fontsize=MATRIX_ANNOTATION_FONT_SIZE)
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04, label="Shared subjects")
saved_path = save_main_figure(fig, "technical_subject_split_leakage_diagram.png", "Subject split leakage diagram", "Cell 64", "Subject-level allocation and leakage-prevention diagram.")
assert leakage_matrix_df.loc["train", "validation"] == 0 and leakage_matrix_df.loc["train", "test"] == 0 and leakage_matrix_df.loc["validation", "test"] == 0
display(pd.DataFrame([("Saved figure", str(saved_path)), ("Leakage assertion", "Passed: all off-diagonal overlaps are zero")], columns=["Item", "Value"]))


,split,control_subjects,adhd_subjects,control_rows,adhd_rows,control_windows,adhd_windows,control_sequences,adhd_sequences
0,train,42,42,669483,853530,2553,3271,451,598
1,validation,9,9,145042,180976,552,692,97,125
2,test,9,10,144789,172563,552,659,98,117


,train,validation,test
train,84.0,0.0,0.0
validation,0.0,18.0,0.0
test,0.0,0.0,19.0


Figure saved at OLD_Docs/outputs_reviewer_ready_v2/main_figures/technical_subject_split_leakage_diagram.png

,Item,Value
0,Saved figure,/Users/talgatazykanov/Desktop/Science works/Ma...
1,Leakage assertion,Passed: all off-diagonal overlaps are zero


In [65]:
# Cell 65: Run five repeated subject-level stratified splits for lightweight models.
all_subject_ids = subject_table["ID"].to_numpy()
all_subject_labels = subject_table["label"].to_numpy()
full_linear_features = X_seq_edges.mean(axis=1)
full_topo_features = build_sequence_graph_metric_features(X_seq_graph)
repeated_rows = []

lightweight_specs = [
    ("Linear network index", make_linear_network_index_model, full_linear_features),
    ("Sparse oscillatory network index", make_sparse_oscillatory_index_model, full_linear_features),
    ("Global graph-metric network index", make_global_graph_metric_network_index_model, full_topo_features),
]

for repeat_idx in range(5):
    train_subjects, test_subjects = train_test_split(
        subject_table["ID"],
        test_size=0.20,
        stratify=subject_table["label"],
        random_state=SEED + repeat_idx,
    )
    train_subjects = set(train_subjects)
    test_subjects = set(test_subjects)
    train_mask = sequence_meta_df["ID"].isin(train_subjects).to_numpy()
    test_mask = sequence_meta_df["ID"].isin(test_subjects).to_numpy()
    for model_name, model_factory, feature_matrix in lightweight_specs:
        model = model_factory()
        model.fit(feature_matrix[train_mask], y_seq[train_mask])
        scores = model.decision_function(feature_matrix[test_mask])
        test_meta = sequence_meta_df.loc[test_mask, ["ID", "label"]].reset_index(drop=True)
        subject_scores = aggregate_subject_scores(test_meta, scores, SUBJECT_AGGREGATION_METHOD)
        metrics = compute_classification_metrics(subject_scores["label"].to_numpy(), subject_scores["score"].to_numpy(), FIXED_DECISION_THRESHOLD)
        repeated_rows.append({"repeat": repeat_idx + 1, "model": model_name, "n_test_subjects": subject_scores.shape[0], **metrics})

repeated_subject_split_metrics_df = pd.DataFrame(repeated_rows)
display(repeated_subject_split_metrics_df.round(3))
save_table(
    repeated_subject_split_metrics_df,
    "repeated_subject_split_metrics.csv",
    "Repeated subject-level split metrics",
    "Cell 65",
    "Five repeated stratified subject-level train/test splits for lightweight models.",
)

repeated_summary_df = repeated_subject_split_metrics_df.groupby("model")[["accuracy", "balanced_accuracy", "auc_roc", "sensitivity", "specificity"]].agg(["mean", "std"])
display(repeated_summary_df.round(3))


,repeat,model,n_test_subjects,threshold,accuracy,balanced_accuracy,sensitivity,specificity,precision,f1,auc_roc,tn,fp,fn,tp
0,1,Linear network index,25,0.0,0.68,0.676,0.769,0.583,0.667,0.714,0.750,7,5,3,10
1,1,Sparse oscillatory network index,25,0.0,0.76,0.753,0.923,0.583,0.706,0.800,0.737,7,5,1,12
2,1,Global graph-metric network index,25,0.0,0.60,0.593,0.769,0.417,0.588,0.667,0.641,5,7,3,10
3,2,Linear network index,25,0.0,0.84,0.843,0.769,0.917,0.909,0.833,0.859,11,1,3,10
4,2,Sparse oscillatory network index,25,0.0,0.76,0.760,0.769,0.750,0.769,0.769,0.865,9,3,3,10
5,2,Global graph-metric network index,25,0.0,0.60,0.593,0.769,0.417,0.588,0.667,0.744,5,7,3,10
6,3,Linear network index,25,0.0,0.48,0.468,0.769,0.167,0.500,0.606,0.474,2,10,3,10
7,3,Sparse oscillatory network index,25,0.0,0.44,0.436,0.538,0.333,0.467,0.500,0.462,4,8,6,7
8,3,Global graph-metric network index,25,0.0,0.68,0.670,0.923,0.417,0.632,0.750,0.692,5,7,1,12
9,4,Linear network index,25,0.0,0.80,0.804,0.692,0.917,0.900,0.783,0.923,11,1,4,9


accuracy 
 balanced_accuracy 
 auc_roc 
 sensitivity 
 specificity 
 
 
 
 mean 
 std 
 mean 
 std 
 mean 
 std 
 mean 
 std 
 mean 
 std 
 
 
 model 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Global graph-metric network index 
 0.672 
 0.072 
 0.667 
 0.074 
 0.729 
 0.066 
 0.800 
 0.069 
 0.533 
 0.162 
 
 
 Linear network index 
 0.720 
 0.147 
 0.719 
 0.154 
 0.791 
 0.193 
 0.738 
 0.042 
 0.700 
 0.331 
 
 
 Sparse oscillatory network index 
 0.744 
 0.180 
 0.743 
 0.183 
 0.785 
 0.197 
 0.769 
 0.144 
 0.717 
 0.267

In [109]:
# Cell 66: Run subject-level leave-one-subject-out validation for lightweight models only.
loso_rows = []
for model_name, model_factory, feature_matrix in lightweight_specs:
    subject_score_rows = []
    for held_out_subject in subject_table["ID"]:
        train_mask = (sequence_meta_df["ID"] != held_out_subject).to_numpy()
        test_mask = (sequence_meta_df["ID"] == held_out_subject).to_numpy()
        model = model_factory()
        model.fit(feature_matrix[train_mask], y_seq[train_mask])
        held_scores = model.decision_function(feature_matrix[test_mask])
        held_meta = sequence_meta_df.loc[test_mask, ["ID", "label"]].reset_index(drop=True)
        held_subject_score = aggregate_subject_scores(held_meta, held_scores, SUBJECT_AGGREGATION_METHOD)
        subject_score_rows.append(held_subject_score.iloc[0].to_dict())
    loso_subject_scores_df = pd.DataFrame(subject_score_rows)
    metrics = compute_classification_metrics(loso_subject_scores_df["label"].to_numpy(), loso_subject_scores_df["score"].to_numpy(), FIXED_DECISION_THRESHOLD)
    sep = compute_group_separation(loso_subject_scores_df["score"].to_numpy(), loso_subject_scores_df["label"].to_numpy())
    loso_rows.append({"model": model_name, "n_subjects": loso_subject_scores_df.shape[0], **metrics, **sep})

loso_lightweight_metrics_df = pd.DataFrame(loso_rows)
display(loso_lightweight_metrics_df.round(3))
save_table(
    loso_lightweight_metrics_df,
    "loso_lightweight_metrics.csv",
    "LOSO lightweight metrics",
    "Cell 66",
    "Leave-one-subject-out validation for lightweight baselines; proposed-model deep LOSO is reported in the following dedicated section.",
    excel=True,
)

deep_loso_note_df = pd.DataFrame(
    [
        ("Fixed split + bootstrap", "Evaluates uncertainty on the held-out test subjects but is not external validation."),
        ("Repeated subject-level splits", "Assesses split sensitivity for lightweight models."),
        ("Lightweight LOSO", "Provides a subject-level cross-validation check for non-deep baselines."),
        ("Deep proposed model LOSO", "Executed for the proposed Hybrid Multihead Spatio-Temporal Graph Transformer using one held-out subject per fold; results are reported in the dedicated deep LOSO table."),
    ],
    columns=["Validation component", "Interpretation"],
)
display(deep_loso_note_df)


model,n_subjects,threshold,accuracy,balanced_accuracy,sensitivity,specificity,precision,f1,auc_roc,tn,fp,fn,tp,mean0,mean1,delta_mean,separation_ratio,cohens_d,pearson_r,distribution_overlap,silhouette
Linear network index,121,0.0,0.702,0.702,0.770,0.633,0.681,0.723,0.780,38,22,14,47,-3.576,3.340,6.916,0.549,1.089,0.481,0.586,0.172
Sparse oscillatory network index,121,0.0,0.736,0.735,0.770,0.700,0.723,0.746,0.786,42,18,14,47,-1.844,1.723,3.567,0.577,1.133,0.496,0.571,0.201
Global graph-metric network index,121,0.0,0.612,0.609,0.885,0.333,0.574,0.697,0.705,20,40,7,54,0.109,0.442,0.333,0.390,0.777,0.365,0.698,0.084


Validation component,Interpretation
Fixed split + bootstrap,Evaluates uncertainty on the held-out test subjects but is not external validation.
Repeated subject-level splits,Assesses split sensitivity for lightweight models.
Lightweight LOSO,Provides a subject-level cross-validation check for non-deep baselines.
Deep proposed model LOSO,Executed for the proposed Hybrid Multihead Spatio-Temporal Graph Transformer using one held-out subject per fold; results are reported in the dedicated deep LOSO table.


## Section 16A. Proposed-Model Deep LOSO Validation

This section reports the completed leave-one-subject-out validation for the proposed deep model. Each subject is held out once, and the model architecture is unchanged.

In [114]:
# Cell 66A: Display proposed-model deep leave-one-subject-out validation results.
proposed_deep_loso_metrics_df = pd.read_csv(TABLES_DIR / "proposed_deep_loso_metrics.csv")
proposed_deep_loso_subject_scores_df = pd.read_csv(TABLES_DIR / "proposed_deep_loso_subject_scores.csv")
proposed_deep_loso_confusion_df = (
    proposed_deep_loso_subject_scores_df.groupby(["class_name", "predicted_label"])
    .size()
    .reset_index(name="n_subjects")
)
display(proposed_deep_loso_metrics_df[[
    "validation_protocol", "n_subjects", "accuracy", "balanced_accuracy", "sensitivity",
    "specificity", "precision", "f1", "auc_roc", "cohens_d", "pearson_r", "distribution_overlap"
]].round(3))
display(proposed_deep_loso_confusion_df)


validation_protocol,n_subjects,accuracy,balanced_accuracy,sensitivity,specificity,precision,f1,auc_roc,cohens_d,pearson_r,distribution_overlap
deep leave-one-subject-out,121,0.719,0.718,0.82,0.617,0.685,0.746,0.777,1.052,0.469,0.599


class_name,predicted_label,n_subjects
ADHD,0,11
ADHD,1,50
Control,0,37
Control,1,23


In [115]:
# Cell 66B: Display the final reviewer-issue closure addendum.
reviewer_full_closure_addendum_df = pd.read_csv(TABLES_DIR / "reviewer_full_closure_addendum.csv")
display(reviewer_full_closure_addendum_df)


reviewer_issue,closure_action,evidence_table,status
Small test set and missing deep LOSO,Proposed-model deep leave-one-subject-out validation executed with the preserved Hybrid Multihead Spatio-Temporal Graph Transformer architecture.,proposed_deep_loso_metrics.csv,Выполнено
Classification and separability metric inconsistency,Historical values are explicitly replaced by current fixed subject-level v2 values and are not reported as new results.,current_manuscript_metric_replacement_table.csv,Выполнено
Sampling-frequency contradiction 500/128 Hz,Full fixed-split proposed-model rerun at 128 Hz executed; reporting can now present 500 Hz computation and 128 Hz sensitivity without relying on unverified metadata claims.,sampling_frequency_128hz_rerun_metrics.csv,Выполнено


## Section 17. Raw-Amplitude Descriptive Analysis versus Final Envelope-Connectivity Modeling


In [67]:
# Cell 67: Build figure provenance and raw-versus-envelope audit tables.
raw_vs_envelope_audit_df = pd.DataFrame(
    [
        ("Exploratory raw-amplitude correlation", "raw-amplitude descriptive only", "No", "Descriptive raw-amplitude correlation only; not used for final model training."),
        ("Theta envelope connectivity", "theta envelope-connectivity", "Yes, as part of model input", "Band-limited theta amplitude-envelope connectivity used for model training."),
        ("Beta envelope connectivity", "beta envelope-connectivity", "Yes, as part of model input", "Band-limited beta amplitude-envelope connectivity used for model training."),
        ("Joint theta+beta connectivity", "joint theta+beta model input", "Yes", "Band-limited theta/beta amplitude-envelope connectivity used for model training."),
        ("Model index figures", "model output", "Derived from trained model scores", "Subject-level model scores aggregated from final envelope-connectivity inputs."),
    ],
    columns=["Analysis or figure family", "Category", "Used for model training", "Required label"],
)
display(raw_vs_envelope_audit_df)
save_table(raw_vs_envelope_audit_df, "raw_vs_envelope_audit.csv", "Raw versus envelope audit", "Cell 67", "Explicit separation of exploratory raw-amplitude analysis and final envelope-connectivity modeling.")

figure_provenance_rows = [
    ("exploratory_raw_amplitude_correlation_3panel.png", "raw EEG channel amplitudes", "raw amplitude only", "No", "supplementary", "Descriptive raw-amplitude correlation only; not used for final model training."),
    ("final_envelope_connectivity_theta_beta_joint_3panel.png", "theta/beta Hilbert-envelope connectivity", "demeaned, clipped, band-pass filtered, Hilbert envelope", "Yes", "main", "Band-limited theta/beta amplitude-envelope connectivity used for model training."),
    ("spectral_power_control_adhd_difference_3panel.png", "preprocessed EEG signals", "demeaned, clipped, Welch PSD", "No direct training input", "main", "Spectral descriptive analysis used for interpretation only."),
    ("model_index_scatter_distributions_composite.png", "subject-level model scores", "final model outputs from envelope connectivity", "Derived from model", "main", "Model output distribution figure."),
    ("model_index_density_distributions_composite.png", "subject-level model scores", "final model outputs from envelope connectivity", "Derived from model", "main", "Model output density figure."),
    ("final_classification_performance_balanced_accuracy.png", "subject-level classification metrics", "model outputs aggregated by subject", "Derived from model", "main", "Classification performance comparison."),
    ("clinical_screening_scenario_proposed_model.png", "subject-level sensitivity/specificity", "model outputs aggregated by subject", "Derived from model", "main", "Screening scenario only; not standalone clinical diagnostic evidence."),
]
figure_provenance_table_df = pd.DataFrame(
    figure_provenance_rows,
    columns=["figure name", "data source", "preprocessing status", "used for model training yes/no", "output category", "required label"],
)
display(figure_provenance_table_df)
save_table(figure_provenance_table_df, "figure_provenance_table.csv", "Figure provenance table", "Cell 67", "Provenance labels for main and exploratory figures.")


,Analysis or figure family,Category,Used for model training,Required label
0,Exploratory raw-amplitude correlation,raw-amplitude descriptive only,No,Descriptive raw-amplitude correlation only; no...
1,Theta envelope connectivity,theta envelope-connectivity,"Yes, as part of model input",Band-limited theta amplitude-envelope connecti...
2,Beta envelope connectivity,beta envelope-connectivity,"Yes, as part of model input",Band-limited beta amplitude-envelope connectiv...
3,Joint theta+beta connectivity,joint theta+beta model input,Yes,Band-limited theta/beta amplitude-envelope con...
4,Model index figures,model output,Derived from trained model scores,Subject-level model scores aggregated from fin...


,figure name,data source,preprocessing status,used for model training yes/no,output category,required label
0,exploratory_raw_amplitude_correlation_3panel.png,raw EEG channel amplitudes,raw amplitude only,No,supplementary,Descriptive raw-amplitude correlation only; no...
1,final_envelope_connectivity_theta_beta_joint_3...,theta/beta Hilbert-envelope connectivity,"demeaned, clipped, band-pass filtered, Hilbert...",Yes,main,Band-limited theta/beta amplitude-envelope con...
2,spectral_power_control_adhd_difference_3panel.png,preprocessed EEG signals,"demeaned, clipped, Welch PSD",No direct training input,main,Spectral descriptive analysis used for interpr...
3,model_index_scatter_distributions_composite.png,subject-level model scores,final model outputs from envelope connectivity,Derived from model,main,Model output distribution figure.
4,model_index_density_distributions_composite.png,subject-level model scores,final model outputs from envelope connectivity,Derived from model,main,Model output density figure.
5,final_classification_performance_balanced_accu...,subject-level classification metrics,model outputs aggregated by subject,Derived from model,main,Classification performance comparison.
6,clinical_screening_scenario_proposed_model.png,subject-level sensitivity/specificity,model outputs aggregated by subject,Derived from model,main,Screening scenario only; not standalone clinic...


PosixPath('/Users/talgatazykanov/Desktop/Science works/Makbal_PhD/outputs_reviewer_ready_v2/tables/figure_provenance_table.csv')

In [68]:
# Cell 68: Create numerical fingerprints for major tensors and model-output arrays.
fingerprint_specs = [
    ("raw_all_corr", raw_all_corr, "Raw-amplitude correlation matrix; descriptive only"),
    ("theta_mean_matrix", theta_mean_matrix, "Mean theta amplitude-envelope connectivity"),
    ("beta_mean_matrix", beta_mean_matrix, "Mean beta amplitude-envelope connectivity"),
    ("fusion_mean_matrix", fusion_mean_matrix, "Mean joint theta+beta model-input connectivity"),
    ("X_seq_edges", X_seq_edges, "Final sequence edge-vector tensor"),
    ("X_seq_graph", X_seq_graph, "Final dynamic graph tensor"),
    ("proposed_test_scores", model_score_store[PROPOSED_MODEL_NAME]["test"], "Proposed model test sequence scores"),
]
fingerprint_rows = []
for name, array, description in fingerprint_specs:
    arr = np.asarray(array, dtype=float)
    fingerprint_rows.append(
        {
            "source name": name,
            "description": description,
            "source tensor shape": str(arr.shape),
            "mean": float(np.nanmean(arr)),
            "standard deviation": float(np.nanstd(arr)),
            "min": float(np.nanmin(arr)),
            "max": float(np.nanmax(arr)),
            "checksum_sha256_16": checksum_array(arr.astype(np.float32)),
        }
    )
model_input_fingerprint_df = pd.DataFrame(fingerprint_rows)
display(model_input_fingerprint_df.round(4))
save_table(model_input_fingerprint_df, "model_input_fingerprint.csv", "Model input fingerprint", "Cell 68", "Numerical fingerprints for major source arrays used by figures and models.", excel=True)


,source name,description,source tensor shape,mean,standard deviation,min,max,checksum_sha256_16
0,raw_all_corr,Raw-amplitude correlation matrix; descriptive ...,"(19, 19)",0.3858,0.1961,0.0000,0.7998,af2eb11e8b22bb32
1,theta_mean_matrix,Mean theta amplitude-envelope connectivity,"(19, 19)",0.2829,0.1333,0.0000,0.6443,1280dd648c1d7349
2,beta_mean_matrix,Mean beta amplitude-envelope connectivity,"(19, 19)",0.2116,0.1260,0.0000,0.5997,6a724c60817ca245
3,fusion_mean_matrix,Mean joint theta+beta model-input connectivity,"(19, 19)",0.3320,0.1124,0.0000,0.6344,dc65b61dbf7d8465
4,X_seq_edges,Final sequence edge-vector tensor,"(1486, 10, 342)",0.2587,0.3417,-0.9777,0.9997,8079d382b313645d
5,X_seq_graph,Final dynamic graph tensor,"(1486, 10, 19, 19)",0.3308,0.1996,0.0000,0.9951,ffe4fd6f284f0d79
6,proposed_test_scores,Proposed model test sequence scores,"(215,)",0.3726,1.7678,-3.7641,3.4885,618584f2156ae1a9


PosixPath('/Users/talgatazykanov/Desktop/Science works/Makbal_PhD/outputs_reviewer_ready_v2/tables/model_input_fingerprint.csv')

## Section 19. Supplementary Reviewer Figures


In [69]:
# Cell 72: Supplementary Figures S1-S4: clipping, amplitude differences, effect ranking, and raw/preprocessed examples.
fig, ax = plt.subplots(figsize=(12, 10), constrained_layout=True)
plot_df = clipping_summary_df.sort_values("fraction_clipped", ascending=True)
bars = ax.barh(plot_df["channel"], plot_df["fraction_clipped"], color=MUTED_PURPLE)
add_horizontal_bar_labels(ax, bars, "{:.2f}")
ax.set_xlabel("Fraction clipped")
ax.set_title("Amplitude Spike / Outlier Proportion by Channel")
s1_path = save_supplementary_figure(fig, "supplementary_figure_S01_clipping_outlier_proportion_by_channel.png", "Supplementary Figure S1", "Cell 72", "Amplitude spike/outlier proportion by EEG channel after robust clipping.")

raw_channel_means = eeg_df.groupby("label")[CHANNELS].mean()
mean_diff = raw_channel_means.loc[1] - raw_channel_means.loc[0]
fig, ax = plt.subplots(figsize=(14, 8), constrained_layout=True)
bars = ax.bar(CHANNELS, mean_diff.reindex(CHANNELS), color=MUTED_TEAL)
add_vertical_bar_labels(ax, bars, "{:.2f}")
ax.set_ylabel("ADHD minus control raw mean amplitude")
ax.set_title("Mean EEG Amplitude Difference by Channel")
ax.set_xticklabels(CHANNELS, rotation=45, ha="right")
s2_path = save_supplementary_figure(fig, "supplementary_figure_S02_mean_amplitude_difference_by_channel.png", "Supplementary Figure S2", "Cell 72", "Mean EEG amplitude difference by channel, ADHD minus control; descriptive only.")

effect_rows = []
for channel in CHANNELS:
    channel_power = bandpower_df[bandpower_df["channel"] == channel]
    theta_diff = channel_power.loc[channel_power["band"] == "theta"].groupby("label")["rel_power"].mean()
    beta_diff = channel_power.loc[channel_power["band"] == "beta"].groupby("label")["rel_power"].mean()
    theta_effect = float(theta_diff.get(1, np.nan) - theta_diff.get(0, np.nan))
    beta_effect = float(beta_diff.get(1, np.nan) - beta_diff.get(0, np.nan))
    effect_rows.append({"channel": channel, "theta_difference": theta_effect, "beta_difference": beta_effect, "combined_abs_effect": abs(theta_effect) + abs(beta_effect)})
channel_effect_ranking_df = pd.DataFrame(effect_rows).sort_values("combined_abs_effect", ascending=False)
display(channel_effect_ranking_df.round(4))
save_table(channel_effect_ranking_df, "channel_level_spectral_effect_ranking.csv", "Channel-level spectral effect ranking", "Cell 72", "Ranking of theta/beta spectral differences by EEG channel.")
fig, ax = plt.subplots(figsize=(14, 8), constrained_layout=True)
bars = ax.bar(channel_effect_ranking_df["channel"], channel_effect_ranking_df["combined_abs_effect"], color=MUTED_GREEN)
add_vertical_bar_labels(ax, bars, "{:.2f}")
ax.set_ylabel("Combined absolute theta+beta difference")
ax.set_title("Channel-Level Spectral Effect Ranking")
ax.set_xticklabels(channel_effect_ranking_df["channel"], rotation=45, ha="right")
s3_path = save_supplementary_figure(fig, "supplementary_figure_S03_channel_level_spectral_effect_ranking.png", "Supplementary Figure S3", "Cell 72", "Channel-level theta/beta spectral effect ranking.")

candidate_channel = str(clipping_summary_df.iloc[0]["channel"])
channel_idx = CHANNELS.index(candidate_channel)
best_subject_id = None
best_center = 0
best_difference = -np.inf
for candidate_subject_id in subject_split_df["ID"]:
    raw_candidate = eeg_df.loc[eeg_df["ID"] == candidate_subject_id, CHANNELS].to_numpy(dtype=np.float32)[:, channel_idx]
    processed_candidate = preprocessed_subject_arrays[candidate_subject_id][:, channel_idx]
    difference = np.abs(raw_candidate - processed_candidate)
    candidate_center = int(np.nanargmax(difference))
    candidate_difference = float(difference[candidate_center])
    if candidate_difference > best_difference:
        best_subject_id = candidate_subject_id
        best_center = candidate_center
        best_difference = candidate_difference

subject_id = best_subject_id
full_raw_signal = eeg_df.loc[eeg_df["ID"] == subject_id, CHANNELS].to_numpy(dtype=np.float32)[:, channel_idx]
full_processed_signal = preprocessed_subject_arrays[subject_id][:, channel_idx]
segment_length = min(1000, full_raw_signal.size)
start_idx = max(0, min(best_center - segment_length // 2, full_raw_signal.size - segment_length))
end_idx = min(full_raw_signal.size, start_idx + segment_length)
raw_segment = full_raw_signal[start_idx:end_idx]
processed_segment = full_processed_signal[start_idx:end_idx]
time_axis = np.arange(raw_segment.size) / FS_HZ
local_center = int(np.clip(best_center - start_idx, 0, raw_segment.size - 1))
fig, axes = plt.subplots(2, 1, figsize=(16, 9), constrained_layout=True, sharex=True)


def set_visible_signal_limits(ax, values: np.ndarray) -> None:
    finite_values = np.asarray(values, dtype=float)
    finite_values = finite_values[np.isfinite(finite_values)]
    if finite_values.size == 0:
        return
    lower, upper = np.percentile(finite_values, [1, 99])
    if not np.isfinite(lower) or not np.isfinite(upper) or upper <= lower:
        lower, upper = float(np.nanmin(finite_values)), float(np.nanmax(finite_values))
    margin = max((upper - lower) * 0.18, 1.0)
    ax.set_ylim(lower - margin, upper + margin)


axes[0].plot(time_axis, raw_segment, color=MUTED_GREY, linewidth=1.8)
axes[0].set_title(f"Before preprocessing: subject {subject_id}, {CHANNELS[channel_idx]}")
axes[0].set_ylabel("Raw amplitude")
axes[1].plot(time_axis, processed_segment, color=MUTED_BLUE, linewidth=1.8)
axes[1].set_title(f"After preprocessing: subject {subject_id}, {CHANNELS[channel_idx]}")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Preprocessed amplitude")
for ax in axes:
    ax.axvline(time_axis[local_center], color=MUTED_RED, linestyle="--", linewidth=1.4)
    ax.set_xlim(time_axis[0], time_axis[-1])
    ax.grid(True, alpha=0.18, linewidth=0.8)
set_visible_signal_limits(axes[0], raw_segment)
set_visible_signal_limits(axes[1], processed_segment)
s4_path = save_supplementary_figure(fig, "supplementary_figure_S04_raw_vs_preprocessed_segment.png", "Supplementary Figure S4", "Cell 72", "Raw EEG segment above and preprocessed segment below for direct visual comparison.")


Figure saved at OLD_Docs/outputs_reviewer_ready_v2/supplementary_figures/supplementary_figure_S01_clipping_outlier_proportion_by_channel.png

Figure saved at OLD_Docs/outputs_reviewer_ready_v2/supplementary_figures/supplementary_figure_S02_mean_amplitude_difference_by_channel.png

,channel,theta_difference,beta_difference,combined_abs_effect
16,Fz,-0.0455,0.0581,0.1036
6,P3,-0.0473,0.0543,0.1016
4,C3,-0.0453,0.0514,0.0967
18,Pz,-0.0428,0.0537,0.0965
2,F3,-0.0434,0.0490,0.0924
8,O1,-0.0405,0.0403,0.0809
12,T7,-0.0362,0.0445,0.0808
14,P7,-0.0342,0.0415,0.0757
5,C4,-0.0285,0.0372,0.0657
17,Cz,-0.0242,0.0367,0.0609


Figure saved at OLD_Docs/outputs_reviewer_ready_v2/supplementary_figures/supplementary_figure_S03_channel_level_spectral_effect_ranking.png

Figure saved at OLD_Docs/outputs_reviewer_ready_v2/supplementary_figures/supplementary_figure_S04_raw_vs_preprocessed_segment.png

In [70]:
# Cell 73: Supplementary Figures S5-S8: raw-correlation channel summaries, difference maps, eigenvectors, and communities.
mean_abs_corr = pd.DataFrame(
    {
        "channel": CHANNELS,
        "all_subjects": np.mean(np.abs(raw_all_corr - np.eye(len(CHANNELS))), axis=1),
        "control": np.mean(np.abs(raw_control_corr - np.eye(len(CHANNELS))), axis=1),
        "ADHD": np.mean(np.abs(raw_adhd_corr - np.eye(len(CHANNELS))), axis=1),
    }
)
fig, ax = plt.subplots(figsize=(14, 8), constrained_layout=True)
bars = ax.bar(mean_abs_corr["channel"], mean_abs_corr["ADHD"] - mean_abs_corr["control"], color=MUTED_PURPLE)
add_vertical_bar_labels(ax, bars, "{:.2f}")
ax.set_ylabel("ADHD-control mean absolute raw correlation")
ax.set_title("Raw-Amplitude Mean Absolute Interchannel Correlation by Channel")
ax.set_xticklabels(CHANNELS, rotation=45, ha="right")
s5_path = save_supplementary_figure(fig, "supplementary_figure_S05_raw_mean_absolute_correlation_by_channel.png", "Supplementary Figure S5", "Cell 73", "Raw-amplitude mean absolute interchannel correlation by channel; descriptive only.")

raw_corr_difference = raw_adhd_corr - raw_control_corr
fig, ax = plt.subplots(figsize=(18, 16), constrained_layout=True)
plot_matrix_on_axis(ax, raw_corr_difference, "Raw Correlation Difference: ADHD-Control", CHANNELS, CHANNELS, cmap="coolwarm", center=0.0, vmin=-np.max(np.abs(raw_corr_difference)), vmax=np.max(np.abs(raw_corr_difference)), colorbar_label="Difference", annotate=True)
s6_path = save_supplementary_figure(fig, "supplementary_figure_S06_raw_correlation_difference_map.png", "Supplementary Figure S6", "Cell 73", "Raw-amplitude correlation difference map, ADHD minus control; descriptive only.")

eigvals, eigvecs = np.linalg.eigh(raw_all_corr)
leading = np.abs(eigvecs[:, np.argmax(eigvals)])
fig, ax = plt.subplots(figsize=(14, 8), constrained_layout=True)
bars = ax.bar(CHANNELS, leading, color=MUTED_BLUE)
add_vertical_bar_labels(ax, bars, "{:.2f}")
ax.set_ylabel("Absolute loading")
ax.set_title("Leading Eigenvector Loading of Raw Correlation Matrix")
ax.set_xticklabels(CHANNELS, rotation=45, ha="right")
s7_path = save_supplementary_figure(fig, "supplementary_figure_S07_raw_correlation_leading_eigenvector.png", "Supplementary Figure S7", "Cell 73", "Leading eigenvector loading of the raw-amplitude correlation matrix; descriptive only.")

community_model = SpectralClustering(n_clusters=4, affinity="precomputed", random_state=SEED, assign_labels="kmeans")
community_labels = community_model.fit_predict(np.abs(raw_all_corr))
order = np.lexsort((np.arange(len(CHANNELS)), community_labels))
ordered_channels = [CHANNELS[i] for i in order]
fig, axes = plt.subplots(1, 2, figsize=(34, 16), constrained_layout=True)
plot_matrix_on_axis(axes[0], raw_control_corr[np.ix_(order, order)], "Control community-ordered raw correlation", ordered_channels, ordered_channels, cmap="coolwarm", center=0.0, vmin=-1, vmax=1, colorbar_label="r", annotate=True)
plot_matrix_on_axis(axes[1], raw_adhd_corr[np.ix_(order, order)], "ADHD community-ordered raw correlation", ordered_channels, ordered_channels, cmap="coolwarm", center=0.0, vmin=-1, vmax=1, colorbar_label="r", annotate=True)
s8_path = save_supplementary_figure(fig, "supplementary_figure_S08_community_structured_raw_correlation.png", "Supplementary Figure S8", "Cell 73", "Community-structured raw-amplitude correlation matrices, control and ADHD side by side.")


Figure saved at OLD_Docs/outputs_reviewer_ready_v2/supplementary_figures/supplementary_figure_S05_raw_mean_absolute_correlation_by_channel.png

Figure saved at OLD_Docs/outputs_reviewer_ready_v2/supplementary_figures/supplementary_figure_S06_raw_correlation_difference_map.png

Figure saved at OLD_Docs/outputs_reviewer_ready_v2/supplementary_figures/supplementary_figure_S07_raw_correlation_leading_eigenvector.png

Figure saved at OLD_Docs/outputs_reviewer_ready_v2/supplementary_figures/supplementary_figure_S08_community_structured_raw_correlation.png

In [71]:
# Cell 74: Supplementary Figures S9-S12: dynamic-state occupancy, PCA, ROC, and sparse edge weights.
from sklearn.decomposition import PCA

sequence_mean_features = X_seq_edges.mean(axis=1)
kmeans = KMeans(n_clusters=3, random_state=SEED, n_init=20)
state_labels = kmeans.fit_predict(sequence_mean_features)
state_df = sequence_meta_df[["ID", "label", "class_name"]].copy()
state_df["state"] = state_labels
occupancy_df = state_df.groupby(["class_name", "state"]).size().rename("n_sequences").reset_index()
occupancy_df["occupancy"] = occupancy_df["n_sequences"] / occupancy_df.groupby("class_name")["n_sequences"].transform("sum")
fig, ax = plt.subplots(figsize=(12, 8), constrained_layout=True)
states = sorted(occupancy_df["state"].unique())
x = np.arange(len(states))
width = 0.36
for offset, class_name, color in [(-width / 2, "Control", MUTED_BLUE), (width / 2, "ADHD", MUTED_RED)]:
    vals = occupancy_df[occupancy_df["class_name"] == class_name].set_index("state").reindex(states)["occupancy"].fillna(0).to_numpy()
    bars = ax.bar(x + offset, vals, width=width, color=color, label=class_name)
    for bar in bars:
        ax.annotate(f"{bar.get_height():.2f}", (bar.get_x() + bar.get_width() / 2, bar.get_height()), xytext=(0, 6), textcoords="offset points", ha="center", va="bottom", fontsize=PLOT_FONT_SIZE, clip_on=False)
ax.set_ylim(0, max(1.0, occupancy_df["occupancy"].max()) * 1.20)
ax.set_xticks(x)
ax.set_xticklabels([f"State {s}" for s in states])
ax.set_ylabel("Sequence occupancy")
ax.set_title("Dynamic-State Occupancy by Group")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.22), ncol=2)
s9_path = save_supplementary_figure(fig, "supplementary_figure_S09_dynamic_state_occupancy.png", "Supplementary Figure S9", "Cell 74", "Dynamic-state occupancy by group using KMeans on mean sequence connectivity features.")

pca = PCA(n_components=min(20, sequence_mean_features.shape[1]), random_state=SEED)
pca.fit(StandardScaler().fit_transform(sequence_mean_features))
explained = np.cumsum(pca.explained_variance_ratio_)
fig, ax = plt.subplots(figsize=(12, 8), constrained_layout=True)
ax.plot(np.arange(1, explained.size + 1), explained, marker="o", color=MUTED_GREEN)
ax.set_ylim(0, 1.05)
ax.set_xlabel("Number of principal components")
ax.set_ylabel("Cumulative explained variance")
ax.set_title("PCA Explained Variance of Connectivity Features")
s10_path = save_supplementary_figure(fig, "supplementary_figure_S10_pca_explained_variance.png", "Supplementary Figure S10", "Cell 74", "PCA explained variance of connectivity edge features.")

fig, ax = plt.subplots(figsize=(12, 10), constrained_layout=True)
for model_name in classification_results_full_df["model"].tolist():
    subject_scores = aggregate_subject_scores(sequence_meta_test, model_score_store[model_name]["test"], SUBJECT_AGGREGATION_METHOD)
    fpr, tpr, _ = roc_curve(subject_scores["label"].to_numpy(), subject_scores["score"].to_numpy())
    auc_value = roc_auc_score(subject_scores["label"].to_numpy(), subject_scores["score"].to_numpy())
    ax.plot(fpr, tpr, label=f"{DISPLAY_NAME_MAP.get(model_name, model_name)} ({auc_value:.2f})")
ax.plot([0, 1], [0, 1], color=MUTED_GREY, linestyle="--", linewidth=1.4, label="Chance")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC Curves for All Main Models")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.38), ncol=2)
s11_path = save_supplementary_figure(fig, "supplementary_figure_S11_roc_curves_all_models.png", "Supplementary Figure S11", "Cell 74", "ROC curves for all main models on the subject-level test set.")

sparse_estimator = sparse_model.named_steps["logisticregression"] if hasattr(sparse_model, "named_steps") and "logisticregression" in sparse_model.named_steps else sparse_model[-1]
sparse_coef = np.asarray(sparse_estimator.coef_).ravel()
theta_coef = sparse_coef[:n_edges]
beta_coef = sparse_coef[n_edges : 2 * n_edges]
combined_coef_matrix = np.zeros((len(CHANNELS), len(CHANNELS)), dtype=float)
combined_edge_weights = np.abs(theta_coef) + np.abs(beta_coef)
combined_coef_matrix[TRIU_IDX] = combined_edge_weights
combined_coef_matrix = combined_coef_matrix + combined_coef_matrix.T
fig, ax = plt.subplots(figsize=(18, 16), constrained_layout=True)
plot_matrix_on_axis(ax, combined_coef_matrix, "L1 Logistic Edge-Weight Heatmap", CHANNELS, CHANNELS, cmap="viridis", center=None, colorbar_label="|theta weight| + |beta weight|", annotate=True)
s12_path = save_supplementary_figure(fig, "supplementary_figure_S12_l1_logistic_edge_weight_heatmap.png", "Supplementary Figure S12", "Cell 74", "L1 logistic regression edge-weight heatmap for sparse oscillatory model.")


Figure saved at OLD_Docs/outputs_reviewer_ready_v2/supplementary_figures/supplementary_figure_S09_dynamic_state_occupancy.png

Figure saved at OLD_Docs/outputs_reviewer_ready_v2/supplementary_figures/supplementary_figure_S10_pca_explained_variance.png

Figure saved at OLD_Docs/outputs_reviewer_ready_v2/supplementary_figures/supplementary_figure_S11_roc_curves_all_models.png

Figure saved at OLD_Docs/outputs_reviewer_ready_v2/supplementary_figures/supplementary_figure_S12_l1_logistic_edge_weight_heatmap.png

In [72]:
# Cell 75: Supplementary Figures S13-S16: nodal load, confusion matrices, adjacency thresholds, and window/frequency sensitivity.
nodal_load = combined_coef_matrix.sum(axis=1)
fig, ax = plt.subplots(figsize=(14, 8), constrained_layout=True)
bars = ax.bar(CHANNELS, nodal_load, color=MUTED_TEAL)
add_vertical_bar_labels(ax, bars, "{:.2f}")
ax.set_ylabel("Nodal discriminative load")
ax.set_title("Nodal Discriminative Load by EEG Channel")
ax.set_xticklabels(CHANNELS, rotation=45, ha="right")
s13_path = save_supplementary_figure(fig, "supplementary_figure_S13_nodal_discriminative_load.png", "Supplementary Figure S13", "Cell 75", "Nodal discriminative load derived from sparse edge weights.")

model_order = classification_results_full_df["model"].tolist()
n_cols = 4
n_rows = int(math.ceil(len(model_order) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.0 * n_cols, 4.0 * n_rows), constrained_layout=True)
axes = np.asarray(axes).ravel()
for ax, model_name in zip(axes, model_order):
    subject_scores = aggregate_subject_scores(sequence_meta_test, model_score_store[model_name]["test"], SUBJECT_AGGREGATION_METHOD)
    preds = (subject_scores["score"].to_numpy() >= FIXED_DECISION_THRESHOLD).astype(int)
    cm = confusion_matrix(subject_scores["label"].to_numpy(), preds, labels=[0, 1])
    image = ax.imshow(cm, cmap="Blues", vmin=0)
    ax.set_title(DISPLAY_NAME_MAP.get(model_name, model_name))
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Control", "ADHD"])
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["Control", "ADHD"])
    add_matrix_annotations(ax, image, cm, fmt=".0f", fontsize=PLOT_FONT_SIZE)
for ax in axes[len(model_order):]:
    ax.axis("off")
s14_path = save_supplementary_figure(fig, "supplementary_figure_S14_confusion_matrices_all_models.png", "Supplementary Figure S14", "Cell 75", "Confusion matrices for all main models.")

threshold_rows = []
for threshold in THRESHOLD_CANDIDATES:
    X_topo_train_thr = build_sequence_graph_metric_features(X_seq_graph_train, threshold=threshold)
    X_topo_test_thr = build_sequence_graph_metric_features(X_seq_graph_test, threshold=threshold)
    thr_model = make_global_graph_metric_network_index_model()
    thr_model.fit(X_topo_train_thr, y_train)
    thr_scores = thr_model.decision_function(X_topo_test_thr)
    subject_scores = aggregate_subject_scores(sequence_meta_test, thr_scores, SUBJECT_AGGREGATION_METHOD)
    metrics = compute_classification_metrics(subject_scores["label"].to_numpy(), subject_scores["score"].to_numpy(), FIXED_DECISION_THRESHOLD)
    threshold_rows.append({"adjacency_threshold": threshold, **metrics})
threshold_sensitivity_df = pd.DataFrame(threshold_rows)
display(threshold_sensitivity_df.round(3))
save_table(threshold_sensitivity_df, "adjacency_threshold_sensitivity.csv", "Adjacency threshold sensitivity", "Cell 75", "Lightweight graph-metric sensitivity to adjacency threshold values.")
fig, ax = plt.subplots(figsize=(12, 8), constrained_layout=True)
for metric, color in [("balanced_accuracy", MUTED_BLUE), ("auc_roc", MUTED_RED)]:
    ax.plot(threshold_sensitivity_df["adjacency_threshold"], threshold_sensitivity_df[metric], marker="o", label=metric, color=color)
ax.set_xlabel("Adjacency threshold")
ax.set_ylabel("Metric value")
ax.set_ylim(0, 1.05)
ax.set_title("Threshold-Sensitivity Plot")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.22), ncol=2)
s15_path = save_supplementary_figure(fig, "supplementary_figure_S15_threshold_sensitivity.png", "Supplementary Figure S15", "Cell 75", "Threshold-sensitivity plot for graph-metric adjacency thresholds.")

window_sensitivity_df = sampling_frequency_impact_df.copy().sort_values("sampling_frequency_hz")
frequency_labels = [f"{value:.0f} Hz" for value in window_sensitivity_df["sampling_frequency_hz"]]
x = np.arange(len(frequency_labels))
fig, axes = plt.subplots(1, 2, figsize=(16, 6.5), constrained_layout=True)

duration_width = 0.34
window_bars = axes[0].bar(
    x - duration_width / 2,
    window_sensitivity_df["window_length_seconds"],
    width=duration_width,
    color=MUTED_BLUE,
    label="Window duration",
)
sequence_bars = axes[0].bar(
    x + duration_width / 2,
    window_sensitivity_df["approx_sequence_duration_seconds"],
    width=duration_width,
    color=MUTED_RED,
    label="Sequence duration",
)
axes[0].set_xticks(x)
axes[0].set_xticklabels(frequency_labels)
axes[0].set_ylabel("Duration (s)")
axes[0].set_title("Window and Sequence Duration")
add_vertical_bar_labels(axes[0], window_bars, "{:.2f}")
add_vertical_bar_labels(axes[0], sequence_bars, "{:.2f}")
axes[0].legend(loc="upper right", frameon=False)

resolution_bars = axes[1].bar(
    x,
    window_sensitivity_df["frequency_resolution_hz"],
    width=0.46,
    color=MUTED_GREEN,
)
axes[1].set_xticks(x)
axes[1].set_xticklabels(frequency_labels)
axes[1].set_ylabel("Frequency resolution (Hz)")
axes[1].set_title("Frequency Resolution")
add_vertical_bar_labels(axes[1], resolution_bars, "{:.2f}")
fig.suptitle("Window-Length and Frequency-Resolution Sensitivity", fontsize=PLOT_TITLE_SIZE)
s16_path = save_supplementary_figure(fig, "supplementary_figure_S16_window_frequency_sensitivity.png", "Supplementary Figure S16", "Cell 75", "Window-duration and frequency-resolution sensitivity under 500 Hz and 128 Hz.")


Figure saved at OLD_Docs/outputs_reviewer_ready_v2/supplementary_figures/supplementary_figure_S13_nodal_discriminative_load.png

Figure saved at OLD_Docs/outputs_reviewer_ready_v2/supplementary_figures/supplementary_figure_S14_confusion_matrices_all_models.png

,adjacency_threshold,threshold,accuracy,balanced_accuracy,sensitivity,specificity,precision,f1,auc_roc,tn,fp,fn,tp
0,0.2,0.0,0.421,0.411,0.6,0.222,0.462,0.522,0.444,2,7,4,6
1,0.3,0.0,0.368,0.356,0.6,0.111,0.429,0.500,0.433,1,8,4,6
2,0.4,0.0,0.579,0.567,0.8,0.333,0.571,0.667,0.633,3,6,2,8
3,0.5,0.0,0.526,0.517,0.7,0.333,0.538,0.609,0.533,3,6,3,7


Figure saved at OLD_Docs/outputs_reviewer_ready_v2/supplementary_figures/supplementary_figure_S15_threshold_sensitivity.png

Figure saved at OLD_Docs/outputs_reviewer_ready_v2/supplementary_figures/supplementary_figure_S16_window_frequency_sensitivity.png

## Section 20. Model Architecture Preservation Audit


In [73]:
# Cell 76: Create model architecture and hyperparameter audit tables.
architecture_rows = [
    {
        "model name": "Linear network index",
        "input dimension": X_lin_train.shape[1],
        "output dimension": 1,
        "hidden size": "not applicable",
        "dropout": "not applicable",
        "number of transformer heads": "not applicable",
        "number of encoder layers": "not applicable",
        "GRU hidden size": "not applicable",
        "optimizer": "sklearn LogisticRegression",
        "learning rate": "liblinear default",
        "weight decay": "L2 via C=1.0",
        "batch size": "not applicable",
        "max epochs": "not applicable",
        "early stopping": "not applicable",
    },
    {
        "model name": "Sparse oscillatory network index",
        "input dimension": X_lin_train.shape[1],
        "output dimension": 1,
        "hidden size": "not applicable",
        "dropout": "not applicable",
        "number of transformer heads": "not applicable",
        "number of encoder layers": "not applicable",
        "GRU hidden size": "not applicable",
        "optimizer": "sklearn LogisticRegression",
        "learning rate": "liblinear default",
        "weight decay": "L1 via C=0.25",
        "batch size": "not applicable",
        "max epochs": "max_iter=5000",
        "early stopping": "not applicable",
    },
    {
        "model name": "Global graph-metric network index",
        "input dimension": X_topo_train.shape[1],
        "output dimension": 1,
        "hidden size": "not applicable",
        "dropout": "not applicable",
        "number of transformer heads": "not applicable",
        "number of encoder layers": "not applicable",
        "GRU hidden size": "not applicable",
        "optimizer": "sklearn LogisticRegression",
        "learning rate": "liblinear default",
        "weight decay": "L2 via C=1.0",
        "batch size": "not applicable",
        "max epochs": "not applicable",
        "early stopping": "not applicable",
    },
    {
        "model name": "GRU dynamic connectivity index",
        "input dimension": X_seq_edges_train_std.shape[-1],
        "output dimension": 1,
        "hidden size": 64,
        "dropout": 0.25,
        "number of transformer heads": "not applicable",
        "number of encoder layers": "not applicable",
        "GRU hidden size": 64,
        "optimizer": "AdamW",
        "learning rate": 1e-3,
        "weight decay": 1e-4,
        "batch size": BATCH_SIZE,
        "max epochs": 12,
        "early stopping": "patience=3",
    },
    {
        "model name": "Static GCN brain-network index",
        "input dimension": f"{len(CHANNELS)} x {len(CHANNELS)} static adjacency",
        "output dimension": 1,
        "hidden size": 32,
        "dropout": 0.20,
        "number of transformer heads": "not applicable",
        "number of encoder layers": "not applicable",
        "GRU hidden size": "not applicable",
        "optimizer": "AdamW",
        "learning rate": 1e-3,
        "weight decay": 1e-4,
        "batch size": BATCH_SIZE,
        "max epochs": 12,
        "early stopping": "patience=3",
    },
    {
        "model name": "Spatio-temporal GCN brain-network index",
        "input dimension": f"{SEQ_LEN} x {len(CHANNELS)} x {len(CHANNELS)} dynamic adjacency",
        "output dimension": 1,
        "hidden size": 32,
        "dropout": 0.20,
        "number of transformer heads": "not applicable",
        "number of encoder layers": "not applicable",
        "GRU hidden size": "not applicable",
        "optimizer": "AdamW",
        "learning rate": 1e-3,
        "weight decay": 1e-4,
        "batch size": BATCH_SIZE,
        "max epochs": 12,
        "early stopping": "patience=3",
    },
    {
        "model name": PROPOSED_MODEL_NAME,
        "input dimension": X_seq_edges_train_std.shape[-1],
        "output dimension": 1,
        "hidden size": proposed_architecture_kwargs["d_model"],
        "dropout": proposed_architecture_kwargs["dropout"],
        "number of transformer heads": proposed_architecture_kwargs["n_heads"],
        "number of encoder layers": proposed_architecture_kwargs["n_layers"],
        "GRU hidden size": proposed_architecture_kwargs["gru_hidden"],
        "optimizer": "AdamW",
        "learning rate": proposed_training_kwargs["lr"],
        "weight decay": proposed_training_kwargs["weight_decay"],
        "batch size": BATCH_SIZE,
        "max epochs": proposed_training_kwargs["max_epochs"],
        "early stopping": f"patience={proposed_training_kwargs['patience']}",
    },
    {
        "model name": ablation_name,
        "input dimension": X_seq_edges_train_std.shape[-1],
        "output dimension": 1,
        "hidden size": ablation_architecture_kwargs["d_model"],
        "dropout": ablation_architecture_kwargs["dropout"],
        "number of transformer heads": ablation_architecture_kwargs["n_heads"],
        "number of encoder layers": ablation_architecture_kwargs["n_layers"],
        "GRU hidden size": ablation_architecture_kwargs["gru_hidden"],
        "optimizer": "AdamW",
        "learning rate": ablation_training_kwargs["lr"],
        "weight decay": ablation_training_kwargs["weight_decay"],
        "batch size": BATCH_SIZE,
        "max epochs": ablation_training_kwargs["max_epochs"],
        "early stopping": f"patience={ablation_training_kwargs['patience']}",
    },
]
model_architecture_audit_df = pd.DataFrame(architecture_rows)
display(model_architecture_audit_df)
save_table(model_architecture_audit_df, "model_architecture_audit.csv", "Model architecture audit", "Cell 76", "Architecture and training-configuration audit for all evaluated models.", excel=True)

model_hyperparameter_table_df = model_architecture_audit_df.copy()
model_hyperparameter_table_df["original notebook comparison"] = np.where(
    model_hyperparameter_table_df["model name"].eq(PROPOSED_MODEL_NAME),
    "Matches the executed v1/v2 proposed architecture: Transformer branch, positional embedding, GRU branch, fusion, LayerNorm, dropout, scalar output.",
    "Executed configuration preserved from the v1 notebook definitions.",
)
display(model_hyperparameter_table_df)
save_table(model_hyperparameter_table_df, "model_hyperparameter_table.csv", "Model hyperparameter table", "Cell 76", "Hyperparameter table with original-notebook comparison notes.", excel=True)


,model name,input dimension,output dimension,hidden size,dropout,number of transformer heads,number of encoder layers,GRU hidden size,optimizer,learning rate,weight decay,batch size,max epochs,early stopping
0,Linear network index,342,1,not applicable,not applicable,not applicable,not applicable,not applicable,sklearn LogisticRegression,liblinear default,L2 via C=1.0,not applicable,not applicable,not applicable
1,Sparse oscillatory network index,342,1,not applicable,not applicable,not applicable,not applicable,not applicable,sklearn LogisticRegression,liblinear default,L1 via C=0.25,not applicable,max_iter=5000,not applicable
2,Global graph-metric network index,7,1,not applicable,not applicable,not applicable,not applicable,not applicable,sklearn LogisticRegression,liblinear default,L2 via C=1.0,not applicable,not applicable,not applicable
3,GRU dynamic connectivity index,342,1,64,0.25,not applicable,not applicable,64,AdamW,0.001,0.0001,64,12,patience=3
4,Static GCN brain-network index,19 x 19 static adjacency,1,32,0.2,not applicable,not applicable,not applicable,AdamW,0.001,0.0001,64,12,patience=3
5,Spatio-temporal GCN brain-network index,10 x 19 x 19 dynamic adjacency,1,32,0.2,not applicable,not applicable,not applicable,AdamW,0.001,0.0001,64,12,patience=3
6,Hybrid Multihead Spatio-Temporal graph transfo...,342,1,64,0.25,4,1,64,AdamW,0.0004,0.0003,64,16,patience=5
7,Hybrid Multihead Spatio-Temporal graph transfo...,342,1,64,0.15,4,1,64,AdamW,0.0005,0.0001,64,16,patience=5


,model name,input dimension,output dimension,hidden size,dropout,number of transformer heads,number of encoder layers,GRU hidden size,optimizer,learning rate,weight decay,batch size,max epochs,early stopping,original notebook comparison
0,Linear network index,342,1,not applicable,not applicable,not applicable,not applicable,not applicable,sklearn LogisticRegression,liblinear default,L2 via C=1.0,not applicable,not applicable,not applicable,Executed configuration preserved from the v1 n...
1,Sparse oscillatory network index,342,1,not applicable,not applicable,not applicable,not applicable,not applicable,sklearn LogisticRegression,liblinear default,L1 via C=0.25,not applicable,max_iter=5000,not applicable,Executed configuration preserved from the v1 n...
2,Global graph-metric network index,7,1,not applicable,not applicable,not applicable,not applicable,not applicable,sklearn LogisticRegression,liblinear default,L2 via C=1.0,not applicable,not applicable,not applicable,Executed configuration preserved from the v1 n...
3,GRU dynamic connectivity index,342,1,64,0.25,not applicable,not applicable,64,AdamW,0.001,0.0001,64,12,patience=3,Executed configuration preserved from the v1 n...
4,Static GCN brain-network index,19 x 19 static adjacency,1,32,0.2,not applicable,not applicable,not applicable,AdamW,0.001,0.0001,64,12,patience=3,Executed configuration preserved from the v1 n...
5,Spatio-temporal GCN brain-network index,10 x 19 x 19 dynamic adjacency,1,32,0.2,not applicable,not applicable,not applicable,AdamW,0.001,0.0001,64,12,patience=3,Executed configuration preserved from the v1 n...
6,Hybrid Multihead Spatio-Temporal graph transfo...,342,1,64,0.25,4,1,64,AdamW,0.0004,0.0003,64,16,patience=5,Matches the executed v1/v2 proposed architectu...
7,Hybrid Multihead Spatio-Temporal graph transfo...,342,1,64,0.15,4,1,64,AdamW,0.0005,0.0001,64,16,patience=5,Executed configuration preserved from the v1 n...


PosixPath('/Users/talgatazykanov/Desktop/Science works/Makbal_PhD/outputs_reviewer_ready_v2/tables/model_hyperparameter_table.csv')

## Section 22. Overfitting and Generalization Analysis


In [74]:
# Cell 78: Create train/validation/test comparison, generalization gaps, and overfitting audit tables.
train_validation_test_comparison_df = all_subject_results_df.copy()
train_validation_test_comparison_df["display_model"] = train_validation_test_comparison_df["model"].map(DISPLAY_NAME_MAP)
display(train_validation_test_comparison_df[["display_model", "split", "accuracy", "balanced_accuracy", "auc_roc", "cohens_d", "distribution_overlap"]].round(3))
save_table(train_validation_test_comparison_df, "train_validation_test_comparison.csv", "Train validation test comparison", "Cell 78", "Subject-level train/validation/test metrics for all models.")

gap_rows = []
for model_name in train_validation_test_comparison_df["model"].unique():
    model_df = train_validation_test_comparison_df[train_validation_test_comparison_df["model"] == model_name].set_index("split")
    if {"train", "test"}.issubset(model_df.index):
        gap_rows.append(
            {
                "model": DISPLAY_NAME_MAP.get(model_name, model_name),
                "train_auc_minus_test_auc": float(model_df.loc["train", "auc_roc"] - model_df.loc["test", "auc_roc"]),
                "train_balanced_accuracy_minus_test_balanced_accuracy": float(model_df.loc["train", "balanced_accuracy"] - model_df.loc["test", "balanced_accuracy"]),
                "train_cohens_d_minus_test_cohens_d": float(model_df.loc["train", "cohens_d"] - model_df.loc["test", "cohens_d"]),
                "train_overlap_minus_test_overlap": float(model_df.loc["train", "distribution_overlap"] - model_df.loc["test", "distribution_overlap"]),
            }
        )
generalization_gap_table_df = pd.DataFrame(gap_rows)
display(generalization_gap_table_df.round(3))
save_table(generalization_gap_table_df, "generalization_gap_table.csv", "Generalization gap table", "Cell 78", "Train-test gaps in AUC, balanced accuracy, Cohen's d, and overlap.")

overfitting_audit_table_df = generalization_gap_table_df.copy()
overfitting_audit_table_df["interpretation"] = np.where(
    overfitting_audit_table_df["train_auc_minus_test_auc"] > 0.20,
    "Large train-test AUC gap; interpret test performance conservatively.",
    "No large AUC gap by the 0.20 descriptive threshold, but external validation is still required.",
)
display(overfitting_audit_table_df)
save_table(overfitting_audit_table_df, "overfitting_audit_table.csv", "Overfitting audit table", "Cell 78", "Generalization-gap interpretation for reviewer response.")


,display_model,split,accuracy,balanced_accuracy,auc_roc,cohens_d,distribution_overlap
0,Linear network index,train,1.000,1.000,1.000,4.707,0.019
1,Linear network index,validation,0.611,0.611,0.679,0.747,0.709
2,Linear network index,test,0.526,0.517,0.467,0.040,0.984
3,Sparse oscillatory index,train,1.000,1.000,1.000,3.689,0.065
4,Sparse oscillatory index,validation,0.611,0.611,0.667,0.765,0.702
5,Sparse oscillatory index,test,0.474,0.467,0.467,0.092,0.963
6,Global graph-metric index,train,0.679,0.679,0.826,1.308,0.513
7,Global graph-metric index,validation,0.667,0.667,0.605,0.480,0.810
8,Global graph-metric index,test,0.579,0.567,0.633,0.551,0.783
9,GRU dynamic index,train,0.512,0.512,0.432,-0.278,0.889


,model,train_auc_minus_test_auc,train_balanced_accuracy_minus_test_balanced_accuracy,train_cohens_d_minus_test_cohens_d,train_overlap_minus_test_overlap
0,Linear network index,0.533,0.483,4.667,-0.965
1,Sparse oscillatory index,0.533,0.533,3.597,-0.898
2,Global graph-metric index,0.193,0.112,0.757,-0.270
3,GRU dynamic index,-0.224,-0.060,-0.942,0.149
4,Static GCN index,-0.066,0.000,-0.147,-0.058
5,Spatio-temporal GCN index,0.018,0.000,0.189,0.075
6,Proposed Hybrid ST Graph Transformer,-0.392,-0.206,-1.174,0.109
7,Proposed model ablation,-0.372,-0.170,-1.109,0.085


,model,train_auc_minus_test_auc,train_balanced_accuracy_minus_test_balanced_accuracy,train_cohens_d_minus_test_cohens_d,train_overlap_minus_test_overlap,interpretation
0,Linear network index,0.533333,0.483333,4.666920,-0.965264,Large train-test AUC gap; interpret test perfo...
1,Sparse oscillatory index,0.533333,0.533333,3.596972,-0.898098,Large train-test AUC gap; interpret test perfo...
2,Global graph-metric index,0.192630,0.111905,0.756715,-0.269698,No large AUC gap by the 0.20 descriptive thres...
3,GRU dynamic index,-0.223583,-0.060317,-0.941828,0.149460,No large AUC gap by the 0.20 descriptive thres...
4,Static GCN index,-0.066327,0.000000,-0.147271,-0.057513,No large AUC gap by the 0.20 descriptive thres...
5,Spatio-temporal GCN index,0.018254,0.000000,0.189164,0.075270,No large AUC gap by the 0.20 descriptive thres...
6,Proposed Hybrid ST Graph Transformer,-0.391837,-0.205556,-1.173617,0.108518,No large AUC gap by the 0.20 descriptive thres...
7,Proposed model ablation,-0.371655,-0.169841,-1.108560,0.084905,No large AUC gap by the 0.20 descriptive thres...


PosixPath('/Users/talgatazykanov/Desktop/Science works/Makbal_PhD/outputs_reviewer_ready_v2/tables/overfitting_audit_table.csv')

In [75]:
# Cell 79: Generate learning curves for dynamic models with loss and validation AUC columns where available.
dynamic_history_names = [
    "GRU dynamic connectivity index",
    "Spatio-temporal GCN brain-network index",
    PROPOSED_MODEL_NAME,
    ablation_name,
]
fig, axes = plt.subplots(2, 2, figsize=(22, 14), constrained_layout=True)
for model_name in dynamic_history_names:
    if model_name not in model_histories:
        continue
    history = model_histories[model_name]
    label = DISPLAY_NAME_MAP.get(model_name, model_name)
    axes[0, 0].plot(history["epoch"], history["train_loss"], marker="o", label=label)
    axes[0, 1].plot(history["epoch"], history["val_loss"], marker="o", label=label)
    if "val_sequence_auc" in history.columns:
        axes[1, 0].plot(history["epoch"], history["val_sequence_auc"], marker="o", label=label)
    elif "val_auc" in history.columns:
        axes[1, 0].plot(history["epoch"], history["val_auc"], marker="o", label=label)
    if "val_subject_auc" in history.columns:
        axes[1, 1].plot(history["epoch"], history["val_subject_auc"], marker="o", label=label)
for ax, title, ylabel in [
    (axes[0, 0], "Train loss", "Loss"),
    (axes[0, 1], "Validation loss", "Loss"),
    (axes[1, 0], "Sequence-level validation AUC", "AUC"),
    (axes[1, 1], "Subject-level validation AUC", "AUC"),
]:
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    if ylabel == "AUC":
        ax.set_ylim(0, 1.05)
    ax.legend(loc="best")
saved_path = save_main_figure(fig, "technical_learning_curves_dynamic_models.png", "Dynamic learning curves", "Cell 79", "Training and validation dynamics for dynamic/deep models.")
display(pd.DataFrame([("Saved figure", str(saved_path))], columns=["Item", "Value"]))

ablation_summary_df = pd.DataFrame(
    [
        {
            "configuration": "Proposed main",
            "dropout": proposed_architecture_kwargs["dropout"],
            "weight_decay": proposed_training_kwargs["weight_decay"],
            "test_balanced_accuracy": float(v2_prop_class["balanced_accuracy"]),
            "test_auc_roc": float(v2_prop_class["auc_roc"]),
        },
        {
            "configuration": "Proposed reduced-regularization ablation",
            "dropout": ablation_architecture_kwargs["dropout"],
            "weight_decay": ablation_training_kwargs["weight_decay"],
            "test_balanced_accuracy": float(classification_results_full_df[classification_results_full_df["model"] == ablation_name].iloc[0]["balanced_accuracy"]),
            "test_auc_roc": float(classification_results_full_df[classification_results_full_df["model"] == ablation_name].iloc[0]["auc_roc"]),
        },
    ]
)
display(ablation_summary_df.round(3))
save_table(ablation_summary_df, "ablation_summary.csv", "Ablation summary", "Cell 79", "Proposed main model versus reduced-regularization ablation.")


Figure saved at OLD_Docs/outputs_reviewer_ready_v2/main_figures/technical_learning_curves_dynamic_models.png

,Item,Value
0,Saved figure,/Users/talgatazykanov/Desktop/Science works/Ma...


,configuration,dropout,weight_decay,test_balanced_accuracy,test_auc_roc
0,Proposed main,0.25,0.0,0.622,0.744
1,Proposed reduced-regularization ablation,0.15,0.0,0.622,0.733


PosixPath('/Users/talgatazykanov/Desktop/Science works/Makbal_PhD/outputs_reviewer_ready_v2/tables/ablation_summary.csv')

In [76]:
# Cell 80: Create a conservative interpretation table for generalization.
conservative_interpretation_table_df = pd.DataFrame(
    [
        ("Subject-level fixed split performance", "Preliminary within-dataset discrimination under a subject-level split", "External clinical diagnostic validity"),
        ("Bootstrap confidence intervals", "Uncertainty is wide on the small test set", "Precision comparable to large multicenter validation"),
        ("Repeated lightweight splits", "Some split-sensitivity context for non-deep baselines", "Full repeated deep-model validation"),
        ("LOSO lightweight validation", "Baseline subject-level cross-validation context", "Transformer/GRU LOSO equivalence"),
        ("Envelope connectivity model input", "Final model uses theta/beta amplitude-envelope connectivity", "Raw-amplitude correlations as final model evidence"),
        ("Clinical screening scenario", "Potential decision-support framing at assumed prevalences", "Standalone ADHD diagnosis"),
    ],
    columns=["result component", "what the result supports", "what it does not support"],
)
display(conservative_interpretation_table_df)
save_table(conservative_interpretation_table_df, "conservative_interpretation_table.csv", "Conservative interpretation table", "Cell 80", "Reviewer-safe interpretation of what the results support and do not support.")


,result component,what the result supports,what it does not support
0,Subject-level fixed split performance,Preliminary within-dataset discrimination unde...,External clinical diagnostic validity
1,Bootstrap confidence intervals,Uncertainty is wide on the small test set,Precision comparable to large multicenter vali...
2,Repeated lightweight splits,Some split-sensitivity context for non-deep ba...,Full repeated deep-model validation
3,LOSO lightweight validation,Baseline subject-level cross-validation context,Transformer/GRU LOSO equivalence
4,Envelope connectivity model input,Final model uses theta/beta amplitude-envelope...,Raw-amplitude correlations as final model evid...
5,Clinical screening scenario,Potential decision-support framing at assumed ...,Standalone ADHD diagnosis


PosixPath('/Users/talgatazykanov/Desktop/Science works/Makbal_PhD/outputs_reviewer_ready_v2/tables/conservative_interpretation_table.csv')

## Section 24. Equation and Variable Explanation Tables


In [77]:
# Cell 82: Generate reviewer-safe equation, variable, and hyperparameter explanation tables.
adjacency_threshold_explanation_df = pd.DataFrame(
    [
        ("A_t(i,j)=0", "No edge is retained between channels i and j at window t after thresholding or diagonal removal."),
        ("Self-connection removal", "The diagonal of each connectivity matrix is set to zero before graph modeling."),
        ("Thresholded weak edge removal", f"Edges with absolute connectivity below {SELECTED_THRESHOLD:.2f} are omitted in graph-metric summaries."),
        ("Selected threshold", f"{SELECTED_THRESHOLD:.2f}"),
        ("Justification", "Preserves stronger envelope-connectivity edges while enabling graph metrics; sensitivity across candidate thresholds is reported in Supplementary Figure S15."),
    ],
    columns=["item", "explanation"],
)
display(adjacency_threshold_explanation_df)
save_table(adjacency_threshold_explanation_df, "adjacency_threshold_explanation.csv", "Adjacency threshold explanation", "Cell 82", "Reviewer-safe explanation of thresholded adjacency notation.")

standardization_equation_explanation_df = pd.DataFrame(
    [
        ("edge_value", "A single upper-triangular theta or beta envelope-connectivity feature from a sequence window."),
        ("train_mean", "Feature-wise mean computed only from training sequences."),
        ("train_standard_deviation", "Feature-wise standard deviation computed only from training sequences."),
        ("z = (edge_value - train_mean) / train_standard_deviation", "Standardized edge feature used for validation/test transformation."),
        ("training-only statistics", "Validation and test sequences do not contribute to mean or standard deviation estimation."),
        ("no validation/test leakage", "The same training scaler is applied unchanged to validation and test sets."),
    ],
    columns=["variable or equation", "explanation"],
)
display(standardization_equation_explanation_df)
save_table(standardization_equation_explanation_df, "standardization_equation_explanation.csv", "Standardization equation explanation", "Cell 82", "Reviewer-safe explanation of edge-feature standardization variables.")

hyperparameter_justification_df = pd.DataFrame(
    [
        ("threshold 0.40", SELECTED_THRESHOLD, "Used for graph-metric summaries; sensitivity is reported."),
        ("window length 512", SELECTED_WINDOW, "Preserved from original notebook; duration depends on sampling-frequency resolution."),
        ("stride 256", SELECTED_STEP, "50% overlap under the fixed sample-count design."),
        ("sequence length 10", SEQ_LEN, "Preserved dynamic sequence length."),
        ("sequence stride 5", SEQ_STRIDE, "Preserved sequence step in windows."),
        ("theta band", MODELING_BANDS["theta"], "Preserved band-limited envelope-connectivity input."),
        ("beta band", MODELING_BANDS["beta"], "Preserved band-limited envelope-connectivity input."),
        ("dropout", proposed_architecture_kwargs["dropout"], "Executed proposed model regularization."),
        ("weight decay", proposed_training_kwargs["weight_decay"], "Executed proposed model AdamW regularization."),
        ("learning rate", proposed_training_kwargs["lr"], "Executed proposed model optimizer step size."),
        ("batch size", BATCH_SIZE, "Executed mini-batch size."),
    ],
    columns=["hyperparameter", "executed value", "justification"],
)
display(hyperparameter_justification_df)
save_table(hyperparameter_justification_df, "hyperparameter_justification.csv", "Hyperparameter justification", "Cell 82", "Reviewer-safe justification of major preprocessing, graph, and model hyperparameters.")


,item,explanation
0,"A_t(i,j)=0",No edge is retained between channels i and j a...
1,Self-connection removal,The diagonal of each connectivity matrix is se...
2,Thresholded weak edge removal,Edges with absolute connectivity below 0.40 ar...
3,Selected threshold,0.40
4,Justification,Preserves stronger envelope-connectivity edges...


,variable or equation,explanation
0,edge_value,A single upper-triangular theta or beta envelo...
1,train_mean,Feature-wise mean computed only from training ...
2,train_standard_deviation,Feature-wise standard deviation computed only ...
3,z = (edge_value - train_mean) / train_standard...,Standardized edge feature used for validation/...
4,training-only statistics,Validation and test sequences do not contribut...
5,no validation/test leakage,The same training scaler is applied unchanged ...


,hyperparameter,executed value,justification
0,threshold 0.40,0.4,Used for graph-metric summaries; sensitivity i...
1,window length 512,512,Preserved from original notebook; duration dep...
2,stride 256,256,50% overlap under the fixed sample-count design.
3,sequence length 10,10,Preserved dynamic sequence length.
4,sequence stride 5,5,Preserved sequence step in windows.
5,theta band,"(4.0, 8.0)",Preserved band-limited envelope-connectivity i...
6,beta band,"(13.0, 30.0)",Preserved band-limited envelope-connectivity i...
7,dropout,0.25,Executed proposed model regularization.
8,weight decay,0.0003,Executed proposed model AdamW regularization.
9,learning rate,0.0004,Executed proposed model optimizer step size.


PosixPath('/Users/talgatazykanov/Desktop/Science works/Makbal_PhD/outputs_reviewer_ready_v2/tables/hyperparameter_justification.csv')

## Section 25. Technical ROC Curves and Output Manifest


In [78]:
# Cell 83: Generate technical subject-level ROC curves for all evaluated models.
fig, ax = plt.subplots(figsize=(12, 10), constrained_layout=True)
roc_table_rows = []
for model_name in classification_results_full_df["model"].tolist():
    subject_scores = aggregate_subject_scores(sequence_meta_test, model_score_store[model_name]["test"], SUBJECT_AGGREGATION_METHOD)
    fpr, tpr, thresholds = roc_curve(subject_scores["label"].to_numpy(), subject_scores["score"].to_numpy())
    auc_value = roc_auc_score(subject_scores["label"].to_numpy(), subject_scores["score"].to_numpy())
    display_name = DISPLAY_NAME_MAP.get(model_name, model_name)
    ax.plot(fpr, tpr, label=f"{display_name} ({auc_value:.2f})")
    roc_table_rows.append({"model": display_name, "auc_roc": float(auc_value), "n_subjects": int(subject_scores.shape[0])})
ax.plot([0, 1], [0, 1], color=MUTED_GREY, linestyle="--", linewidth=1.4, label="Chance")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("Subject-Level ROC Curves on Independent Test Set")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.38), ncol=2)
saved_path = save_main_figure(fig, "technical_subject_level_roc_curves_all_models.png", "Subject-level ROC curves", "Cell 83", "Subject-level ROC curves for all evaluated models.")
roc_curve_summary_df = pd.DataFrame(roc_table_rows)
display(roc_curve_summary_df.round(3))
save_table(roc_curve_summary_df, "roc_curve_summary.csv", "ROC curve summary", "Cell 83", "AUC values underlying the subject-level ROC curve figure.")


Figure saved at OLD_Docs/outputs_reviewer_ready_v2/main_figures/technical_subject_level_roc_curves_all_models.png

,model,auc_roc,n_subjects
0,Proposed Hybrid ST Graph Transformer,0.744,19
1,Proposed model ablation,0.733,19
2,GRU dynamic index,0.656,19
3,Global graph-metric index,0.633,19
4,Linear network index,0.467,19
5,Spatio-temporal GCN index,0.478,19
6,Static GCN index,0.444,19
7,Sparse oscillatory index,0.467,19


PosixPath('/Users/talgatazykanov/Desktop/Science works/Makbal_PhD/outputs_reviewer_ready_v2/tables/roc_curve_summary.csv')